# EGMS-Drive publication figures and tables: readable, self-contained reproduction

Run all cells to regenerate **Figures 2–4, Figure S1, Table 2, and Tables S2–S4** from the same Python source, frozen YAML protocols, and machine-readable numeric inputs used by the GitHub release. Every program, configuration, and CSV input appears below as complete readable text in its own `%%writefile` cell. No source is hidden in an encoded or compressed payload, and the notebook does not embed or read any PNG, PDF, SVG, Word, archive, or compressed data file as a plotting input.

Evidence boundary: Study 1 is an audited exported-summary reproduction; Studies 2–3 are controlled synthetic mechanism surrogates; the power analysis is prospective. These outputs are not CARLA, public-dataset, real-vehicle, or empirical LLM evidence.

## 1. Check the runtime and prepare a clean working directory

In [ ]:
import importlib.metadata
import importlib.util
import os
import subprocess
import sys
import tempfile
from pathlib import Path

try:
    from packaging.requirements import Requirement
except ModuleNotFoundError:
    from pip._vendor.packaging.requirements import Requirement

try:
    IS_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IS_COLAB = False

requirements = {
    "numpy": "numpy>=2.0,<3",
    "pandas": "pandas>=2.2,<3",
    "scipy": "scipy>=1.12,<2",
    "matplotlib": "matplotlib>=3.8,<4",
    "seaborn": "seaborn>=0.13,<1",
    "yaml": "PyYAML>=6.0,<7",
    "PIL": "Pillow>=10,<13",
    "sklearn": "scikit-learn>=1.4,<2",
}


def needs_install(module, spec):
    if importlib.util.find_spec(module) is None:
        return True
    requirement = Requirement(spec)
    try:
        installed = importlib.metadata.version(requirement.name)
    except importlib.metadata.PackageNotFoundError:
        return True
    return not requirement.specifier.contains(installed, prereleases=True)


missing_or_incompatible = [
    spec for module, spec in requirements.items() if needs_install(module, spec)
]
if missing_or_incompatible:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--disable-pip-version-check",
            *missing_or_incompatible,
        ],
        check=True,
    )

print("Runtime:", "Google Colab" if IS_COLAB else "local validation")
for distribution in ("numpy", "pandas", "scipy", "matplotlib", "seaborn", "PyYAML", "Pillow", "scikit-learn"):
    try:
        print(f"{distribution}: {importlib.metadata.version(distribution)}")
    except importlib.metadata.PackageNotFoundError:
        pass

CANONICAL_PATHS = [
  "run_publication.py",
  "run_studies.py",
  "src/egms_publication/__init__.py",
  "src/egms_publication/figures.py",
  "src/egms_publication/runner.py",
  "src/egms_publication/style.py",
  "src/egms_publication/tables.py",
  "src/egms_publication/utils.py",
  "src/egms_publication/validation.py",
  "src/egms_power/__init__.py",
  "src/egms_power/cli.py",
  "src/egms_power/config.py",
  "src/egms_power/pipeline.py",
  "src/egms_power/statistics.py",
  "src/egms_studies23/__init__.py",
  "src/egms_studies23/common.py",
  "src/egms_studies23/plots.py",
  "src/egms_studies23/reporting.py",
  "src/egms_studies23/runner.py",
  "src/egms_studies23/study2.py",
  "src/egms_studies23/study3.py",
  "src/egms_studies23/validation.py",
  "configs/power_protocol.yaml",
  "configs/studies23.yaml",
  "data/study1/study1_figure_inputs.csv",
  "data/study1/study1_table2_inputs.csv",
  "data/studies23_frozen/tables/table_s2_primary_contrasts.csv",
  "data/studies23_frozen/tables/table_s3_by_regime.csv",
  "data/studies23_frozen/tables/table_s3_latency.csv",
  "data/studies23_frozen/tables/table_s3_main.csv",
  "data/studies23_frozen/tables/table_s3_negative_controls.csv",
  "data/studies23_frozen/tables/table_s3_primary_contrasts.csv"
]
requested_work_root = os.environ.get("EGMS_COLAB_WORK_ROOT", "").strip()
if requested_work_root:
    WORK_ROOT = Path(requested_work_root).expanduser().resolve()
    WORK_ROOT.mkdir(parents=True, exist_ok=False)
else:
    runtime_parent = Path("/content") if IS_COLAB else None
    WORK_ROOT = Path(tempfile.mkdtemp(prefix="EGMS_Drive_Publication_", dir=runtime_parent))
for relative in CANONICAL_PATHS:
    (WORK_ROOT / relative).parent.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_ROOT)
print("Working directory:", WORK_ROOT)
print(f"Prepared directories for {len(CANONICAL_PATHS)} readable source/config/CSV files")


## 2. Write and SHA-256-verify the canonical release inputs

The following cells show each complete Python, YAML, and CSV file directly. They are ordinary readable source cells; editing a cell changes the file that is written and causes the later SHA-256 verification to fail until the expected release hash is deliberately updated.

### `run_publication.py`

In [ ]:
%%writefile run_publication.py
#!/usr/bin/env python3
from pathlib import Path
import sys

ROOT = Path(__file__).resolve().parent
sys.path.insert(0, str(ROOT / "src"))

from egms_publication.runner import main


if __name__ == "__main__":
    raise SystemExit(main())



### `run_studies.py`

In [ ]:
%%writefile run_studies.py
#!/usr/bin/env python3
"""Run, reanalyze, or validate the controlled synthetic Studies 2-3."""

from pathlib import Path
import sys


ROOT = Path(__file__).resolve().parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from egms_studies23.runner import main  # noqa: E402


if __name__ == "__main__":
    main()


### `src/egms_publication/__init__.py`

In [ ]:
%%writefile src/egms_publication/__init__.py
"""Publication-level reproducibility utilities for EGMS-Drive."""

__version__ = "1.1.0"


### `src/egms_publication/figures.py`

In [ ]:
%%writefile src/egms_publication/figures.py
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from .style import GRAY, NAVY, RED, TEAL, TEXT, apply_style, format_axes, panel_title, save_figure
from .utils import select_one


def _as_bool(value: object) -> bool:
    """Parse CSV booleans without treating the string ``False`` as true."""

    if isinstance(value, bool):
        return value
    normalized = str(value).strip().lower()
    if normalized in {"true", "1", "yes"}:
        return True
    if normalized in {"false", "0", "no"}:
        return False
    raise ValueError(f"Cannot interpret boolean value: {value!r}")


def _horizontal_ci(
    ax: plt.Axes,
    *,
    estimate: float,
    low: float,
    high: float,
    y: float,
    color: str,
    marker: str = "o",
    filled: bool = False,
    size: float = 6.3,
) -> None:
    ax.errorbar(
        estimate,
        y,
        xerr=np.array([[estimate - low], [high - estimate]]),
        fmt=marker,
        markersize=size,
        markerfacecolor=color if filled else "white",
        markeredgecolor=color,
        markeredgewidth=1.35,
        ecolor=color,
        elinewidth=1.55,
        capsize=3.8,
        capthick=1.55,
        zorder=3,
    )


def _endpoint_label(
    ax: plt.Axes,
    text: str,
    *,
    low: float,
    high: float,
    y: float,
    side: str,
    top_row: bool = False,
) -> None:
    anchor = high if side == "right" else low
    dx = 5 if side == "right" else -5
    dy = -5 if top_row else 5
    ax.annotate(
        text,
        (anchor, y),
        xytext=(dx, dy),
        textcoords="offset points",
        ha="left" if side == "right" else "right",
        va="top" if top_row else "bottom",
        fontsize=8.2,
        color=TEXT,
    )


def _benefit_oriented(row: pd.Series) -> tuple[float, float, float]:
    estimate = float(row["estimate"])
    low = float(row["ci_low"])
    high = float(row["ci_high"])
    if row["orientation"] == "lower_is_better":
        return -estimate, -high, -low
    return estimate, low, high


def build_figure2(study1_csv: Path, stem: Path) -> dict[str, Path]:
    """Generate manuscript Figure 2 from the frozen Study-1 audit snapshot.

    This is a graphical reproduction of exported/audited summary estimates.
    Study-1 raw data, model training code, and some original CI procedures are
    unavailable; this function must not be described as a raw recomputation.
    """

    frame = pd.read_csv(study1_csv)
    apply_style()
    width_px, height_px = 3810, 2522
    fig, axes = plt.subplots(2, 2, figsize=(width_px / 600, height_px / 600), dpi=600)
    fig.subplots_adjust(left=0.105, right=0.982, bottom=0.105, top=0.950, wspace=0.43, hspace=0.50)

    # (a) Offline action and probability scores, displayed in a benefit direction.
    ax = axes[0, 0]
    metrics = ["Macro-F1", "NLL", "Brier", "ECE"]
    ys = np.arange(3, -1, -1, dtype=float)
    for row_index, (metric, y) in enumerate(zip(metrics, ys)):
        row = select_one(frame, panel="action", metric=metric)
        estimate, low, high = _benefit_oriented(row)
        supported = low > 0
        color = NAVY if supported else RED
        _horizontal_ci(
            ax,
            estimate=estimate,
            low=low,
            high=high,
            y=y,
            color=color,
            filled=supported,
        )
        ax.annotate(
            f"{estimate:+.4f}",
            (high, y),
            xytext=(5, -5 if row_index == 0 else 5),
            textcoords="offset points",
            ha="left",
            va="top" if row_index == 0 else "bottom",
            fontsize=8.2,
        )
    ax.axvline(0, color=TEAL, ls="--", lw=1.4)
    ax.set_xlim(-0.0077, 0.0425)
    ax.set_xticks([0.00, 0.01, 0.02, 0.03, 0.04])
    ax.set_yticks(ys, metrics)
    ax.set_ylim(-0.15, 3.15)
    ax.set_xlabel("Benefit-oriented raw difference")
    panel_title(ax, "(a) Offline action and probability scores")
    format_axes(ax)

    # (b) Event and completion proxies, also transformed to a benefit direction.
    ax = axes[0, 1]
    metrics = ["Collision", "Near miss", "Critical event", "Route completion"]
    labels = ["Collision", "Near miss", "Critical event", "Route completion"]
    ys = np.arange(3, -1, -1, dtype=float)
    label_sides = ["right", "left", "left", "right"]
    for row_index, (metric, y, side) in enumerate(zip(metrics, ys, label_sides)):
        row = select_one(frame, panel="event", metric=metric)
        estimate, low, high = _benefit_oriented(row)
        _horizontal_ci(
            ax,
            estimate=estimate,
            low=low,
            high=high,
            y=y,
            color=RED,
            filled=False,
        )
        anchor = high if side == "right" else low
        ax.annotate(
            f"{estimate:+.2f} pp",
            (anchor, y),
            xytext=(5 if side == "right" else -5, -5 if row_index == 0 else 5),
            textcoords="offset points",
            ha="left" if side == "right" else "right",
            va="top" if row_index == 0 else "bottom",
            fontsize=8.2,
        )
    ax.axvline(0, color=TEAL, ls="--", lw=1.4)
    ax.set_xlim(-7.6, 20.6)
    ax.set_yticks(ys, labels)
    ax.set_ylim(-0.15, 3.15)
    ax.set_xlabel("Benefit-oriented paired difference (pp)")
    panel_title(ax, "(b) Event and completion proxies")
    format_axes(ax)

    # (c) Conditional TTC descriptor. Marginal intervals are audit-snapshot values.
    ax = axes[1, 0]
    conditions = [
        ("Baseline B", 0.34, NAVY, "right"),
        ("Structured fusion", 0.66, RED, "left"),
    ]
    for condition, y, color, side in conditions:
        row = select_one(frame, panel="conditional_ttc", metric="TTC-P5", condition=condition)
        estimate, low, high = (float(row[key]) for key in ("estimate", "ci_low", "ci_high"))
        _horizontal_ci(ax, estimate=estimate, low=low, high=high, y=y, color=color)
        _endpoint_label(
            ax,
            f"{estimate:.3f} s",
            low=low,
            high=high,
            y=y,
            side=side,
        )
        ax.plot([], [], "o", color=color, markerfacecolor="white", markeredgewidth=1.2,
                markersize=5.3, label=condition)
    ax.set_xlim(0.58, 1.39)
    ax.set_ylim(0.18, 0.82)
    ax.set_yticks([])
    ax.set_xlabel("Collision-free TTC-P5 (s)")
    panel_title(ax, "(c) Conditional TTC descriptor")
    format_axes(ax)
    ax.legend(loc="upper center", ncol=2, bbox_to_anchor=(0.50, 1.00), handletextpad=0.4, columnspacing=1.0)

    # (d) Kinematic smoothness proxy. Marginal intervals are audit-snapshot values.
    ax = axes[1, 1]
    conditions = [
        ("Baseline B", 0.34, NAVY, "left"),
        ("Structured fusion", 0.66, RED, "right"),
    ]
    for condition, y, color, side in conditions:
        row = select_one(frame, panel="jerk", metric="Jerk-P95", condition=condition)
        estimate, low, high = (float(row[key]) for key in ("estimate", "ci_low", "ci_high"))
        _horizontal_ci(ax, estimate=estimate, low=low, high=high, y=y, color=color)
        _endpoint_label(
            ax,
            f"{estimate:.2f}",
            low=low,
            high=high,
            y=y,
            side=side,
        )
        ax.plot([], [], "o", color=color, markerfacecolor="white", markeredgewidth=1.2,
                markersize=5.3, label=condition)
    ax.set_xlim(20.4, 35.7)
    ax.set_xticks(np.arange(22, 35, 2))
    ax.set_ylim(0.18, 0.82)
    ax.set_yticks([])
    ax.set_xlabel("Mean episode jerk-P95 (m/s³)")
    panel_title(ax, "(d) Kinematic smoothness proxy")
    format_axes(ax)
    ax.legend(loc="upper center", ncol=2, bbox_to_anchor=(0.50, 1.00), handletextpad=0.4, columnspacing=1.0)

    return save_figure(fig, stem, width_px=width_px, height_px=height_px)


def build_figure3(contrast_csv: Path, stem: Path) -> dict[str, Path]:
    """Generate manuscript Figure 3 directly from Study-2 contrast CSV rows."""

    contrasts = pd.read_csv(contrast_csv)
    apply_style()
    width_px, height_px = 3810, 2472
    fig, axes = plt.subplots(2, 2, figsize=(width_px / 600, height_px / 600), dpi=600)
    fig.subplots_adjust(left=0.085, right=0.982, bottom=0.105, top=0.950, wspace=0.42, hspace=0.50)
    specifications = [
        {
            "comparison": "Full vs Equal weighting",
            "title": "(a) Reliability weighting",
            "xlim": (-0.0125, 0.0020),
            "xlabel": "Full − equal weighting (adverse NLL)",
            "better": "Full better ←",
        },
        {
            "comparison": "Full vs No alignment",
            "title": "(b) Cross-modal alignment",
            "xlim": (-0.45, 3.08),
            "xlabel": "Full − no alignment (clean SAS)",
            "better": "→ Full better",
        },
        {
            "comparison": "Full vs No temporal consistency",
            "title": "(c) Temporal consistency",
            "xlim": (-0.070, 0.0108),
            "xlabel": "Full − no temporal (one-step error)",
            "better": "Full better ←",
        },
        {
            "comparison": "Full vs No modality dropout-distillation",
            "title": "(d) Modality dropout–distillation",
            "xlim": (-0.00325, 0.00072),
            "xlabel": "Full − no dropout–distillation (F1 drop)",
            "better": "Full better ←",
        },
    ]

    for panel_index, (ax, specification) in enumerate(zip(axes.flat, specifications)):
        row = select_one(contrasts, comparison=specification["comparison"])
        estimate = float(row["difference_full_minus_ablation"])
        low = float(row["ci_low"])
        high = float(row["ci_high"])
        supported = _as_bool(row["mechanism_support_rule_met"])
        color = NAVY if supported else RED
        _horizontal_ci(
            ax,
            estimate=estimate,
            low=low,
            high=high,
            y=0.14,
            color=color,
            filled=supported,
        )
        ax.axvline(0, color=TEAL, ls="--", lw=1.4)
        ax.set_xlim(*specification["xlim"])
        ax.set_ylim(-0.34, 0.34)
        ax.set_yticks([])
        ax.set_xlabel(specification["xlabel"])
        if panel_index == 0:
            ax.set_xticks([-0.012, -0.009, -0.006, -0.003, 0.000])
        panel_title(ax, specification["title"])
        format_axes(ax)
        ax.text(0.02, 0.88, specification["better"], transform=ax.transAxes,
                ha="left", va="top", fontsize=8.0, color=NAVY)
        status = "Effect supported" if supported else "Benefit not clearly\nestablished"
        ax.text(0.98, 0.88, status, transform=ax.transAxes, ha="right", va="top",
                fontsize=8.0, color=color, fontweight="bold")
        annotation = (
            f"Δ={estimate:+.4f} [{low:+.4f}, {high:+.4f}]\n"
            f"favorable replicates {int(row['favorable_training_replicates'])}/"
            f"{int(row['total_training_replicates'])}; Holm p={float(row['holm_adjusted_p']):.4f}"
        )
        ax.text(0.50, 0.12, annotation, transform=ax.transAxes, ha="center", va="bottom",
                fontsize=7.5, color=TEXT)

    return save_figure(fig, stem, width_px=width_px, height_px=height_px)


def build_figure4(tables_dir: Path, stem: Path) -> dict[str, Path]:
    """Generate manuscript Figure 4 from frozen Study-3 derived tables."""

    regime_rows = pd.read_csv(tables_dir / "table_s3_by_regime.csv")
    main_rows = pd.read_csv(tables_dir / "table_s3_main.csv")
    contrast_rows = pd.read_csv(tables_dir / "table_s3_primary_contrasts.csv")
    control_rows = pd.read_csv(tables_dir / "table_s3_negative_controls.csv")
    latency_rows = pd.read_csv(tables_dir / "table_s3_latency.csv")

    apply_style()
    width_px, height_px = 3810, 2485
    fig, axes = plt.subplots(2, 2, figsize=(width_px / 600, height_px / 600), dpi=600)
    fig.subplots_adjust(
        left=0.160,
        right=0.982,
        bottom=0.115,
        top=0.895,
        wspace=0.45,
        hspace=0.66,
    )

    # (a) Graph mechanism by interaction regime.
    ax = axes[0, 0]
    regimes = ["independent", "spatial", "history_dependent"]
    regime_labels = ["Independent", "Spatial", "History-dependent"]
    method_specs = [
        ("Full temporal graph", -0.16, "o", NAVY, True, "Full temporal graph"),
        ("No graph", 0.00, "s", GRAY, False, "No graph"),
        ("Spatial-only graph", 0.16, "^", RED, False, "Spatial-only graph"),
    ]
    legend_handles: list[object] = []
    legend_labels: list[str] = []
    for method, offset, marker, color, filled, short_label in method_specs:
        for index, regime in enumerate(regimes):
            row = select_one(
                regime_rows,
                method=method,
                regime=regime,
                metric="minFDE_K (m)",
            )
            estimate, low, high = (float(row[key]) for key in ("estimate", "ci_low", "ci_high"))
            container = ax.errorbar(
                index + offset,
                estimate,
                yerr=np.array([[estimate - low], [high - estimate]]),
                fmt=marker,
                markersize=6.0,
                markerfacecolor=color if filled else "white",
                markeredgecolor=color,
                markeredgewidth=1.3,
                ecolor=color,
                elinewidth=1.5,
                capsize=3.5,
                capthick=1.5,
                zorder=3,
            )
            if index == 0:
                legend_handles.append(container)
                legend_labels.append(short_label)
    ax.set_xlim(-0.38, 2.38)
    ax.set_ylim(1.50, 2.68)
    ax.set_xticks(range(3), regime_labels)
    ax.set_yticks(np.arange(1.6, 2.61, 0.2))
    ax.set_ylabel("minFDE$_6$ (m)")
    panel_title(ax, "(a) Graph mechanism by interaction regime")
    format_axes(ax, grid_axis="y")

    independent = select_one(control_rows, negative_control="independent: Full vs No graph")
    spatial = select_one(control_rows, negative_control="spatial: Full vs Spatial-only graph")
    ax.text(
        0.02,
        -0.18,
        (
            f"Independent Δ={float(independent['difference_full_minus_ablation']):+.3f} m: "
            "within ±0.25-m descriptive range\n"
            f"Spatial Δ={float(spatial['difference_full_minus_ablation']):+.3f} m: "
            "range not met (not TOST)"
        ),
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=6.9,
        color=GRAY,
        linespacing=1.05,
    )
    fig.legend(
        legend_handles,
        legend_labels,
        loc="upper center",
        bbox_to_anchor=(0.50, 0.995),
        ncol=3,
        frameon=False,
        fontsize=7.0,
        handletextpad=0.35,
        columnspacing=1.0,
    )

    # (b) Predicted-intent gating.
    ax = axes[0, 1]
    gating_specs = [
        ("Full temporal graph", "Full", 1.0, NAVY, True, "right"),
        ("No predicted-intent gating", "No intent gate", 0.0, RED, False, "left"),
    ]
    for method, label, y, color, filled, side in gating_specs:
        row = select_one(main_rows, method=method, metric="Brier-minFDE_6")
        estimate, low, high = (float(row[key]) for key in ("estimate", "ci_low", "ci_high"))
        _horizontal_ci(ax, estimate=estimate, low=low, high=high, y=y, color=color, filled=filled)
        _endpoint_label(ax, f"{estimate:.3f}", low=low, high=high, y=y, side=side, top_row=y == 1.0)
    gate_contrast = select_one(
        contrast_rows,
        comparison="Full temporal graph vs No predicted-intent gating",
    )
    ax.text(
        0.04,
        0.50,
        (
            f"Paired Δ={float(gate_contrast['difference_full_minus_ablation']):+.3f} "
            f"[{float(gate_contrast['ci_low']):+.3f}, "
            f"{float(gate_contrast['ci_high']):+.3f}]\n"
            "Effect supported"
        ),
        transform=ax.transAxes,
        ha="left",
        va="center",
        fontsize=7.6,
        fontweight="bold",
        color=NAVY,
        linespacing=1.05,
    )
    ax.set_xlim(2.2, 10.55)
    ax.set_ylim(-0.24, 1.24)
    ax.set_yticks([1, 0], ["Full", "No intent gate"])
    ax.set_xlabel("Brier-minFDE$_6$ (lower is better)")
    panel_title(ax, "(b) Predicted-intent gating")
    format_axes(ax)

    # (c) Top-1 multimodal output versus independently fitted K=1.
    ax = axes[1, 0]
    top1_specs = [
        ("Full temporal graph", "Full top-1", 1.0, NAVY, "right"),
        ("Single mode K=1", "Independent K=1", 0.0, RED, "left"),
    ]
    for method, label, y, color, side in top1_specs:
        row = select_one(main_rows, method=method, metric="MR_1 at 2 m")
        estimate, low, high = (100.0 * float(row[key]) for key in ("estimate", "ci_low", "ci_high"))
        _horizontal_ci(ax, estimate=estimate, low=low, high=high, y=y, color=color)
        _endpoint_label(ax, f"{estimate:.2f}%", low=low, high=high, y=y, side=side, top_row=y == 1.0)
    top1_contrast = select_one(
        contrast_rows,
        comparison="Full top-1 vs independently trained Single mode K=1",
    )
    ax.text(
        0.04,
        0.50,
        (
            f"Paired Δ={100 * float(top1_contrast['difference_full_minus_ablation']):+.2f} pp "
            f"[{100 * float(top1_contrast['ci_low']):+.2f}, "
            f"{100 * float(top1_contrast['ci_high']):+.2f}]\n"
            "Benefit not clearly established"
        ),
        transform=ax.transAxes,
        ha="left",
        va="center",
        fontsize=7.4,
        fontweight="bold",
        color=RED,
        linespacing=1.05,
    )
    ax.set_xlim(75.70, 79.75)
    ax.set_ylim(-0.24, 1.24)
    ax.set_yticks([1, 0], ["Full top-1", "Independent K=1"])
    ax.set_xlabel("MR$_1$ at 2 m (%) — lower is better")
    panel_title(ax, "(c) Top-1 multimodal output vs K=1")
    format_axes(ax)

    # (d) CPU timing diagnostic.
    ax = axes[1, 1]
    latency_specs = [
        ("Full temporal graph", "Full", 4.0, NAVY, True, "right"),
        ("No graph", "No graph", 3.0, GRAY, False, "right"),
        ("Spatial-only graph", "Spatial-only", 2.0, GRAY, False, "right"),
        ("No predicted-intent gating", "No intent gate", 1.0, GRAY, False, "left"),
        ("Single mode K=1", "K=1", 0.0, RED, False, "right"),
    ]
    latency_metric = "End-to-end batch-1 CPU latency P95 (ms)"
    for method, label, y, color, filled, side in latency_specs:
        row = select_one(latency_rows, method=method, metric=latency_metric)
        estimate, low, high = (float(row[key]) for key in ("estimate", "ci_low", "ci_high"))
        _horizontal_ci(ax, estimate=estimate, low=low, high=high, y=y, color=color, filled=filled)
        _endpoint_label(ax, f"{estimate:.3f}", low=low, high=high, y=y, side=side, top_row=y == 4.0)
    ax.set_xlim(0.58, 1.055)
    ax.set_ylim(-0.24, 4.24)
    ax.set_yticks([4, 3, 2, 1, 0], ["Full", "No graph", "Spatial-only", "No intent gate", "K=1"])
    ax.set_xlabel("Batch-1 CPU latency P95 (ms)")
    panel_title(ax, "(d) Deployment timing diagnostic")
    format_axes(ax)

    return save_figure(fig, stem, width_px=width_px, height_px=height_px)



### `src/egms_publication/runner.py`

In [ ]:
%%writefile src/egms_publication/runner.py
from __future__ import annotations

import argparse
import json
from pathlib import Path
import shutil
import zipfile

import pandas as pd

from egms_power.pipeline import run_power_analysis

from .figures import build_figure2, build_figure3, build_figure4
from .tables import build_main_table2, build_table_s2, build_table_s3, build_table_s4
from .utils import sha256, write_table
from .validation import validate_outputs, verify_manifest, write_manifest


SENTINEL = ".egms_publication_output"
REQUIRED_CONTROLLED_TABLES = (
    "table_s2_primary_contrasts.csv",
    "table_s3_by_regime.csv",
    "table_s3_latency.csv",
    "table_s3_main.csv",
    "table_s3_negative_controls.csv",
    "table_s3_primary_contrasts.csv",
)


def project_root() -> Path:
    return Path(__file__).resolve().parents[2]


def _prepare_output(root: Path, requested: Path, overwrite: bool) -> Path:
    root = root.resolve()
    output = requested if requested.is_absolute() else root / requested
    output = output.resolve()
    if output == root or root not in output.parents:
        raise ValueError("Output must be a named child of the repository root")
    if output.exists():
        if not overwrite:
            raise FileExistsError(f"Output exists: {output}; pass --overwrite")
        if not (output / SENTINEL).is_file():
            raise ValueError(f"Refusing to replace a directory without {SENTINEL}: {output}")
        shutil.rmtree(output)
    output.mkdir(parents=True)
    (output / SENTINEL).write_text("Managed EGMS-Drive publication output.\n", encoding="utf-8")
    return output


def _publication_input_tables(root: Path) -> Path:
    """Return the small, frozen Study 2--3 tables used by publication mode.

    This source-only release intentionally omits persisted prediction frames and
    previously generated result directories.  A full controlled-synthetic refit
    remains available through ``run_studies.py run``; publication mode uses only
    the six verified summary tables needed by Figures 3--4 and Table 2.
    """

    tables = root / "data" / "studies23_frozen" / "tables"
    missing = [name for name in REQUIRED_CONTROLLED_TABLES if not (tables / name).is_file()]
    if missing:
        raise FileNotFoundError(
            "Missing publication input table(s): " + ", ".join(missing)
        )
    return tables


def _write_numbering_map(output: Path) -> None:
    text = """# Publication and Supplementary Numbering Map

The current compact manuscript contains Supplementary Sections S1-S3. Its
Supplementary Section S2 contains Figure S1 and Tables S2-S4; there is no
Supplementary Section S4 in the compact manuscript.

## Current compact manuscript

| Manuscript item | Generated file |
|---|---|
| Figure 2 | `manuscript/figures/Figure_2_Study1.*` |
| Figure 3 | `manuscript/figures/Figure_3_Study2.*` |
| Figure 4 | `manuscript/figures/Figure_4_Study3.*` |
| Table 2 | `manuscript/tables/Table_2_main_effects.*` |
| Figure S1 | `power_full/figures/figure1_unpaired_power_curve.*` |
| Table S2 | `supplement/compact/Tables/Table_S2_planning_summary.*` |
| Table S3 | `supplement/compact/Tables/Table_S3_condensed_planning.*` |
| Table S4 | `supplement/compact/Tables/Table_S4_validation_checks.*` |
| Supplement S3 inventory | `supplement/compact/S3_Digital_Reproducibility_Inventory.*` |

## Earlier full-supplement numbering

The earlier full version used Sections S2-S4. Its power Figures S1-S6 map to
`power_full/figures/figure1_*` through `figure6_*`; Tables S2-S12 map to
`power_full/tables/table1_*` through `table11_*`; Table S13 maps to
`power_full/validation_report.csv`. The former S2 equations were prospective
and did not produce empirical outputs.
"""
    path = output / "supplement" / "NUMBERING_MAP.md"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def _write_supplement_text(output: Path, table_s2: pd.DataFrame, table_s3: pd.DataFrame) -> None:
    compact = output / "supplement" / "compact"
    compact.mkdir(parents=True, exist_ok=True)
    power_text = f"""# S2 Compact Prospective Collision-Power Analysis

All quantities here are prospective planning results conditional on assumed
event risks and dependence. They are not EGMS-Drive performance, CARLA
outcomes, or observed safety benefits.

The headline independent calculation requires
{table_s2.loc[table_s2['Item'].eq('Required independent n'), 'Value'].iloc[0]}.
The paired and cluster-aware sensitivity summary is generated in Table S3.
Figure S1 is generated by the power-analysis engine at
`../../power_full/figures/figure1_unpaired_power_curve.*`.

Generated compact tables:

- `Tables/Table_S2_planning_summary.*`
- `Tables/Table_S3_condensed_planning.*`
- `Tables/Table_S4_validation_checks.*`

The complete six-figure and eleven-table planning outputs remain in
`../../power_full/`.
"""
    (compact / "S2_Compact_Prospective_Collision_Power_Analysis.md").write_text(power_text, encoding="utf-8")
    inventory_text = """# S3 Digital Reproducibility Inventory and Availability Limitations

The generated inventory lists source, configuration, frozen numeric inputs,
derived tables, figures, and validation artifacts with SHA-256 hashes. Study 1
is limited to an exported/audited summary snapshot; the original model,
checkpoint, split manifest, raw frames, and several interval algorithms are
not available. Studies 2-3 are executable controlled synthetic mechanism
surrogates, not the full neural EGMS-Drive architecture. The power analysis is
prospective. No CARLA, public-dataset, real-vehicle, or empirical LLM result is
contained in this package.
"""
    (compact / "S3_Digital_Reproducibility_Inventory.md").write_text(inventory_text, encoding="utf-8")
    earlier = output / "supplement" / "earlier_full_numbering"
    earlier.mkdir(parents=True, exist_ok=True)
    earlier_text = """# Earlier Full-Supplement S2-S4 Crosswalk

- **S2** contained prospective evidence-selection and LLM formulations only.
  Those equations were untested and generated no empirical figure or table.
- **S3** is reproduced programmatically in `../../power_full/`: six figures,
  eleven detailed planning tables, and the 33-check validation report.
- **S4** is the programmatically generated artifact inventory stored beside
  this file as `S4_Digital_Reproducibility_Inventory.*`.

The current compact manuscript renumbers compact power as S2 and the artifact
inventory as S3. See `../NUMBERING_MAP.md`. No figure in either numbering
scheme is used as a plotting input.
"""
    (earlier / "README.md").write_text(earlier_text, encoding="utf-8")


def _write_inventory(root: Path, output: Path) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    candidates = []
    for base in (root / "configs", root / "data", root / "src"):
        candidates.extend(
            path
            for path in base.rglob("*")
            if path.is_file()
            and "__pycache__" not in path.parts
            and path.suffix.lower() not in {".pyc", ".pyo"}
        )
    candidates.extend(
        path for path in output.rglob("*")
        if path.is_file() and path.name not in {"artifact_manifest.json", "publication_outputs.zip"}
    )
    for path in sorted(candidates):
        if path.is_relative_to(output):
            category = "generated_output"
            relative = f"outputs/{path.relative_to(output)}"
        elif path.is_relative_to(root / "src"):
            category = "source_code"
            relative = str(path.relative_to(root))
        elif path.is_relative_to(root / "configs"):
            category = "configuration"
            relative = str(path.relative_to(root))
        else:
            category = "frozen_numeric_input"
            relative = str(path.relative_to(root))
        rows.append(
            {
                "category": category,
                "path": relative,
                "bytes": path.stat().st_size,
                "sha256": sha256(path),
            }
        )
    frame = pd.DataFrame(rows)
    stem = output / "supplement" / "compact" / "S3_Digital_Reproducibility_Inventory"
    write_table(frame, stem)
    legacy_stem = (
        output
        / "supplement"
        / "earlier_full_numbering"
        / "S4_Digital_Reproducibility_Inventory"
    )
    write_table(frame, legacy_stem)
    return frame


def _zip_outputs(output: Path) -> Path:
    """Write a byte-reproducible ZIP with stable member order and metadata."""

    destination = output / "publication_outputs.zip"
    with zipfile.ZipFile(destination, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=9) as archive:
        for path in sorted(output.rglob("*")):
            if path.is_file() and path != destination:
                relative = path.relative_to(output).as_posix()
                info = zipfile.ZipInfo(relative, date_time=(1980, 1, 1, 0, 0, 0))
                info.compress_type = zipfile.ZIP_DEFLATED
                info.create_system = 3
                info.external_attr = 0o100644 << 16
                archive.writestr(
                    info,
                    path.read_bytes(),
                    compress_type=zipfile.ZIP_DEFLATED,
                    compresslevel=9,
                )
    return destination


def run_publication(
    output_request: Path,
    *,
    overwrite: bool = False,
) -> dict[str, object]:
    root = project_root()
    output = _prepare_output(root, output_request, overwrite)
    tables_dir = _publication_input_tables(root)

    power_output = output / "power_full"
    run_power_analysis(root / "configs" / "power_protocol.yaml", power_output)

    manuscript_figures = output / "manuscript" / "figures"
    build_figure2(
        root / "data" / "study1" / "study1_figure_inputs.csv",
        manuscript_figures / "Figure_2_Study1",
    )
    build_figure3(
        tables_dir / "table_s2_primary_contrasts.csv",
        manuscript_figures / "Figure_3_Study2",
    )
    build_figure4(tables_dir, manuscript_figures / "Figure_4_Study3")

    table2 = build_main_table2(
        root / "data" / "study1" / "study1_table2_inputs.csv",
        tables_dir,
        output / "manuscript" / "tables" / "Table_2_main_effects",
    )
    compact_tables = output / "supplement" / "compact" / "Tables"
    table_s2 = build_table_s2(
        root / "configs" / "power_protocol.yaml",
        power_output,
        compact_tables / "Table_S2_planning_summary",
    )
    table_s3 = build_table_s3(power_output, compact_tables / "Table_S3_condensed_planning")
    table_s4 = build_table_s4(power_output, compact_tables / "Table_S4_validation_checks")
    _write_numbering_map(output)
    _write_supplement_text(output, table_s2, table_s3)
    _write_inventory(root, output)
    result = {
        "status": "completed",
        "output_directory": str(output.relative_to(root)),
        "output_zip": str((output / "publication_outputs.zip").relative_to(root)),
        "table_rows": {
            "Table 2": len(table2),
            "Table S2": len(table_s2),
            "Table S3": len(table_s3),
            "Table S4": len(table_s4),
        },
        "validation": None,
    }
    validation = validate_outputs(root, output)
    result["validation"] = validation
    (output / "run_summary.json").write_text(json.dumps(result, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    manifest = write_manifest(root, output)
    verify_manifest(output, manifest)
    _zip_outputs(output)
    return result


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description="Regenerate EGMS-Drive publication figures and tables")
    parser.add_argument("--output", type=Path, default=Path("outputs/publication_reproduction"))
    parser.add_argument("--overwrite", action="store_true")
    return parser


def main(argv: list[str] | None = None) -> int:
    args = build_parser().parse_args(argv)
    result = run_publication(
        args.output,
        overwrite=args.overwrite,
    )
    print(json.dumps(result, ensure_ascii=False, indent=2))
    return 0


### `src/egms_publication/style.py`

In [ ]:
%%writefile src/egms_publication/style.py
from __future__ import annotations

from pathlib import Path
import struct

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt


NAVY = "#174A7E"
RED = "#D1495B"
TEAL = "#2A9D8F"
GRAY = "#6B7280"
GRID = "#D6D6D6"
TEXT = "#111111"


def apply_style(scale: float = 1.0) -> None:
    """Apply the manuscript's Figure-S1-derived graphics profile."""

    plt.rcParams.update(
        {
            "font.family": "STIXGeneral",
            "mathtext.fontset": "stix",
            "font.size": 9.3 * scale,
            "axes.labelsize": 10.0 * scale,
            "axes.labelpad": 5.0 * scale,
            "axes.linewidth": 0.9 * scale,
            "axes.edgecolor": "black",
            "axes.grid": False,
            "axes.axisbelow": True,
            "grid.color": GRID,
            "grid.linestyle": "-",
            "grid.linewidth": 0.6 * scale,
            "grid.alpha": 0.70,
            "xtick.labelsize": 8.5 * scale,
            "ytick.labelsize": 8.5 * scale,
            "xtick.direction": "out",
            "ytick.direction": "out",
            "xtick.major.size": 4.5 * scale,
            "ytick.major.size": 4.5 * scale,
            "xtick.major.width": 0.9 * scale,
            "ytick.major.width": 0.9 * scale,
            "xtick.major.pad": 3.5 * scale,
            "ytick.major.pad": 3.5 * scale,
            "legend.fontsize": 8.0 * scale,
            "legend.frameon": False,
            "figure.facecolor": "white",
            "axes.facecolor": "white",
            "savefig.facecolor": "white",
            "savefig.dpi": 600,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "svg.fonttype": "none",
            "svg.hashsalt": "egms-drive-publication-v1",
            "text.color": TEXT,
            "axes.labelcolor": TEXT,
            "xtick.color": TEXT,
            "ytick.color": TEXT,
        }
    )


def format_axes(ax: plt.Axes, *, grid_axis: str = "x") -> None:
    ax.grid(True, which="major", axis=grid_axis)
    ax.tick_params(axis="both", which="major", top=False, right=False, bottom=True, left=True)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(plt.rcParams["axes.linewidth"])


def panel_title(ax: plt.Axes, text: str, scale: float = 1.0) -> None:
    ax.set_title(text, loc="left", fontsize=10.5 * scale, fontweight="bold", pad=4.5 * scale)


def save_figure(
    fig: plt.Figure,
    stem: Path,
    *,
    width_px: int,
    height_px: int,
    dpi: int = 600,
) -> dict[str, Path]:
    """Save PNG/PDF/SVG without tight cropping and validate each file."""

    stem.parent.mkdir(parents=True, exist_ok=True)
    fig.set_size_inches((width_px + 0.5) / dpi, (height_px + 0.5) / dpi, forward=True)
    paths = {suffix: stem.with_suffix(f".{suffix}") for suffix in ("png", "pdf", "svg")}
    fig.savefig(paths["pdf"], facecolor="white", edgecolor="white", bbox_inches=None, pad_inches=0)
    fig.savefig(paths["svg"], facecolor="white", edgecolor="white", bbox_inches=None, pad_inches=0)
    fig.savefig(
        paths["png"],
        dpi=dpi,
        facecolor="white",
        edgecolor="white",
        bbox_inches=None,
        pad_inches=0,
    )
    plt.close(fig)

    data = paths["png"].read_bytes()
    if not data.startswith(b"\x89PNG\r\n\x1a\n") or not data.endswith(b"IEND\xaeB`\x82"):
        raise ValueError(f"Invalid PNG: {paths['png']}")
    if struct.unpack(">II", data[16:24]) != (width_px, height_px):
        raise ValueError(f"Unexpected PNG dimensions: {paths['png']}")
    if paths["pdf"].stat().st_size < 1024 or not paths["pdf"].read_bytes().startswith(b"%PDF"):
        raise ValueError(f"Invalid or empty PDF: {paths['pdf']}")
    svg_head = paths["svg"].read_text(encoding="utf-8", errors="replace")[:1000]
    if "<svg" not in svg_head:
        raise ValueError(f"Invalid SVG: {paths['svg']}")
    return paths


### `src/egms_publication/tables.py`

In [ ]:
%%writefile src/egms_publication/tables.py
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
import yaml

from .utils import select_one, write_table


def _as_bool(value: object) -> bool:
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in {"true", "1", "yes", "pass"}


def _signed(value: float, decimals: int = 4) -> str:
    return f"{value:+.{decimals}f}"


def build_main_table2(
    study1_csv: Path,
    studies_tables: Path,
    output_stem: Path,
) -> pd.DataFrame:
    """Build the manuscript's 16-row Table 2 from numeric source tables."""

    s1 = pd.read_csv(study1_csv, keep_default_na=False)
    records: list[dict[str, str]] = []
    for row in s1.itertuples(index=False):
        records.append(
            {
                "Study/module": row.study_module,
                "Comparison and endpoint": row.comparison_endpoint,
                "Effect*": row.effect_display.strip(),
                "95% CI": row.ci_display.strip(),
                "Evidence status": row.evidence_status,
            }
        )

    s2 = pd.read_csv(studies_tables / "table_s2_primary_contrasts.csv")
    s2_specs = [
        (
            "Full vs Equal weighting",
            "S2 reliability",
            "Full vs Equal weighting; adverse NLL ↓",
            4,
            "Surrogate reliability response supported",
            "Benefit not clearly established",
        ),
        (
            "Full vs No alignment",
            "S2 alignment",
            "Full vs No alignment; clean SAS ↑",
            4,
            "Surrogate alignment response supported",
            "Benefit not clearly established",
        ),
        (
            "Full vs No temporal consistency",
            "S2 temporal",
            "Full vs No temporal; one-step error ↓",
            4,
            "Surrogate temporal response supported",
            "Benefit not clearly established",
        ),
        (
            "Full vs No modality dropout-distillation",
            "S2 missing modality",
            "Full vs No dropout–distillation; F1 drop ↓",
            4,
            "Surrogate missing-modality response supported",
            "Benefit not clearly established",
        ),
    ]
    for comparison, module, endpoint, decimals, supported_text, unsupported_text in s2_specs:
        row = select_one(s2, comparison=comparison)
        effect = float(row["difference_full_minus_ablation"])
        low = float(row["ci_low"])
        high = float(row["ci_high"])
        records.append(
            {
                "Study/module": module,
                "Comparison and endpoint": endpoint,
                "Effect*": _signed(effect, decimals),
                "95% CI": f"{_signed(low, decimals)} to {_signed(high, decimals)}",
                "Evidence status": supported_text if _as_bool(row["mechanism_support_rule_met"]) else unsupported_text,
            }
        )

    s3 = pd.read_csv(studies_tables / "table_s3_primary_contrasts.csv")
    s3_specs = [
        (
            "Full temporal graph vs No graph",
            "S3 temporal graph",
            "Full vs No graph; history minFDE₆ ↓",
            1.0,
            4,
            " m",
            "History-dependent graph response supported",
            "Benefit not clearly established",
        ),
        (
            "Full temporal graph vs Spatial-only graph",
            "S3 temporal relation",
            "Full vs Spatial-only; history minFDE₆ ↓",
            1.0,
            4,
            " m",
            "Temporal-relation response supported",
            "Benefit not clearly established",
        ),
        (
            "Full temporal graph vs No predicted-intent gating",
            "S3 intent gate",
            "Full vs No intent gate; Brier-minFDE₆ ↓",
            1.0,
            4,
            "",
            "Intent-gating joint-endpoint response supported",
            "Benefit not clearly established",
        ),
        (
            "Full top-1 vs independently trained Single mode K=1",
            "S3 top-1",
            "Full top-1 vs fitted K=1; MR₁ ↓",
            100.0,
            2,
            " pp",
            "Top-1 response supported",
            "Top-1 benefit not clearly established",
        ),
    ]
    for comparison, module, endpoint, scale, decimals, unit, supported_text, unsupported_text in s3_specs:
        row = select_one(s3, comparison=comparison)
        effect = scale * float(row["difference_full_minus_ablation"])
        low = scale * float(row["ci_low"])
        high = scale * float(row["ci_high"])
        records.append(
            {
                "Study/module": module,
                "Comparison and endpoint": endpoint,
                "Effect*": f"{_signed(effect, decimals)}{unit}",
                "95% CI": f"{_signed(low, decimals)} to {_signed(high, decimals)}{unit}",
                "Evidence status": supported_text if _as_bool(row["mechanism_support_rule_met"]) else unsupported_text,
            }
        )

    controls = pd.read_csv(studies_tables / "table_s3_negative_controls.csv")
    control_specs = [
        (
            "independent: Full vs No graph",
            "Independent: Full vs No graph; minFDE₆",
            "Compatible with ±0.25-m descriptive range; not TOST",
        ),
        (
            "spatial: Full vs Spatial-only graph",
            "Spatial: Full vs Spatial-only; minFDE₆",
            "Not wholly within descriptive range; not TOST",
        ),
    ]
    for control_name, endpoint, status in control_specs:
        row = select_one(controls, negative_control=control_name)
        effect = float(row["difference_full_minus_ablation"])
        low = float(row["ci_low"])
        high = float(row["ci_high"])
        records.append(
            {
                "Study/module": "S3 neg. control",
                "Comparison and endpoint": endpoint,
                "Effect*": f"{_signed(effect, 4)} m",
                "95% CI": f"{_signed(low, 4)} to {_signed(high, 4)} m",
                "Evidence status": status,
            }
        )

    table = pd.DataFrame.from_records(records)
    if len(table) != 16:
        raise ValueError(f"Table 2 must contain 16 rows; found {len(table)}")
    write_table(table, output_stem)
    return table


def _load_power_tables(power_output: Path) -> dict[str, pd.DataFrame]:
    names = {
        "independent": "table2_independent_power.csv",
        "null": "table3_null_calibration.csv",
        "coverage": "table4_wilson_coverage.csv",
        "paired": "table5_paired_sensitivity.csv",
        "cluster": "table6_cluster_sensitivity.csv",
        "ablation": "table7_ablation_power.csv",
        "event": "table8_event_rate_sensitivity.csv",
        "precision": "table9_precision_intervals.csv",
        "precision_targets": "table10_precision_targets.csv",
        "allocation": "table11_balanced_allocation.csv",
    }
    return {key: pd.read_csv(power_output / "tables" / name) for key, name in names.items()}


def build_table_s2(protocol_path: Path, power_output: Path, output_stem: Path) -> pd.DataFrame:
    """Build compact Supplementary Table S2 from power-analysis outputs."""

    protocol = yaml.safe_load(protocol_path.read_text(encoding="utf-8"))
    summary = json.loads((power_output / "summary.json").read_text(encoding="utf-8"))
    tables = _load_power_tables(power_output)
    independent = tables["independent"]
    rows = {int(row.episodes_per_arm): row for row in independent.itertuples(index=False)}
    if not {150, 450, 749}.issubset(rows):
        raise ValueError("Power table is missing n=150, 450, or 749")
    null_min = float(tables["null"]["estimated_type1_error"].min())
    null_max = float(tables["null"]["estimated_type1_error"].max())
    cover_min = float(tables["coverage"]["estimated_coverage"].min())
    cover_max = float(tables["coverage"]["estimated_coverage"].max())
    max_mcse = max(
        float(independent["mcse"].max()),
        float(tables["null"]["mcse"].max()),
        float(tables["coverage"]["mcse"].max()),
    )
    primary = protocol["primary_design"]
    simulation = protocol["simulation"]
    scenario = protocol["scenario_matrix"]
    data = [
        ("Study status", "Prospective design only", "No empirical CARLA, public-data, real-vehicle, or LLM result"),
        ("Endpoint and estimand", "Episode collision; candidate − control risk difference", "One prespecified confirmatory primary contrast"),
        ("Planning probabilities", f"Control {primary['control_rate_assumption']:.2f}; candidate {primary['candidate_rate_assumption']:.2f}", "Assumed values, not observed rates"),
        ("Error and power targets", f"Two-sided α = {primary['alpha']:.2f}; power = {primary['target_power']:.2f}", "Independent equal-allocation calculation"),
        ("Required independent n", f"{summary['independent_raw_required_n']:.3f}; rounded to {int(summary['independent_required_n_per_arm'])}/condition", f"Analytic power = {rows[749].analytic_power:.4f}"),
        ("Power at 150 / 450 / 749", f"{rows[150].analytic_power:.4f} / {rows[450].analytic_power:.4f} / {rows[749].analytic_power:.4f} analytic", f"MC = {rows[150].mc_power:.4f} / {rows[450].mc_power:.4f} / {rows[749].mc_power:.4f}; {int(rows[150].mc_repetitions):,} trials"),
        ("Calibration range", f"Type-I {null_min:.5f}–{null_max:.5f}; coverage {cover_min:.5f}–{cover_max:.5f}", f"Maximum MCSE = {max_mcse:.6f}"),
        ("Allocation block", f"{scenario['allocation_block']} episodes/condition", f"{scenario['scenario_families']} conflict families × {scenario['environments']} environments × {scenario['traffic_densities']} densities"),
        ("Master seed", str(simulation["master_seed"]), "Deterministic labeled random-number streams"),
    ]
    table = pd.DataFrame(data, columns=["Item", "Value", "Interpretation"])
    write_table(table, output_stem)
    return table


def build_table_s3(power_output: Path, output_stem: Path) -> pd.DataFrame:
    """Build compact Supplementary Table S3 from detailed power tables."""

    tables = _load_power_tables(power_output)
    paired = tables["paired"]
    rho0 = select_one(paired, correlation=0.0)
    pair_min = int(paired["minimum_pairs_exact"].min())
    pair_max = int(paired["minimum_pairs_exact"].max())
    cluster = select_one(tables["cluster"], cluster_size_cv=0.0, icc=0.05)
    small = select_one(
        tables["ablation"],
        scenario="two_events_in_150_gap",
        alpha=0.05,
    )
    multiplicity = select_one(
        tables["ablation"],
        scenario="two_events_in_150_gap",
        alpha=0.0125,
    )
    event80 = tables["event"].loc[tables["event"]["target_power"].eq(0.8)]
    precision150 = select_one(tables["precision"], episodes_per_arm=150, arm="control")
    targets = tables["precision_targets"].sort_values("target_risk_difference_half_width", ascending=False)
    target_text = "/".join(f"{100 * row.target_risk_difference_half_width:.1f}" for row in targets.itertuples(index=False))
    required_text = "/".join(f"{int(row.required_episodes_per_arm):,}" for row in targets.itertuples(index=False))
    allocation = select_one(tables["allocation"], seeds_per_scenario_environment_density_cell=28)
    data = [
        (
            "Exact paired design",
            "ρ grid −0.04 to 0.60; 6% vs 3% marginals",
            f"{pair_min}–{pair_max} pairs; at ρ=0, {int(rho0['minimum_pairs_exact'])}; power at 450={float(rho0['exact_power_at_450_pairs']):.4f}",
            "Joint distributions are assumed",
        ),
        (
            "Provisional cluster-aware plan",
            "ρ=−0.04; mean cluster=9; ICC=0.05; CV=0; 5% invalid; block=45",
            f"{int(cluster['episodes_per_condition']):,}/condition; {int(cluster['total_episodes_primary_two_condition']):,} total; {int(cluster['seeds_per_45_cell_matrix'])} seeds/cell; approx. power={float(cluster['approx_exact_power_at_effective_pairs']):.4f}",
            "Pilot update required",
        ),
        (
            "Illustrative small effect",
            "6.000% vs 4.667%; α=0.05",
            f"{int(small['required_episodes_per_arm']):,}/condition; power at 150={float(small['power_at_150_per_arm']):.4f}",
            "Assumed contrast, not an ablation result",
        ),
        (
            "Conservative multiplicity bound",
            "Same small effect; α=0.0125",
            f"{int(multiplicity['required_episodes_per_arm']):,}/condition",
            "Bonferroni planning bound, not Holm-derived n",
        ),
        (
            "Event/effect grid",
            "Control 3%–15%; reductions 1–4 pp",
            f"{int(event80['required_episodes_per_arm'].min()):,}–{int(event80['required_episodes_per_arm'].max()):,}/condition for 80% power",
            "All rates and reductions are assumed",
        ),
        (
            "Risk-difference precision",
            "6% vs 3% planning probabilities",
            f"Half-width at n=150: {100 * float(precision150['risk_difference_half_width']):.3f} pp; {target_text} pp targets require {required_text}",
            "Illustrative precision only",
        ),
        (
            "Balanced-allocation diagnostic",
            f"{int(allocation['seeds_per_scenario_environment_density_cell'])} seeds/cell = {int(allocation['episodes_per_condition']):,}/condition",
            f"Independent power={float(allocation['independent_power']):.4f}; ICC-only approximation={float(allocation['design_effect_approx_power']):.4f}",
            "Not the primary paired model",
        ),
    ]
    table = pd.DataFrame(data, columns=["Planning component", "Assumption/model", "Result", "Boundary"])
    write_table(table, output_stem)
    return table


def build_table_s4(power_output: Path, output_stem: Path) -> pd.DataFrame:
    """Build compact Supplementary Table S4 from the 33 validation checks."""

    validation = pd.read_csv(power_output / "validation_report.csv")
    table = pd.DataFrame(
        {
            "ID": validation["check_id"].astype(int),
            "Check": validation["check"],
            "Status": validation["passed"].map(lambda value: "Pass" if _as_bool(value) else "Fail"),
            "Evidence": validation["evidence"],
        }
    )
    if len(table) != 33 or not table["Status"].eq("Pass").all():
        raise ValueError("Table S4 must contain 33 passing checks")
    write_table(table, output_stem)
    return table



### `src/egms_publication/utils.py`

In [ ]:
%%writefile src/egms_publication/utils.py
from __future__ import annotations

import hashlib
import json
from pathlib import Path
from typing import Any

import pandas as pd


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def select_one(frame: pd.DataFrame, **filters: object) -> pd.Series:
    mask = pd.Series(True, index=frame.index)
    for column, value in filters.items():
        mask &= frame[column].astype(str).eq(str(value))
    selected = frame.loc[mask]
    if len(selected) != 1:
        raise ValueError(f"Expected one row for {filters}, found {len(selected)}")
    return selected.iloc[0]


def _latex_escape(value: Any) -> str:
    text = "" if pd.isna(value) else str(value)
    for source, replacement in [
        ("\\", r"\textbackslash{}"),
        ("&", r"\&"),
        ("%", r"\%"),
        ("$", r"\$"),
        ("#", r"\#"),
        ("_", r"\_"),
        ("{", r"\{"),
        ("}", r"\}"),
    ]:
        text = text.replace(source, replacement)
    return text


def write_table(frame: pd.DataFrame, stem: Path) -> dict[str, Path]:
    """Write the same data to CSV, Markdown, and dependency-free LaTeX."""

    stem.parent.mkdir(parents=True, exist_ok=True)
    paths = {suffix: stem.with_suffix(f".{suffix}") for suffix in ("csv", "md", "tex")}
    frame.to_csv(paths["csv"], index=False)
    def markdown_cell(value: Any) -> str:
        if pd.isna(value):
            return ""
        return str(value).replace("|", r"\|").replace("\n", " ")

    markdown_lines = [
        "| " + " | ".join(markdown_cell(column) for column in frame.columns) + " |",
        "| " + " | ".join("---" for _ in frame.columns) + " |",
    ]
    markdown_lines.extend(
        "| " + " | ".join(markdown_cell(value) for value in row) + " |"
        for row in frame.itertuples(index=False, name=None)
    )
    paths["md"].write_text("\n".join(markdown_lines) + "\n", encoding="utf-8")
    alignment = "l" * len(frame.columns)
    lines = [f"\\begin{{tabular}}{{{alignment}}}", "\\hline"]
    lines.append(" & ".join(_latex_escape(column) for column in frame.columns) + r" \\")
    lines.append("\\hline")
    for row in frame.itertuples(index=False, name=None):
        lines.append(" & ".join(_latex_escape(value) for value in row) + r" \\")
    lines.extend(["\\hline", "\\end{tabular}", ""])
    paths["tex"].write_text("\n".join(lines), encoding="utf-8")
    return paths


def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


### `src/egms_publication/validation.py`

In [ ]:
%%writefile src/egms_publication/validation.py
from __future__ import annotations

import json
from pathlib import Path
import xml.etree.ElementTree as ET

import pandas as pd
from PIL import Image

from .utils import sha256, write_json


EXPECTED_FIGURES = {
    "manuscript/figures/Figure_2_Study1.png": (3810, 2522),
    "manuscript/figures/Figure_3_Study2.png": (3810, 2472),
    "manuscript/figures/Figure_4_Study3.png": (3810, 2485),
    "power_full/figures/figure1_unpaired_power_curve.png": (3720, 2846),
}

EXPECTED_TABLE_ROWS = {
    "manuscript/tables/Table_2_main_effects.csv": 16,
    "supplement/compact/Tables/Table_S2_planning_summary.csv": 9,
    "supplement/compact/Tables/Table_S3_condensed_planning.csv": 7,
    "supplement/compact/Tables/Table_S4_validation_checks.csv": 33,
}

POWER_FIGURE_STEMS = (
    "figure1_unpaired_power_curve",
    "figure2_paired_correlation",
    "figure3_cluster_sensitivity",
    "figure4_event_rate_sensitivity",
    "figure5_precision_curve",
    "figure6_allocation_by_seed",
)


def validate_input_boundary(data_root: Path, source_root: Path) -> list[str]:
    """Reject image/Word inputs and source code that attempts to read them."""

    failures: list[str] = []
    disallowed = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".pdf", ".svg", ".docx"}
    for path in data_root.rglob("*"):
        if path.is_file() and path.suffix.lower() in disallowed:
            failures.append(f"Disallowed plotting input: {path}")
    for path in source_root.rglob("*.py"):
        text = path.read_text(encoding="utf-8")
        if ("word" + "/media") in text:
            failures.append(f"Source refers to Word media: {path}")
    return failures


def _validate_png(path: Path, expected: tuple[int, int]) -> None:
    with Image.open(path) as image:
        if image.size != expected:
            raise ValueError(f"{path}: expected {expected}, found {image.size}")
        dpi = image.info.get("dpi", (0.0, 0.0))
        if not all(595.0 <= float(value) <= 605.0 for value in dpi[:2]):
            raise ValueError(f"{path}: expected approximately 600 dpi, found {dpi}")


def _validate_siblings(png_path: Path) -> None:
    pdf = png_path.with_suffix(".pdf")
    svg = png_path.with_suffix(".svg")
    pdf_bytes = pdf.read_bytes()
    if len(pdf_bytes) < 1024 or not pdf_bytes.startswith(b"%PDF") or b"%%EOF" not in pdf_bytes[-2048:]:
        raise ValueError(f"Invalid or empty PDF: {pdf}")
    if svg.stat().st_size < 1024:
        raise ValueError(f"Invalid or empty SVG: {svg}")
    ET.parse(svg)


def validate_outputs(repo_root: Path, output_root: Path) -> dict[str, object]:
    failures = validate_input_boundary(repo_root / "data", repo_root / "src" / "egms_publication")
    for relative, dimensions in EXPECTED_FIGURES.items():
        path = output_root / relative
        try:
            _validate_png(path, dimensions)
            _validate_siblings(path)
        except Exception as exc:  # noqa: BLE001 - collect all QA failures
            failures.append(str(exc))
    for stem in POWER_FIGURE_STEMS:
        png = output_root / "power_full" / "figures" / f"{stem}.png"
        try:
            if not png.is_file() or png.stat().st_size < 1024:
                raise ValueError(f"Missing or empty power PNG: {png}")
            _validate_siblings(png)
        except Exception as exc:  # noqa: BLE001 - collect all QA failures
            failures.append(str(exc))
    for relative, expected_rows in EXPECTED_TABLE_ROWS.items():
        path = output_root / relative
        if not path.is_file():
            failures.append(f"Missing table: {path}")
            continue
        rows = len(pd.read_csv(path))
        if rows != expected_rows:
            failures.append(f"{path}: expected {expected_rows} rows, found {rows}")
    table_s4 = output_root / "supplement/compact/Tables/Table_S4_validation_checks.csv"
    if table_s4.is_file() and not pd.read_csv(table_s4)["Status"].eq("Pass").all():
        failures.append("Table S4 contains a non-passing validation check")

    report = {
        "passed": not failures,
        "failures": failures,
        "figure_checks": EXPECTED_FIGURES,
        "table_row_checks": EXPECTED_TABLE_ROWS,
        "power_figure_format_checks": list(POWER_FIGURE_STEMS),
        "interpretation_boundary": (
            "Study 1 is an exported-summary graphical reproduction; Studies 2–3 are "
            "controlled synthetic mechanism surrogates; power results are prospective "
            "planning quantities. None is CARLA or real-world safety evidence."
        ),
    }
    write_json(output_root / "validation_report.json", report)
    if failures:
        raise RuntimeError("Publication validation failed:\n- " + "\n- ".join(failures))
    return report


def write_manifest(repo_root: Path, output_root: Path) -> Path:
    input_files = sorted(
        path for base in (repo_root / "configs", repo_root / "data")
        for path in base.rglob("*") if path.is_file()
    )
    output_files = sorted(
        path for path in output_root.rglob("*")
        if path.is_file() and path.name not in {"artifact_manifest.json", "publication_outputs.zip"}
    )
    payload = {
        "schema": "egms-publication-artifact-manifest-1.0",
        "inputs": {
            str(path.relative_to(repo_root)): {
                "bytes": path.stat().st_size,
                "sha256": sha256(path),
            }
            for path in input_files
        },
        "outputs": {
            str(path.relative_to(output_root)): {
                "bytes": path.stat().st_size,
                "sha256": sha256(path),
            }
            for path in output_files
        },
    }
    manifest = output_root / "artifact_manifest.json"
    write_json(manifest, payload)
    return manifest


def verify_manifest(output_root: Path, manifest: Path) -> None:
    payload = json.loads(manifest.read_text(encoding="utf-8"))
    for relative, expected in payload["outputs"].items():
        path = output_root / relative
        if not path.is_file() or path.stat().st_size != expected["bytes"] or sha256(path) != expected["sha256"]:
            raise RuntimeError(f"Manifest mismatch: {relative}")


### `src/egms_power/__init__.py`

In [ ]:
%%writefile src/egms_power/__init__.py
"""Prospective protocol and power-analysis utilities for EGMS-Drive."""

from .statistics import (
    exact_mcnemar_power,
    find_minimum_exact_mcnemar_n,
    paired_joint_probabilities,
    required_n_two_independent_proportions,
    two_independent_proportions_power,
)

__all__ = [
    "exact_mcnemar_power",
    "find_minimum_exact_mcnemar_n",
    "paired_joint_probabilities",
    "required_n_two_independent_proportions",
    "two_independent_proportions_power",
]

__version__ = "1.1.0"


### `src/egms_power/cli.py`

In [ ]:
%%writefile src/egms_power/cli.py
"""Command-line entry point."""

from __future__ import annotations

import argparse
import json
from pathlib import Path

from .config import load_protocol


def project_root() -> Path:
    return Path(__file__).resolve().parents[2]


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        description="Run the EGMS-Drive prospective protocol power analysis."
    )
    subparsers = parser.add_subparsers(dest="command", required=True)

    validate = subparsers.add_parser("validate", help="validate protocol YAML")
    validate.add_argument("--config", default="configs/protocol.yaml")

    run = subparsers.add_parser("run", help="generate all results, tables, and figures")
    run.add_argument("--config", default="configs/protocol.yaml")
    run.add_argument("--output", default="outputs")
    return parser


def _resolve_config(value: str) -> Path:
    path = Path(value)
    if path.is_absolute():
        return path
    return project_root() / path


def main(argv: list[str] | None = None) -> int:
    args = build_parser().parse_args(argv)
    config = _resolve_config(args.config)
    if args.command == "validate":
        protocol = load_protocol(config)
        print(
            json.dumps(
                {
                    "status": "valid",
                    "version": protocol["project"]["version"],
                    "protocol_sha256": protocol["_meta"]["sha256"],
                },
                indent=2,
            )
        )
        return 0
    from .pipeline import run_power_analysis

    result = run_power_analysis(config, args.output)
    print(json.dumps({"status": "completed", **result["summary"], "validation": result["validation"]}, indent=2))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### `src/egms_power/config.py`

In [ ]:
%%writefile src/egms_power/config.py
"""Protocol configuration loading and validation."""

from __future__ import annotations

import hashlib
import json
from pathlib import Path
from typing import Any

import yaml

from .statistics import feasible_correlation_bounds


DISCLAIMER = (
    "All numerical outputs are conditional design quantities generated under "
    "prespecified event-rate and dependence assumptions. They are not empirical "
    "performance estimates of EGMS-Drive or any implemented driving system."
)


def load_protocol(path: str | Path) -> dict[str, Any]:
    """Load and validate a YAML protocol."""

    protocol_path = Path(path).expanduser().resolve()
    if not protocol_path.is_file():
        raise FileNotFoundError(f"protocol file not found: {protocol_path}")
    with protocol_path.open("r", encoding="utf-8") as handle:
        protocol = yaml.safe_load(handle)
    if not isinstance(protocol, dict):
        raise ValueError("protocol must be a YAML mapping")
    errors = validate_protocol(protocol)
    if errors:
        raise ValueError("Invalid protocol:\n- " + "\n- ".join(errors))
    protocol["_meta"] = {
        "path": str(protocol_path),
        "sha256": hashlib.sha256(protocol_path.read_bytes()).hexdigest(),
    }
    return protocol


def _require(mapping: dict[str, Any], path: str) -> Any:
    current: Any = mapping
    for part in path.split("."):
        if not isinstance(current, dict) or part not in current:
            raise KeyError(path)
        current = current[part]
    return current


def validate_protocol(protocol: dict[str, Any]) -> list[str]:
    """Return protocol-validation errors; an empty list means valid."""

    errors: list[str] = []
    required = [
        "project.version",
        "project.status",
        "project.empirical_data_present",
        "project.empirical_performance_claims",
        "primary_design.endpoint",
        "primary_design.estimand",
        "primary_design.control_rate_assumption",
        "primary_design.candidate_rate_assumption",
        "primary_design.alpha",
        "primary_design.target_power",
        "simulation.master_seed",
        "simulation.canonical_repetitions",
        "paired_design.correlations",
        "clustered_design.mean_cluster_size",
        "clustered_design.icc_values",
        "scenario_matrix.allocation_block",
    ]
    values: dict[str, Any] = {}
    for key in required:
        try:
            values[key] = _require(protocol, key)
        except KeyError:
            errors.append(f"missing required key: {key}")
    if errors:
        return errors

    if values["project.status"] != "prospective_design_only":
        errors.append("project.status must be 'prospective_design_only'")
    if values["project.empirical_data_present"] is not False:
        errors.append("project.empirical_data_present must be false")
    if values["project.empirical_performance_claims"] is not False:
        errors.append("project.empirical_performance_claims must be false")

    p0 = values["primary_design.control_rate_assumption"]
    p1 = values["primary_design.candidate_rate_assumption"]
    alpha = values["primary_design.alpha"]
    power = values["primary_design.target_power"]
    for name, value in [("control rate", p0), ("candidate rate", p1)]:
        if not isinstance(value, (int, float)) or not 0 < value < 1:
            errors.append(f"{name} must be numeric and in (0, 1)")
    if isinstance(p0, (int, float)) and isinstance(p1, (int, float)) and p0 == p1:
        errors.append("control and candidate rate assumptions must differ")
    if not isinstance(alpha, (int, float)) or not 0 < alpha < 1:
        errors.append("alpha must be in (0, 1)")
    if not isinstance(power, (int, float)) or not 0 < power < 1:
        errors.append("target_power must be in (0, 1)")

    repetitions = values["simulation.canonical_repetitions"]
    if not isinstance(repetitions, int) or repetitions < 10_000:
        errors.append("canonical_repetitions must be an integer >= 10000")
    if not isinstance(values["simulation.master_seed"], int):
        errors.append("master_seed must be an integer")

    if (
        isinstance(p0, (int, float))
        and isinstance(p1, (int, float))
        and 0 < p0 < 1
        and 0 < p1 < 1
    ):
        lower, upper = feasible_correlation_bounds(float(p0), float(p1))
        correlations = values["paired_design.correlations"]
        if not isinstance(correlations, list) or not correlations:
            errors.append("paired_design.correlations must be a non-empty list")
        else:
            for correlation in correlations:
                if not isinstance(correlation, (int, float)):
                    errors.append("all paired correlations must be numeric")
                elif not lower <= correlation <= upper:
                    errors.append(
                        f"paired correlation {correlation} is outside feasible "
                        f"range [{lower:.6f}, {upper:.6f}]"
                    )

    mean_cluster_size = values["clustered_design.mean_cluster_size"]
    if not isinstance(mean_cluster_size, (int, float)) or mean_cluster_size <= 0:
        errors.append("mean_cluster_size must be positive")
    iccs = values["clustered_design.icc_values"]
    if not isinstance(iccs, list) or not iccs:
        errors.append("clustered_design.icc_values must be a non-empty list")
    else:
        for icc in iccs:
            if not isinstance(icc, (int, float)) or not 0 <= icc < 1:
                errors.append("all ICC values must be numeric and in [0, 1)")

    block = values["scenario_matrix.allocation_block"]
    if not isinstance(block, int) or block <= 0:
        errors.append("scenario_matrix.allocation_block must be a positive integer")
    return errors


def canonical_protocol_json(protocol: dict[str, Any]) -> str:
    """Serialize protocol content without local metadata for hashing."""

    payload = {key: value for key, value in protocol.items() if key != "_meta"}
    return json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=False)


### `src/egms_power/pipeline.py`

In [ ]:
%%writefile src/egms_power/pipeline.py
"""End-to-end analysis, reporting, plotting, and provenance pipeline."""

from __future__ import annotations

import hashlib
import json
import math
import os
import platform
import sys
import xml.etree.ElementTree as ET
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import seaborn as sns

from . import __version__
from .config import DISCLAIMER, canonical_protocol_json, load_protocol
from .statistics import (
    design_effect,
    exact_mcnemar_power,
    feasible_correlation_bounds,
    find_minimum_exact_mcnemar_n,
    minimum_n_for_risk_difference_precision,
    normal_risk_difference_half_width,
    paired_joint_probabilities,
    representative_wilson_interval,
    required_n_two_independent_proportions,
    round_up_to_block,
    simulate_paired_power,
    simulate_unpaired_power,
    simulate_wilson_coverage,
    stable_stream_seed,
    two_independent_proportions_power,
)


def project_root() -> Path:
    return Path(__file__).resolve().parents[2]


def _generated_at_utc() -> str:
    """Return a standard reproducible-build timestamp when one is requested."""

    source_date_epoch = os.environ.get("SOURCE_DATE_EPOCH")
    if source_date_epoch is None:
        return datetime.now(timezone.utc).isoformat()
    try:
        timestamp = int(source_date_epoch)
    except ValueError as exc:
        raise ValueError("SOURCE_DATE_EPOCH must be an integer number of seconds") from exc
    return datetime.fromtimestamp(timestamp, timezone.utc).isoformat()


def prepare_output_directory(root: Path, requested: str | Path) -> Path:
    """Prepare a contained output directory without recursively deleting data."""

    root = root.resolve()
    output = Path(requested)
    if not output.is_absolute():
        output = (root / output).resolve()
    else:
        output = output.resolve()
    if output == root or root not in output.parents:
        raise ValueError(
            "Refusing unsafe output path. Output must be a child of the project root."
        )
    sentinel = output / ".egms_power_output"
    existing_files = [path for path in output.rglob("*") if path.is_file()] if output.exists() else []
    if existing_files and not sentinel.exists():
        raise ValueError(
            "Refusing to overwrite a non-empty directory without the EGMS power-analysis sentinel."
        )
    # Remove only files recorded by the prior run manifest. This prevents stale
    # scientific tables while preserving untracked user files.
    prior_manifest = output / "run_manifest.json"
    if sentinel.exists() and prior_manifest.is_file():
        try:
            recorded = json.loads(prior_manifest.read_text(encoding="utf-8")).get(
                "output_files", {}
            )
        except (json.JSONDecodeError, OSError):
            recorded = {}
        for relative in recorded:
            candidate = (root / relative).resolve()
            if output in candidate.parents and candidate.is_file():
                candidate.unlink()
        prior_manifest.unlink(missing_ok=True)
    for subdirectory in ["tables", "figures", "data"]:
        (output / subdirectory).mkdir(parents=True, exist_ok=True)
    sentinel.write_text(
        "Managed EGMS-Drive protocol power-analysis output directory.\n",
        encoding="utf-8",
    )
    return output


def _save_table(frame: pd.DataFrame, stem: Path) -> None:
    frame.to_csv(stem.with_suffix(".csv"), index=False)
    def cell(value: object) -> str:
        if pd.isna(value):
            return ""
        text = f"{value:.6f}" if isinstance(value, (float, np.floating)) else str(value)
        return text.replace("|", r"\|").replace("\n", " ")

    lines = [
        "| " + " | ".join(cell(column) for column in frame.columns) + " |",
        "| " + " | ".join("---" for _ in frame.columns) + " |",
    ]
    lines.extend(
        "| " + " | ".join(cell(value) for value in row) + " |"
        for row in frame.itertuples(index=False, name=None)
    )
    stem.with_suffix(".md").write_text("\n".join(lines) + "\n", encoding="utf-8")
    stem.with_suffix(".tex").write_text(_simple_latex_table(frame), encoding="utf-8")


def _frame_to_markdown(frame: pd.DataFrame, *, float_digits: int | None = None) -> str:
    """Render a DataFrame as Markdown without the optional tabulate package."""

    def cell(value: object) -> str:
        if pd.isna(value):
            return ""
        if float_digits is not None and isinstance(value, (float, np.floating)):
            text = f"{value:.{float_digits}f}"
        else:
            text = str(value)
        return text.replace("|", r"\|").replace("\n", " ")

    lines = [
        "| " + " | ".join(cell(column) for column in frame.columns) + " |",
        "| " + " | ".join("---" for _ in frame.columns) + " |",
    ]
    lines.extend(
        "| " + " | ".join(cell(value) for value in row) + " |"
        for row in frame.itertuples(index=False, name=None)
    )
    return "\n".join(lines)


def _latex_escape(value: object) -> str:
    text = f"{value:.6f}" if isinstance(value, (float, np.floating)) else str(value)
    for source, replacement in [
        ("\\", r"\textbackslash{}"),
        ("&", r"\&"),
        ("%", r"\%"),
        ("$", r"\$"),
        ("#", r"\#"),
        ("_", r"\_"),
        ("{", r"\{"),
        ("}", r"\}"),
    ]:
        text = text.replace(source, replacement)
    return text


def _simple_latex_table(frame: pd.DataFrame) -> str:
    """Create dependency-free tabular LaTeX for release tables."""

    alignment = "l" * len(frame.columns)
    lines = [f"\\begin{{tabular}}{{{alignment}}}", "\\hline"]
    lines.append(" & ".join(_latex_escape(column) for column in frame.columns) + r" \\")
    lines.append("\\hline")
    for row in frame.itertuples(index=False, name=None):
        lines.append(" & ".join(_latex_escape(value) for value in row) + r" \\")
    lines.extend(["\\hline", "\\end{tabular}", ""])
    return "\n".join(lines)


def _style() -> None:
    """Apply a deterministic IEEE-style graphics profile.

    STIXGeneral is distributed with Matplotlib, has a Times-like appearance,
    and is therefore reproducible in both the release environment and Colab.
    Plot titles and prose footnotes are deliberately omitted; the manuscript
    captions carry all descriptive and interpretive text.
    """

    sns.set_theme(style="white", context="paper")
    plt.rcParams.update(
        {
            "font.family": "STIXGeneral",
            "mathtext.fontset": "stix",
            "font.size": 10.5,
            "axes.labelsize": 12.5,
            "axes.labelpad": 6,
            "axes.linewidth": 0.9,
            "axes.edgecolor": "black",
            "axes.grid": False,
            "axes.axisbelow": True,
            "grid.color": "#D6D6D6",
            "grid.linestyle": "-",
            "grid.linewidth": 0.6,
            "grid.alpha": 0.70,
            "xtick.labelsize": 10.5,
            "ytick.labelsize": 10.5,
            "xtick.direction": "out",
            "ytick.direction": "out",
            "xtick.major.size": 4.5,
            "ytick.major.size": 4.5,
            "xtick.major.width": 0.9,
            "ytick.major.width": 0.9,
            "xtick.major.pad": 4,
            "ytick.major.pad": 4,
            "legend.fontsize": 9.5,
            "legend.frameon": False,
            "figure.facecolor": "white",
            "axes.facecolor": "white",
            "figure.dpi": 120,
            "savefig.dpi": 600,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "svg.fonttype": "none",
            "svg.hashsalt": "egms-drive-publication-v1",
        }
    )


def _save_figure(fig: plt.Figure, stem: Path) -> None:
    paths: dict[str, Path] = {}
    try:
        for suffix, kwargs in [
            ("png", {"dpi": 600}),
            ("pdf", {}),
            ("svg", {}),
        ]:
            path = stem.with_suffix(f".{suffix}")
            temporary = path.with_name(path.name + ".tmp")
            fig.savefig(temporary, format=suffix, facecolor="white", **kwargs)
            paths[suffix] = temporary

        png, pdf, svg = (paths[key] for key in ("png", "pdf", "svg"))
        if png.stat().st_size < 1024 or not png.read_bytes().startswith(b"\x89PNG"):
            raise RuntimeError(f"Invalid PNG figure export: {png}")
        pdf_bytes = pdf.read_bytes()
        if len(pdf_bytes) < 1024 or not pdf_bytes.startswith(b"%PDF") or b"%%EOF" not in pdf_bytes[-2048:]:
            raise RuntimeError(f"Invalid PDF figure export: {pdf}")
        if svg.stat().st_size < 1024:
            raise RuntimeError(f"Invalid or empty SVG figure export: {svg}")
        ET.parse(svg)
        for suffix, temporary in paths.items():
            temporary.replace(stem.with_suffix(f".{suffix}"))
    finally:
        plt.close(fig)
        for temporary in paths.values():
            temporary.unlink(missing_ok=True)


def _format_axes(ax: plt.Axes, *, show_grid: bool = True) -> None:
    """Apply a clean boxed-axis treatment with consistent major ticks."""

    ax.grid(show_grid, which="major", axis="both")
    ax.tick_params(
        axis="both",
        which="major",
        top=False,
        right=False,
        bottom=True,
        left=True,
        labelsize=10.5,
    )
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(0.9)


def _format_colorbar(ax: plt.Axes) -> None:
    """Match colorbar labels and numerical ticks to the enlarged axis text."""

    colorbar = ax.collections[0].colorbar
    if colorbar is None:
        return
    colorbar.ax.tick_params(labelsize=10.5, length=4.5, width=0.9, direction="out")
    colorbar.ax.yaxis.label.set_size(12.5)


def _assumptions_table(protocol: dict[str, Any]) -> pd.DataFrame:
    primary = protocol["primary_design"]
    simulation = protocol["simulation"]
    scenario = protocol["scenario_matrix"]
    return pd.DataFrame(
        [
            ("Study status", protocol["project"]["status"], "Prospective design only"),
            ("Primary endpoint", primary["endpoint"], primary["endpoint_definition"]),
            ("Estimand", primary["estimand"], "Risk difference"),
            ("Planning control probability", primary["control_rate_assumption"], "Assumed, not observed"),
            ("Planning candidate probability", primary["candidate_rate_assumption"], "Assumed, not observed"),
            ("Two-sided alpha", primary["alpha"], "One confirmatory primary contrast"),
            ("Target power", primary["target_power"], "Prospective design target"),
            ("Monte Carlo repetitions", simulation["canonical_repetitions"], "Complete simulated trials"),
            ("Master seed", simulation["master_seed"], "Deterministic labeled RNG streams"),
            ("Scenario allocation block", scenario["allocation_block"], "5 scenarios × 3 environments × 3 densities"),
        ],
        columns=["quantity", "value", "interpretation"],
    )


def _independent_results(protocol: dict[str, Any]) -> tuple[pd.DataFrame, dict[str, float]]:
    primary = protocol["primary_design"]
    simulation = protocol["simulation"]
    p0 = float(primary["control_rate_assumption"])
    p1 = float(primary["candidate_rate_assumption"])
    alpha = float(primary["alpha"])
    target = float(primary["target_power"])
    repetitions = int(simulation["canonical_repetitions"])
    master_seed = int(simulation["master_seed"])
    raw_n, required_n = required_n_two_independent_proportions(
        p0, p1, alpha=alpha, target_power=target
    )
    rows: list[dict[str, Any]] = []
    for n in simulation["sample_sizes"]:
        analytic = two_independent_proportions_power(n, p0, p1, alpha=alpha)
        estimate = simulate_unpaired_power(
            int(n),
            p0,
            p1,
            alpha=alpha,
            repetitions=repetitions,
            seed=stable_stream_seed(master_seed, ["unpaired", n, p0, p1, alpha]),
        )
        rows.append(
            {
                "episodes_per_arm": int(n),
                "analytic_power": analytic,
                "mc_power": estimate.estimate,
                "mcse": estimate.mcse,
                "mc_ci_low": estimate.ci_low,
                "mc_ci_high": estimate.ci_high,
                "mc_rejections": estimate.rejections,
                "mc_repetitions": estimate.repetitions,
            }
        )
    summary = {
        "raw_required_n": raw_n,
        "required_n": required_n,
        "power_150": two_independent_proportions_power(150, p0, p1, alpha=alpha),
        "power_450": two_independent_proportions_power(450, p0, p1, alpha=alpha),
        "power_required": two_independent_proportions_power(required_n, p0, p1, alpha=alpha),
    }
    return pd.DataFrame(rows), summary


def _null_calibration(protocol: dict[str, Any]) -> pd.DataFrame:
    primary = protocol["primary_design"]
    simulation = protocol["simulation"]
    p_null = float(primary["control_rate_assumption"])
    alpha = float(primary["alpha"])
    repetitions = int(simulation["canonical_repetitions"])
    master_seed = int(simulation["master_seed"])
    sample_sizes = [150, 450, 749, 900, 1200]
    rows: list[dict[str, Any]] = []
    for n in sample_sizes:
        estimate = simulate_unpaired_power(
            n,
            p_null,
            p_null,
            alpha=alpha,
            repetitions=repetitions,
            seed=stable_stream_seed(master_seed, ["null", n, p_null, alpha]),
        )
        rows.append(
            {
                "episodes_per_arm": n,
                "null_rate_both_arms": p_null,
                "nominal_alpha": alpha,
                "estimated_type1_error": estimate.estimate,
                "mcse": estimate.mcse,
                "mc_ci_low": estimate.ci_low,
                "mc_ci_high": estimate.ci_high,
                "within_3mcse_plus_0_005": abs(estimate.estimate - alpha)
                <= 3.0 * estimate.mcse + 0.005,
            }
        )
    return pd.DataFrame(rows)


def _coverage_calibration(protocol: dict[str, Any]) -> pd.DataFrame:
    primary = protocol["primary_design"]
    simulation = protocol["simulation"]
    repetitions = int(simulation["canonical_repetitions"])
    master_seed = int(simulation["master_seed"])
    confidence = float(simulation["monte_carlo_confidence"])
    rates = [
        float(primary["candidate_rate_assumption"]),
        float(primary["control_rate_assumption"]),
    ]
    rows: list[dict[str, Any]] = []
    for rate in rates:
        for n in [150, 450, 749, 1200]:
            estimate = simulate_wilson_coverage(
                n,
                rate,
                confidence=confidence,
                repetitions=repetitions,
                seed=stable_stream_seed(master_seed, ["coverage", rate, n, confidence]),
            )
            rows.append(
                {
                    "true_probability": rate,
                    "episodes": n,
                    "nominal_coverage": confidence,
                    "estimated_coverage": estimate.estimate,
                    "mcse": estimate.mcse,
                    "mc_ci_low": estimate.ci_low,
                    "mc_ci_high": estimate.ci_high,
                    "coverage_in_prespecified_range": 0.93 <= estimate.estimate <= 0.98,
                }
            )
    return pd.DataFrame(rows)


def _paired_results(protocol: dict[str, Any]) -> tuple[pd.DataFrame, tuple[float, float]]:
    primary = protocol["primary_design"]
    paired = protocol["paired_design"]
    simulation = protocol["simulation"]
    p0 = float(primary["control_rate_assumption"])
    p1 = float(primary["candidate_rate_assumption"])
    alpha = float(primary["alpha"])
    target = float(primary["target_power"])
    repetitions = int(simulation["canonical_repetitions"])
    master_seed = int(simulation["master_seed"])
    bounds = feasible_correlation_bounds(p0, p1)
    rows: list[dict[str, Any]] = []
    for correlation in paired["correlations"]:
        correlation = float(correlation)
        cells = paired_joint_probabilities(p0, p1, correlation)
        minimum_n, power_at_minimum = find_minimum_exact_mcnemar_n(
            p0,
            p1,
            correlation,
            alpha=alpha,
            target_power=target,
        )
        mc = simulate_paired_power(
            minimum_n,
            p0,
            p1,
            correlation,
            alpha=alpha,
            repetitions=repetitions,
            seed=stable_stream_seed(
                master_seed, ["paired", correlation, minimum_n, p0, p1, alpha]
            ),
        )
        rows.append(
            {
                "correlation": correlation,
                "pi00_both_safe": cells["both_safe"],
                "pi01_candidate_only_collision": cells["candidate_only"],
                "pi10_control_only_collision": cells["control_only"],
                "pi11_both_collision": cells["both_collision"],
                "discordant_probability": cells["candidate_only"] + cells["control_only"],
                "exact_power_at_450_pairs": exact_mcnemar_power(
                    450, p0, p1, correlation, alpha=alpha
                ),
                "minimum_pairs_exact": minimum_n,
                "exact_power_at_minimum": power_at_minimum,
                "mc_power_at_minimum": mc.estimate,
                "mcse": mc.mcse,
                "mc_ci_low": mc.ci_low,
                "mc_ci_high": mc.ci_high,
            }
        )
    return pd.DataFrame(rows), bounds


def _cluster_results(
    protocol: dict[str, Any], paired_frame: pd.DataFrame
) -> tuple[pd.DataFrame, dict[str, Any]]:
    primary = protocol["primary_design"]
    paired = protocol["paired_design"]
    clustered = protocol["clustered_design"]
    scenario = protocol["scenario_matrix"]
    p0 = float(primary["control_rate_assumption"])
    p1 = float(primary["candidate_rate_assumption"])
    alpha = float(primary["alpha"])
    planning_correlation = float(paired["planning_correlation"])
    row = paired_frame.loc[
        np.isclose(paired_frame["correlation"], planning_correlation)
    ]
    if row.empty:
        base_pairs, _ = find_minimum_exact_mcnemar_n(
            p0,
            p1,
            planning_correlation,
            alpha=alpha,
            target_power=float(primary["target_power"]),
        )
    else:
        base_pairs = int(row.iloc[0]["minimum_pairs_exact"])
    mean_cluster = float(clustered["mean_cluster_size"])
    invalid_fraction = float(clustered["invalid_episode_fraction"])
    block = int(scenario["allocation_block"])
    rows: list[dict[str, Any]] = []
    for cv in clustered["cluster_size_cv_values"]:
        for icc in clustered["icc_values"]:
            effect = design_effect(mean_cluster, float(icc), float(cv))
            raw_planned = base_pairs * effect / (1.0 - invalid_fraction)
            planned = round_up_to_block(raw_planned, block)
            effective_pairs = planned * (1.0 - invalid_fraction) / effect
            rows.append(
                {
                    "cluster_size_cv": float(cv),
                    "icc": float(icc),
                    "design_effect": effect,
                    "independent_equivalent_pairs_required": base_pairs,
                    "raw_pairs_with_clustering_and_invalid_allowance": raw_planned,
                    "planned_pairs": planned,
                    "episodes_per_condition": planned,
                    "total_episodes_primary_two_condition": 2 * planned,
                    "seeds_per_45_cell_matrix": planned // block,
                    "effective_pairs_after_adjustment": effective_pairs,
                    "approx_exact_power_at_effective_pairs": exact_mcnemar_power(
                        max(1, int(math.floor(effective_pairs))),
                        p0,
                        p1,
                        planning_correlation,
                        alpha=alpha,
                    ),
                }
            )
    frame = pd.DataFrame(rows)
    selected = frame[
        np.isclose(frame["cluster_size_cv"], 0.0)
        & np.isclose(frame["icc"], float(clustered["planning_icc"]))
    ].iloc[0]
    recommendation = {
        "planning_correlation": planning_correlation,
        "planning_icc": float(clustered["planning_icc"]),
        "cluster_size_cv": 0.0,
        "invalid_episode_fraction": invalid_fraction,
        "base_exact_pairs": base_pairs,
        "planned_pairs": int(selected["planned_pairs"]),
        "episodes_per_condition": int(selected["episodes_per_condition"]),
        "total_episodes_primary_two_condition": int(selected["total_episodes_primary_two_condition"]),
        "seeds_per_matrix_cell": int(selected["seeds_per_45_cell_matrix"]),
    }
    return frame, recommendation


def _ablation_results(protocol: dict[str, Any]) -> pd.DataFrame:
    primary = protocol["primary_design"]
    ablation = protocol["ablation_planning"]
    alpha_primary = float(primary["alpha"])
    alpha_multiplicity = float(protocol["multiplicity"]["conservative_ablation_alpha"])
    target = float(primary["target_power"])
    rows: list[dict[str, Any]] = []
    for scenario in ablation["collision_scenarios"]:
        p0 = float(scenario["control_rate"])
        p1 = float(scenario["candidate_rate"])
        for alpha, label in [
            (alpha_primary, "single_prespecified_contrast"),
            (alpha_multiplicity, "four_comparison_bonferroni_planning_bound"),
        ]:
            raw_n, n = required_n_two_independent_proportions(
                p0, p1, alpha=alpha, target_power=target
            )
            rows.append(
                {
                    "scenario": scenario["label"],
                    "control_rate_assumption": p0,
                    "candidate_rate_assumption": p1,
                    "absolute_difference": abs(p0 - p1),
                    "alpha": alpha,
                    "multiplicity_interpretation": label,
                    "raw_required_episodes_per_arm": raw_n,
                    "required_episodes_per_arm": n,
                    "power_at_150_per_arm": two_independent_proportions_power(
                        150, p0, p1, alpha=alpha
                    ),
                    "power_at_450_per_arm": two_independent_proportions_power(
                        450, p0, p1, alpha=alpha
                    ),
                }
            )
    return pd.DataFrame(rows)


def _sensitivity_results(protocol: dict[str, Any]) -> pd.DataFrame:
    primary = protocol["primary_design"]
    sensitivity = protocol["sensitivity"]
    alpha = float(primary["alpha"])
    rows: list[dict[str, Any]] = []
    for control_rate in sensitivity["control_rates"]:
        for reduction in sensitivity["absolute_reductions"]:
            candidate_rate = float(control_rate) - float(reduction)
            if candidate_rate <= 0:
                continue
            for target_power in sensitivity["target_powers"]:
                raw_n, n = required_n_two_independent_proportions(
                    float(control_rate),
                    candidate_rate,
                    alpha=alpha,
                    target_power=float(target_power),
                )
                rows.append(
                    {
                        "control_rate_assumption": float(control_rate),
                        "candidate_rate_assumption": candidate_rate,
                        "absolute_reduction": float(reduction),
                        "target_power": float(target_power),
                        "raw_required_episodes_per_arm": raw_n,
                        "required_episodes_per_arm": n,
                    }
                )
    return pd.DataFrame(rows)


def _precision_results(protocol: dict[str, Any]) -> tuple[pd.DataFrame, pd.DataFrame]:
    primary = protocol["primary_design"]
    sensitivity = protocol["sensitivity"]
    p0 = float(primary["control_rate_assumption"])
    p1 = float(primary["candidate_rate_assumption"])
    rows: list[dict[str, Any]] = []
    for n in sensitivity["representative_ci_sample_sizes"]:
        for label, rate in [("control", p0), ("candidate", p1)]:
            events, low, high, width = representative_wilson_interval(int(n), rate)
            rows.append(
                {
                    "episodes_per_arm": int(n),
                    "arm": label,
                    "assumed_rate": rate,
                    "representative_event_count": events,
                    "wilson_ci_low": low,
                    "wilson_ci_high": high,
                    "wilson_ci_width": width,
                    "risk_difference_half_width": normal_risk_difference_half_width(
                        int(n), p0, p1
                    ),
                }
            )
    precision_rows = []
    for half_width in sensitivity["precision_half_widths"]:
        precision_rows.append(
            {
                "target_risk_difference_half_width": float(half_width),
                "required_episodes_per_arm": minimum_n_for_risk_difference_precision(
                    float(half_width), p0, p1
                ),
            }
        )
    return pd.DataFrame(rows), pd.DataFrame(precision_rows)


def _allocation_results(protocol: dict[str, Any]) -> pd.DataFrame:
    primary = protocol["primary_design"]
    clustered = protocol["clustered_design"]
    scenario = protocol["scenario_matrix"]
    p0 = float(primary["control_rate_assumption"])
    p1 = float(primary["candidate_rate_assumption"])
    alpha = float(primary["alpha"])
    block = int(scenario["allocation_block"])
    effect = design_effect(
        float(clustered["mean_cluster_size"]),
        float(clustered["planning_icc"]),
        0.0,
    )
    rows: list[dict[str, Any]] = []
    for seeds in range(3, 41):
        total = block * seeds
        effective = total / effect
        rows.append(
            {
                "seeds_per_scenario_environment_density_cell": seeds,
                "episodes_per_condition": total,
                "independent_power": two_independent_proportions_power(
                    total, p0, p1, alpha=alpha
                ),
                "effective_n_at_planning_icc": effective,
                "design_effect_approx_power": two_independent_proportions_power(
                    effective, p0, p1, alpha=alpha
                ),
            }
        )
    return pd.DataFrame(rows)


def _plot_all(
    output: Path,
    protocol: dict[str, Any],
    independent: pd.DataFrame,
    paired: pd.DataFrame,
    clustered: pd.DataFrame,
    sensitivity: pd.DataFrame,
    precision: pd.DataFrame,
    allocation: pd.DataFrame,
) -> None:
    _style()
    primary = protocol["primary_design"]
    simulation = protocol["simulation"]
    p0 = float(primary["control_rate_assumption"])
    p1 = float(primary["candidate_rate_assumption"])
    alpha = float(primary["alpha"])
    target = float(primary["target_power"])

    grid = np.arange(
        int(simulation["power_curve_start"]),
        int(simulation["power_curve_stop"]) + 1,
        int(simulation["power_curve_step"]),
    )
    power_grid = pd.DataFrame(
        {
            "episodes_per_arm": grid,
            "analytic_power": [
                two_independent_proportions_power(n, p0, p1, alpha=alpha) for n in grid
            ],
        }
    )
    power_grid.to_csv(output / "data" / "figure1_unpaired_power_curve.csv", index=False)
    fig, ax = plt.subplots(figsize=(6.20, 4.744))
    fig.subplots_adjust(left=0.165, right=0.975, bottom=0.175, top=0.965)
    ax.plot(grid, power_grid["analytic_power"], color="#174A7E", lw=2.2, label="Analytic power")
    ax.scatter(
        independent["episodes_per_arm"],
        independent["mc_power"],
        color="#D1495B",
        s=28,
        zorder=3,
        label="Monte Carlo estimate",
    )
    ax.axhline(target, color="#2A9D8F", ls="--", lw=1.4, label="80% target")
    for n in [150, 450, 749]:
        power = two_independent_proportions_power(n, p0, p1, alpha=alpha)
        ax.annotate(
            f"{n}: {power:.3f}",
            (n, power),
            xytext=(5, 8),
            textcoords="offset points",
            fontsize=10,
        )
    ax.set(
        xlabel=r"Number of episodes per arm, $n$",
        ylabel=r"Statistical power, $1-\beta$",
        xlim=(0, 2000),
        ylim=(0, 1.02),
    )
    ax.set_xticks(np.arange(0, 2001, 250))
    ax.set_yticks(np.arange(0, 1.01, 0.2))
    _format_axes(ax)
    ax.legend(loc="lower right")
    _save_figure(fig, output / "figures" / "figure1_unpaired_power_curve")

    paired.to_csv(output / "data" / "figure2_paired_correlation.csv", index=False)
    fig, ax1 = plt.subplots(figsize=(6.24, 4.744))
    fig.subplots_adjust(left=0.215, right=0.975, bottom=0.175, top=0.965)
    ax1.plot(
        paired["correlation"],
        paired["minimum_pairs_exact"],
        marker="o",
        color="#6A4C93",
        lw=2.2,
        ms=4.5,
    )
    ax1.set(
        xlabel=r"Cross-method collision correlation, $\rho$",
        ylabel=r"Minimum number of pairs for 80% power, $n$",
    )
    correlation_ticks = paired["correlation"].to_numpy()
    ax1.set_xticks(correlation_ticks)
    ax1.set_xticklabels([f"{value:g}" for value in correlation_ticks])
    _format_axes(ax1)
    _save_figure(fig, output / "figures" / "figure2_paired_correlation")

    clustered.to_csv(output / "data" / "figure3_cluster_sensitivity.csv", index=False)
    heat = clustered.pivot(
        index="icc", columns="cluster_size_cv", values="episodes_per_condition"
    )
    fig, ax = plt.subplots(figsize=(5.82, 4.838))
    fig.subplots_adjust(left=0.21, right=0.90, bottom=0.175, top=0.965)
    sns.heatmap(
        heat,
        annot=True,
        fmt=".0f",
        cmap="YlGnBu",
        linewidths=0.6,
        linecolor="#E1E1E1",
        annot_kws={"fontsize": 10.5},
        cbar_kws={"label": r"Number of episodes per condition, $n$", "pad": 0.035},
        ax=ax,
    )
    ax.set(
        xlabel=r"Cluster-size coefficient of variation, $\mathrm{CV}$",
        ylabel=r"Intracluster correlation, $\rho_{\mathrm{ICC}}$",
    )
    ax.set_xticklabels(["0", "0.25", "0.50"], rotation=0)
    ax.set_yticklabels(["0", "0.01", "0.05", "0.10", "0.20"], rotation=0)
    _format_axes(ax, show_grid=False)
    _format_colorbar(ax)
    _save_figure(fig, output / "figures" / "figure3_cluster_sensitivity")

    heat80 = sensitivity[np.isclose(sensitivity["target_power"], 0.80)].pivot(
        index="control_rate_assumption",
        columns="absolute_reduction",
        values="required_episodes_per_arm",
    )
    heat80.to_csv(output / "data" / "figure4_event_rate_sensitivity.csv")
    heat80_display = heat80.copy()
    heat80_display.columns = [100 * float(value) for value in heat80_display.columns]
    heat80_display.index = [100 * float(value) for value in heat80_display.index]
    fig, ax = plt.subplots(figsize=(5.98, 4.932))
    fig.subplots_adjust(left=0.19, right=0.90, bottom=0.175, top=0.965)
    sns.heatmap(
        heat80_display,
        annot=True,
        fmt=".0f",
        cmap="mako_r",
        linewidths=0.6,
        linecolor="#E1E1E1",
        annot_kws={"fontsize": 10.5},
        cbar_kws={"label": r"Number of episodes per arm, $n$", "pad": 0.035},
        ax=ax,
    )
    ax.set(
        xlabel=r"Absolute risk reduction, $\Delta p$ (percentage points)",
        ylabel=r"Control collision risk, $p_0$ (%)",
    )
    ax.set_xticklabels(["1", "2", "3", "4"], rotation=0)
    ax.set_yticklabels(["3", "6", "10", "15"], rotation=0)
    _format_axes(ax, show_grid=False)
    _format_colorbar(ax)
    _save_figure(fig, output / "figures" / "figure4_event_rate_sensitivity")

    precision_curve = pd.DataFrame(
        {
            "episodes_per_arm": grid,
            "risk_difference_half_width": [
                normal_risk_difference_half_width(n, p0, p1) for n in grid
            ],
        }
    )
    precision_curve.to_csv(output / "data" / "figure5_precision_curve.csv", index=False)
    fig, ax = plt.subplots(figsize=(6.084, 4.744))
    fig.subplots_adjust(left=0.225, right=0.975, bottom=0.175, top=0.965)
    ax.plot(
        precision_curve["episodes_per_arm"],
        100 * precision_curve["risk_difference_half_width"],
        color="#E76F51",
        lw=2.2,
    )
    ax.set(
        xlabel=r"Number of episodes per arm, $n$",
        ylabel="95% risk-difference half-width (percentage points)",
        xlim=(0, 2000),
    )
    ax.set_xticks(np.arange(0, 2001, 250))
    _format_axes(ax)
    _save_figure(fig, output / "figures" / "figure5_precision_curve")

    allocation.to_csv(output / "data" / "figure6_allocation_by_seed.csv", index=False)
    fig, ax = plt.subplots(figsize=(6.20, 4.744))
    fig.subplots_adjust(left=0.165, right=0.975, bottom=0.175, top=0.965)
    ax.plot(
        allocation["seeds_per_scenario_environment_density_cell"],
        allocation["independent_power"],
        lw=2.1,
        label="Independent approximation",
        color="#264653",
        marker="o",
        markevery=4,
        ms=4,
        mfc="white",
    )
    ax.plot(
        allocation["seeds_per_scenario_environment_density_cell"],
        allocation["design_effect_approx_power"],
        lw=2.1,
        label=f"ICC={protocol['clustered_design']['planning_icc']:.2f} design-effect approximation",
        color="#F4A261",
        ls="--",
        marker="s",
        markevery=4,
        ms=4,
        mfc="white",
    )
    ax.axhline(target, color="#2A9D8F", ls=":", lw=1.5, label="80% target")
    ax.set(
        xlabel=r"Number of seeds per scenario–environment–density cell, $s$",
        ylabel=r"Approximate power, $1-\beta$",
        xlim=(3, 40),
        ylim=(0, 1.02),
    )
    ax.set_xticks(np.arange(5, 41, 5))
    ax.set_yticks(np.arange(0, 1.01, 0.2))
    _format_axes(ax)
    ax.legend(loc="lower right")
    _save_figure(fig, output / "figures" / "figure6_allocation_by_seed")


def _manuscript_ready_results(
    protocol: dict[str, Any],
    independent_summary: dict[str, float],
    paired: pd.DataFrame,
    recommendation: dict[str, Any],
    ablation: pd.DataFrame,
) -> str:
    rho0 = paired.loc[np.isclose(paired["correlation"], 0.0)].iloc[0]
    rare = ablation[
        (ablation["scenario"] == "two_events_in_150_gap")
        & np.isclose(ablation["alpha"], 0.05)
    ].iloc[0]
    return f"""# Manuscript-ready prospective power-analysis results

> {DISCLAIMER}

Under the prespecified planning scenario of a 6% episode-level collision probability in the control condition and 3% in the candidate condition, a two-sided equal-allocation score-test calculation with alpha = 0.05 and 80% power required {independent_summary['raw_required_n']:.3f} episodes per arm, rounded upward to **{int(independent_summary['required_n'])} episodes per arm** ({2 * int(independent_summary['required_n']):,} total). Under the same independence assumptions, 450 and 150 episodes per arm provided approximate powers of **{independent_summary['power_450']:.3f}** and **{independent_summary['power_150']:.3f}**, respectively. These quantities validate Reviewer 5's concern that the original 450-episode main comparison and 150-episode ablation blocks were underpowered for the assumed 3-percentage-point difference.

Pairing did not have a single fixed benefit because the marginal event probabilities do not determine the paired joint distribution. With zero assumed cross-method collision correlation, the exact two-sided McNemar design required **{int(rho0['minimum_pairs_exact'])} pairs**; at 450 pairs, exact power was **{rho0['exact_power_at_450_pairs']:.3f}**. Across the prespecified feasible correlation grid, the exact requirement ranged from **{int(paired['minimum_pairs_exact'].min())} to {int(paired['minimum_pairs_exact'].max())} pairs**. These are conditional sensitivity results, not estimates of the correlation that a future implementation will exhibit.

The provisional cluster-aware allocation uses cross-method correlation {recommendation['planning_correlation']:.2f}, within-scenario-seed ICC {recommendation['planning_icc']:.2f}, mean cluster size 9, equal cluster sizes, a 5% invalid-episode allowance, and complete 45-episode allocation blocks. It yields **{recommendation['planned_pairs']:,} paired scenario blocks**, corresponding to **{recommendation['episodes_per_condition']:,} episodes per condition** and **{recommendation['total_episodes_primary_two_condition']:,} episodes across the two-condition primary comparison**, or **{recommendation['seeds_per_matrix_cell']} seeds per scenario–environment–density cell**. This value is a conservative planning scenario and must be updated using a blinded pilot estimate of the discordant probabilities and ICC before formal execution.

The 6.000% versus 4.667% ablation contrast seen in the former 150-episode surrogate table corresponds to only two events and would require approximately **{int(rare['required_episodes_per_arm']):,} independent episodes per arm** for 80% power at alpha = 0.05 under those assumed rates. Therefore, collision should not be used to claim individual module effects from 150 episodes. Each learned ablation must be retrained independently across multiple training seeds and assigned a mechanism-specific primary endpoint; collision remains a separately powered safety endpoint.

No values in this report are empirical EGMS-Drive performance results. The future closed-loop study must retain pair IDs, scenario and training-seed clusters, independently generated outcomes in both discordant directions, prespecified exclusions, and the frozen analysis code.
"""


def _results_report(
    protocol: dict[str, Any],
    assumptions: pd.DataFrame,
    independent: pd.DataFrame,
    independent_summary: dict[str, float],
    null: pd.DataFrame,
    coverage: pd.DataFrame,
    paired: pd.DataFrame,
    bounds: tuple[float, float],
    clustered: pd.DataFrame,
    recommendation: dict[str, Any],
    ablation: pd.DataFrame,
    precision_targets: pd.DataFrame,
) -> str:
    return f"""# EGMS-Drive prospective protocol and power-analysis results

> **Design assumptions and power-analysis results only. No empirical EGMS-Drive performance is reported.**
>
> {DISCLAIMER}

## 1. Prespecified assumptions

{_frame_to_markdown(assumptions)}

The assumed 6% and 3% collision probabilities are planning inputs requested by the review analysis. They were not measured from CARLA, public datasets, a trained model, or real vehicles.

## 2. Independent two-arm design

The pooled-null/unpooled-alternative calculation gives a raw requirement of **{independent_summary['raw_required_n']:.3f}**, rounded to **{int(independent_summary['required_n'])} episodes per arm**. The original allocations would provide approximately **{independent_summary['power_450']:.3f} power at 450/arm** and **{independent_summary['power_150']:.3f} power at 150/arm**.

{_frame_to_markdown(independent, float_digits=6)}

## 3. Null calibration

The complete-trial Monte Carlo simulation was also run under p_control = p_candidate = 0.06. The rejection proportions below assess statistical calibration; they are not method comparisons.

{_frame_to_markdown(null, float_digits=6)}

## 4. Wilson-interval coverage calibration

The prospective binomial interval implementation was evaluated through complete samples at both assumed event rates. Coverage is itself a Monte Carlo proportion and includes MCSE and a Wilson Monte Carlo interval.

{_frame_to_markdown(coverage, float_digits=6)}

## 5. Paired exact-McNemar sensitivity

For the 6% and 3% marginals, the feasible Pearson-correlation range is **[{bounds[0]:.6f}, {bounds[1]:.6f}]**. Each row reports all four joint probabilities, allowing both directions of discordance.

{_frame_to_markdown(paired, float_digits=6)}

## 6. Cluster and incomplete-episode sensitivity

The planning ICC is distinct from the cross-method paired correlation. The table uses the approximate design effect and rounds upward to complete 45-episode scenario blocks.

{_frame_to_markdown(clustered, float_digits=6)}

Provisional scenario: **{recommendation['planned_pairs']:,} paired scenario blocks**, equal to **{recommendation['episodes_per_condition']:,} episodes per condition** and **{recommendation['total_episodes_primary_two_condition']:,} total episodes for the primary two-condition contrast** ({recommendation['seeds_per_matrix_cell']} seeds per scenario–environment–density cell), under correlation {recommendation['planning_correlation']:.2f}, ICC {recommendation['planning_icc']:.2f}, and 5% invalid episodes. A blinded pilot should replace these dependence assumptions.

## 7. Rare-event and ablation planning

{_frame_to_markdown(ablation, float_digits=6)}

The four module ablations have separate mechanism-specific endpoints in `docs/statistical_analysis_plan.md`. A 150-episode block is not treated as adequate evidence of a collision-rate module effect.

## 8. Precision targets

{_frame_to_markdown(precision_targets, float_digits=6)}

## Interpretation boundary

The outputs answer **how many future episodes may be needed under stated assumptions**. They do not answer how EGMS-Drive performs. The repository deliberately contains no simulated F1, latency, VRAM, LLM consistency, or method-ranking outputs.
"""


def _validation_checks(
    protocol: dict[str, Any],
    independent_summary: dict[str, float],
    independent: pd.DataFrame,
    null: pd.DataFrame,
    coverage: pd.DataFrame,
    paired: pd.DataFrame,
    report_text: str,
) -> pd.DataFrame:
    p0 = float(protocol["primary_design"]["control_rate_assumption"])
    p1 = float(protocol["primary_design"]["candidate_rate_assumption"])
    alpha = float(protocol["primary_design"]["alpha"])
    lower, upper = feasible_correlation_bounds(p0, p1)
    expected = [
        (1, "Protocol schema loaded", True, protocol["project"]["version"]),
        (2, "Prospective-only status", protocol["project"]["status"] == "prospective_design_only", protocol["project"]["status"]),
        (3, "No empirical data flag", protocol["project"]["empirical_data_present"] is False, str(protocol["project"]["empirical_data_present"])),
        (4, "No empirical performance claims flag", protocol["project"]["empirical_performance_claims"] is False, str(protocol["project"]["empirical_performance_claims"])),
        (5, "Primary endpoint defined", bool(protocol["primary_design"]["endpoint"]), protocol["primary_design"]["endpoint"]),
        (6, "Estimand defined", bool(protocol["primary_design"]["estimand"]), protocol["primary_design"]["estimand"]),
        (7, "Alpha valid", 0 < alpha < 1, f"alpha={alpha}"),
        (8, "Target power valid", 0 < protocol["primary_design"]["target_power"] < 1, f"power={protocol['primary_design']['target_power']}"),
        (9, "Two-sided analysis", protocol["primary_design"]["sidedness"] == "two-sided", protocol["primary_design"]["sidedness"]),
        (10, "Probability bounds", 0 < p1 < 1 and 0 < p0 < 1, f"p0={p0}, p1={p1}"),
        (11, "Nonzero planning effect", not math.isclose(p0, p1), f"difference={p0-p1:.6f}"),
        (12, "Headline raw sample-size check", abs(independent_summary["raw_required_n"] - 748.388) < 0.01, f"raw_n={independent_summary['raw_required_n']:.6f}"),
        (13, "Headline ceiling sample-size check", independent_summary["required_n"] == 749, f"n={independent_summary['required_n']}"),
        (14, "450-episode power check", abs(independent_summary["power_450"] - 0.583696) < 1e-5, f"power={independent_summary['power_450']:.6f}"),
        (15, "150-episode power check", abs(independent_summary["power_150"] - 0.239938) < 1e-5, f"power={independent_summary['power_150']:.6f}"),
        (16, "Complete-trial simulation", independent["mc_repetitions"].min() >= 10000, f"min repetitions={independent['mc_repetitions'].min()}"),
        (17, "Monte Carlo uncertainty reported", independent["mcse"].notna().all(), "MCSE and Wilson CI columns present"),
        (18, "Monte Carlo precision", independent["mcse"].max() <= 0.005, f"max MCSE={independent['mcse'].max():.6f}"),
        (19, "Null calibration scenarios", len(null) >= 3, f"rows={len(null)}"),
        (20, "Null calibration within tolerance", bool(null["within_3mcse_plus_0_005"].all()), f"range={null['estimated_type1_error'].min():.4f}-{null['estimated_type1_error'].max():.4f}"),
        (21, "Wilson coverage calibration", bool(coverage["coverage_in_prespecified_range"].all()), f"range={coverage['estimated_coverage'].min():.4f}-{coverage['estimated_coverage'].max():.4f}"),
        (22, "Paired distributions valid", bool((paired[["pi00_both_safe", "pi01_candidate_only_collision", "pi10_control_only_collision", "pi11_both_collision"]] >= 0).all().all()) and bool(np.allclose(paired[["pi00_both_safe", "pi01_candidate_only_collision", "pi10_control_only_collision", "pi11_both_collision"]].sum(axis=1), 1.0)), "nonnegative cells summing to one"),
        (23, "Control marginal recovered", bool(np.allclose(paired["pi10_control_only_collision"] + paired["pi11_both_collision"], p0)), "pi10+pi11=p_control"),
        (24, "Candidate marginal recovered", bool(np.allclose(paired["pi01_candidate_only_collision"] + paired["pi11_both_collision"], p1)), "pi01+pi11=p_candidate"),
        (25, "Both discordant directions possible", bool((paired["pi10_control_only_collision"] > 0).all() and (paired["pi01_candidate_only_collision"] > 0).all()), "pi10>0 and pi01>0"),
        (26, "Exact paired test specified", protocol["primary_design"]["paired_test"] == "exact_mcnemar", protocol["primary_design"]["paired_test"]),
        (27, "ICC range valid", all(0 <= value < 1 for value in protocol["clustered_design"]["icc_values"]), str(protocol["clustered_design"]["icc_values"])),
        (28, "Positive cluster size", protocol["clustered_design"]["mean_cluster_size"] > 0, str(protocol["clustered_design"]["mean_cluster_size"])),
        (29, "Primary comparison count", protocol["multiplicity"]["primary_comparisons"] == 1, str(protocol["multiplicity"]["primary_comparisons"])),
        (30, "Secondary multiplicity plan", protocol["multiplicity"]["secondary_adjustment"] == "holm", protocol["multiplicity"]["secondary_adjustment"]),
        (31, "Scenario allocation complete", protocol["scenario_matrix"]["allocation_block"] == protocol["scenario_matrix"]["scenario_families"] * protocol["scenario_matrix"]["environments"] * protocol["scenario_matrix"]["traffic_densities"], f"block={protocol['scenario_matrix']['allocation_block']}"),
        (32, "Interpretation disclaimer present", "not empirical" in report_text.lower() or "no empirical" in report_text.lower(), "report disclaimer found"),
        (33, "Legacy performance claims absent", all(term not in report_text.lower() for term in ["f1 of 0.868", "reduced collisions by 51.9", "achieved a collision rate of 2.9"]), "claim guard passed"),
    ]
    return pd.DataFrame(expected, columns=["check_id", "check", "passed", "evidence"])


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _write_manifest(
    root: Path,
    output: Path,
    protocol: dict[str, Any],
    validation: pd.DataFrame,
) -> dict[str, Any]:
    source_files = sorted((root / "src").rglob("*.py"))
    output_files = sorted(
        path for path in output.rglob("*") if path.is_file() and path.name != "run_manifest.json"
    )
    manifest = {
        "project": protocol["project"]["title"],
        "version": __version__,
        "status": protocol["project"]["status"],
        "generated_at_utc": _generated_at_utc(),
        "disclaimer": DISCLAIMER,
        "protocol_sha256": hashlib.sha256(canonical_protocol_json(protocol).encode("utf-8")).hexdigest(),
        "master_seed": protocol["simulation"]["master_seed"],
        "canonical_repetitions": protocol["simulation"]["canonical_repetitions"],
        "validation_checks_passed": int(validation["passed"].sum()),
        "validation_checks_total": int(len(validation)),
        "runtime": {
            "python": sys.version.split()[0],
            "platform": platform.platform(),
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "scipy": scipy.__version__,
            "matplotlib": matplotlib.__version__,
            "seaborn": sns.__version__,
        },
        "source_files": {
            str(path.relative_to(root)): _sha256(path) for path in source_files
        },
        "output_files": {
            str(path.relative_to(root)): _sha256(path) for path in output_files
        },
    }
    (output / "run_manifest.json").write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )
    return manifest


def run_power_analysis(
    config_path: str | Path,
    output_dir: str | Path = "outputs",
) -> dict[str, Any]:
    """Run the canonical protocol and power-analysis workflow."""

    root = project_root()
    protocol = load_protocol(config_path)
    output = prepare_output_directory(root, output_dir)

    assumptions = _assumptions_table(protocol)
    independent, independent_summary = _independent_results(protocol)
    null = _null_calibration(protocol)
    coverage = _coverage_calibration(protocol)
    paired, bounds = _paired_results(protocol)
    clustered, recommendation = _cluster_results(protocol, paired)
    ablation = _ablation_results(protocol)
    sensitivity = _sensitivity_results(protocol)
    precision, precision_targets = _precision_results(protocol)
    allocation = _allocation_results(protocol)

    tables = {
        "table1_protocol_assumptions": assumptions,
        "table2_independent_power": independent,
        "table3_null_calibration": null,
        "table4_wilson_coverage": coverage,
        "table5_paired_sensitivity": paired,
        "table6_cluster_sensitivity": clustered,
        "table7_ablation_power": ablation,
        "table8_event_rate_sensitivity": sensitivity,
        "table9_precision_intervals": precision,
        "table10_precision_targets": precision_targets,
        "table11_balanced_allocation": allocation,
    }
    for name, frame in tables.items():
        _save_table(frame, output / "tables" / name)

    _plot_all(
        output,
        protocol,
        independent,
        paired,
        clustered,
        sensitivity,
        precision,
        allocation,
    )

    report = _results_report(
        protocol,
        assumptions,
        independent,
        independent_summary,
        null,
        coverage,
        paired,
        bounds,
        clustered,
        recommendation,
        ablation,
        precision_targets,
    )
    (output / "POWER_ANALYSIS_RESULTS.md").write_text(report, encoding="utf-8")
    manuscript = _manuscript_ready_results(
        protocol, independent_summary, paired, recommendation, ablation
    )
    (output / "MANUSCRIPT_READY_RESULTS.md").write_text(manuscript, encoding="utf-8")

    summary = {
        "disclaimer": DISCLAIMER,
        "independent_raw_required_n": independent_summary["raw_required_n"],
        "independent_required_n_per_arm": independent_summary["required_n"],
        "independent_power_at_450": independent_summary["power_450"],
        "independent_power_at_150": independent_summary["power_150"],
        "feasible_paired_correlation": {"lower": bounds[0], "upper": bounds[1]},
        "cluster_aware_provisional_recommendation": recommendation,
    }
    (output / "summary.json").write_text(
        json.dumps(summary, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
    )

    validation = _validation_checks(
        protocol, independent_summary, independent, null, coverage, paired, report
    )
    validation.to_csv(output / "validation_report.csv", index=False)
    if not bool(validation["passed"].all()):
        failures = validation.loc[~validation["passed"], ["check_id", "check", "evidence"]]
        raise RuntimeError("Validation failed:\n" + failures.to_string(index=False))
    manifest = _write_manifest(root, output, protocol, validation)

    return {
        "output_directory": str(output),
        "summary": summary,
        "validation": {
            "passed": int(validation["passed"].sum()),
            "total": len(validation),
        },
        "manifest": manifest,
    }


### `src/egms_power/statistics.py`

In [ ]:
%%writefile src/egms_power/statistics.py
"""Statistical calculations for a prospective collision-outcome study.

The functions in this module operate on prespecified design assumptions. They
do not simulate or estimate EGMS-Drive performance.
"""

from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Iterable

import numpy as np
from scipy.stats import binom, norm


@dataclass(frozen=True)
class MonteCarloEstimate:
    """A binomial Monte Carlo estimate with uncertainty."""

    estimate: float
    rejections: int
    repetitions: int
    mcse: float
    ci_low: float
    ci_high: float


def _validate_probability(value: float, name: str, *, strict: bool = False) -> None:
    lower_ok = value > 0 if strict else value >= 0
    upper_ok = value < 1 if strict else value <= 1
    if not (lower_ok and upper_ok):
        bounds = "(0, 1)" if strict else "[0, 1]"
        raise ValueError(f"{name} must be in {bounds}; received {value!r}")


def _validate_design(alpha: float, target_power: float | None = None) -> None:
    _validate_probability(alpha, "alpha", strict=True)
    if target_power is not None:
        _validate_probability(target_power, "target_power", strict=True)


def required_n_two_independent_proportions(
    p_control: float,
    p_candidate: float,
    *,
    alpha: float = 0.05,
    target_power: float = 0.80,
) -> tuple[float, int]:
    """Return raw and ceiling sample sizes per arm for a two-sided score test.

    This uses a pooled variance under the null and an unpooled variance under
    the alternative. Equal allocation is assumed.
    """

    _validate_probability(p_control, "p_control")
    _validate_probability(p_candidate, "p_candidate")
    _validate_design(alpha, target_power)
    delta = abs(p_control - p_candidate)
    if delta == 0:
        raise ValueError("p_control and p_candidate must differ for power planning")

    pooled = (p_control + p_candidate) / 2.0
    z_alpha = norm.ppf(1.0 - alpha / 2.0)
    z_power = norm.ppf(target_power)
    null_sd = math.sqrt(2.0 * pooled * (1.0 - pooled))
    alt_sd = math.sqrt(
        p_control * (1.0 - p_control)
        + p_candidate * (1.0 - p_candidate)
    )
    raw_n = ((z_alpha * null_sd + z_power * alt_sd) ** 2) / (delta**2)
    return float(raw_n), int(math.ceil(raw_n))


def two_independent_proportions_power(
    n_per_arm: float,
    p_control: float,
    p_candidate: float,
    *,
    alpha: float = 0.05,
) -> float:
    """Approximate two-sided power for equal-sized independent arms."""

    if n_per_arm <= 0:
        raise ValueError("n_per_arm must be positive")
    _validate_probability(p_control, "p_control")
    _validate_probability(p_candidate, "p_candidate")
    _validate_design(alpha)
    if p_control == p_candidate:
        return float(alpha)

    pooled = (p_control + p_candidate) / 2.0
    delta = abs(p_control - p_candidate)
    z_alpha = norm.ppf(1.0 - alpha / 2.0)
    null_sd = math.sqrt(2.0 * pooled * (1.0 - pooled))
    alt_sd = math.sqrt(
        p_control * (1.0 - p_control)
        + p_candidate * (1.0 - p_candidate)
    )
    shifted_mean = math.sqrt(float(n_per_arm)) * delta
    upper = norm.cdf((shifted_mean - z_alpha * null_sd) / alt_sd)
    lower = norm.cdf((-shifted_mean - z_alpha * null_sd) / alt_sd)
    return float(upper + lower)


def pooled_two_proportion_pvalues(
    control_events: np.ndarray,
    candidate_events: np.ndarray,
    n_per_arm: int,
) -> np.ndarray:
    """Vectorized two-sided pooled z-test p-values."""

    if n_per_arm <= 0:
        raise ValueError("n_per_arm must be positive")
    x0 = np.asarray(control_events, dtype=float)
    x1 = np.asarray(candidate_events, dtype=float)
    pooled = (x0 + x1) / (2.0 * n_per_arm)
    variance = pooled * (1.0 - pooled) * (2.0 / n_per_arm)
    difference = x0 / n_per_arm - x1 / n_per_arm
    z = np.divide(
        difference,
        np.sqrt(variance),
        out=np.zeros_like(difference, dtype=float),
        where=variance > 0,
    )
    pvalues = 2.0 * norm.sf(np.abs(z))
    return np.where(variance > 0, pvalues, 1.0)


def wilson_interval(successes: int, total: int, confidence: float = 0.95) -> tuple[float, float]:
    """Wilson score interval for a binomial proportion."""

    if total <= 0:
        raise ValueError("total must be positive")
    if not 0 <= successes <= total:
        raise ValueError("successes must be between 0 and total")
    _validate_probability(confidence, "confidence", strict=True)
    z = norm.ppf(1.0 - (1.0 - confidence) / 2.0)
    p_hat = successes / total
    denominator = 1.0 + (z**2) / total
    center = (p_hat + (z**2) / (2.0 * total)) / denominator
    half = (
        z
        * math.sqrt(
            p_hat * (1.0 - p_hat) / total
            + (z**2) / (4.0 * total**2)
        )
        / denominator
    )
    return max(0.0, center - half), min(1.0, center + half)


def _mc_estimate(rejections: np.ndarray, confidence: float = 0.95) -> MonteCarloEstimate:
    decisions = np.asarray(rejections, dtype=bool)
    total = int(decisions.size)
    if total == 0:
        raise ValueError("at least one Monte Carlo repetition is required")
    count = int(decisions.sum())
    estimate = count / total
    mcse = math.sqrt(estimate * (1.0 - estimate) / total)
    low, high = wilson_interval(count, total, confidence)
    return MonteCarloEstimate(estimate, count, total, mcse, low, high)


def simulate_unpaired_power(
    n_per_arm: int,
    p_control: float,
    p_candidate: float,
    *,
    alpha: float = 0.05,
    repetitions: int = 10_000,
    seed: int = 20260810,
) -> MonteCarloEstimate:
    """Estimate power by simulating complete independent two-arm trials."""

    if n_per_arm <= 0 or repetitions <= 0:
        raise ValueError("n_per_arm and repetitions must be positive")
    _validate_probability(p_control, "p_control")
    _validate_probability(p_candidate, "p_candidate")
    _validate_design(alpha)
    rng = np.random.default_rng(seed)
    control_events = rng.binomial(n_per_arm, p_control, size=repetitions)
    candidate_events = rng.binomial(n_per_arm, p_candidate, size=repetitions)
    pvalues = pooled_two_proportion_pvalues(
        control_events, candidate_events, n_per_arm
    )
    return _mc_estimate(pvalues < alpha)


def simulate_wilson_coverage(
    n: int,
    true_probability: float,
    *,
    confidence: float = 0.95,
    repetitions: int = 10_000,
    seed: int = 20260810,
) -> MonteCarloEstimate:
    """Estimate coverage of a Wilson interval through complete samples."""

    if n <= 0 or repetitions <= 0:
        raise ValueError("n and repetitions must be positive")
    _validate_probability(true_probability, "true_probability")
    _validate_probability(confidence, "confidence", strict=True)
    rng = np.random.default_rng(seed)
    counts = rng.binomial(n, true_probability, size=repetitions)
    p_hat = counts / n
    z = norm.ppf(1.0 - (1.0 - confidence) / 2.0)
    denominator = 1.0 + (z**2) / n
    center = (p_hat + (z**2) / (2.0 * n)) / denominator
    half = (
        z
        * np.sqrt(p_hat * (1.0 - p_hat) / n + (z**2) / (4.0 * n**2))
        / denominator
    )
    covered = (center - half <= true_probability) & (true_probability <= center + half)
    return _mc_estimate(covered, confidence=confidence)


def feasible_correlation_bounds(p_control: float, p_candidate: float) -> tuple[float, float]:
    """Fréchet-compatible Pearson-correlation bounds for paired Bernoulli data."""

    _validate_probability(p_control, "p_control", strict=True)
    _validate_probability(p_candidate, "p_candidate", strict=True)
    scale = math.sqrt(
        p_control * (1.0 - p_control)
        * p_candidate * (1.0 - p_candidate)
    )
    lower_joint = max(0.0, p_control + p_candidate - 1.0)
    upper_joint = min(p_control, p_candidate)
    center = p_control * p_candidate
    return (lower_joint - center) / scale, (upper_joint - center) / scale


def paired_joint_probabilities(
    p_control: float,
    p_candidate: float,
    correlation: float,
) -> dict[str, float]:
    """Return the four paired collision probabilities under a correlation.

    ``control_only`` is pi_10 and ``candidate_only`` is pi_01.
    """

    lower, upper = feasible_correlation_bounds(p_control, p_candidate)
    tolerance = 1e-12
    if correlation < lower - tolerance or correlation > upper + tolerance:
        raise ValueError(
            f"correlation {correlation} is infeasible; valid range is "
            f"[{lower:.6f}, {upper:.6f}]"
        )
    scale = math.sqrt(
        p_control * (1.0 - p_control)
        * p_candidate * (1.0 - p_candidate)
    )
    both_collision = p_control * p_candidate + correlation * scale
    control_only = p_control - both_collision
    candidate_only = p_candidate - both_collision
    both_safe = 1.0 - both_collision - control_only - candidate_only
    values = {
        "both_safe": both_safe,
        "candidate_only": candidate_only,
        "control_only": control_only,
        "both_collision": both_collision,
    }
    if any(value < -tolerance for value in values.values()):
        raise ValueError("paired joint probabilities include a negative cell")
    values = {key: max(0.0, float(value)) for key, value in values.items()}
    total = sum(values.values())
    return {key: value / total for key, value in values.items()}


def exact_mcnemar_pvalue(control_only: int, candidate_only: int) -> float:
    """Two-sided exact McNemar p-value from the discordant cells."""

    if control_only < 0 or candidate_only < 0:
        raise ValueError("discordant counts cannot be negative")
    discordant = int(control_only + candidate_only)
    if discordant == 0:
        return 1.0
    smaller = int(min(control_only, candidate_only))
    return float(min(1.0, 2.0 * binom.cdf(smaller, discordant, 0.5)))


def _mcnemar_conditional_rejection_probability(
    discordant_counts: np.ndarray,
    theta_control_only: float,
    alpha: float,
) -> np.ndarray:
    """P(reject | D=d) for the exact two-sided McNemar test."""

    d = np.asarray(discordant_counts, dtype=int)
    critical = binom.ppf(alpha / 2.0, d, 0.5)
    critical = np.nan_to_num(critical, nan=-1).astype(int)
    null_cdf = binom.cdf(critical, d, 0.5)
    critical = np.where(null_cdf > alpha / 2.0, critical - 1, critical)
    critical = np.maximum(critical, -1)
    lower = np.where(
        critical >= 0,
        binom.cdf(critical, d, theta_control_only),
        0.0,
    )
    upper_threshold = d - critical
    upper = np.where(
        upper_threshold <= d,
        binom.sf(upper_threshold - 1, d, theta_control_only),
        0.0,
    )
    return lower + upper


def exact_mcnemar_power(
    n_pairs: int,
    p_control: float,
    p_candidate: float,
    correlation: float,
    *,
    alpha: float = 0.05,
) -> float:
    """Unconditional power of the exact two-sided McNemar test."""

    if n_pairs <= 0:
        raise ValueError("n_pairs must be positive")
    _validate_design(alpha)
    cells = paired_joint_probabilities(p_control, p_candidate, correlation)
    q10 = cells["control_only"]
    q01 = cells["candidate_only"]
    discordance = q10 + q01
    if discordance == 0:
        return float(alpha if p_control == p_candidate else 0.0)
    theta = q10 / discordance
    d = np.arange(n_pairs + 1)
    conditional = _mcnemar_conditional_rejection_probability(d, theta, alpha)
    weights = binom.pmf(d, n_pairs, discordance)
    return float(np.sum(weights * conditional))


def approximate_mcnemar_required_n(
    p_control: float,
    p_candidate: float,
    correlation: float,
    *,
    alpha: float = 0.05,
    target_power: float = 0.80,
) -> float:
    """Asymptotic paired sample-size approximation using discordant cells."""

    _validate_design(alpha, target_power)
    cells = paired_joint_probabilities(p_control, p_candidate, correlation)
    q10 = cells["control_only"]
    q01 = cells["candidate_only"]
    qd = q10 + q01
    delta = abs(q10 - q01)
    if delta == 0:
        raise ValueError("discordant probabilities imply no paired effect")
    z_alpha = norm.ppf(1.0 - alpha / 2.0)
    z_power = norm.ppf(target_power)
    numerator = (
        z_alpha * math.sqrt(qd)
        + z_power * math.sqrt(max(0.0, qd - delta**2))
    ) ** 2
    return float(numerator / delta**2)


def find_minimum_exact_mcnemar_n(
    p_control: float,
    p_candidate: float,
    correlation: float,
    *,
    alpha: float = 0.05,
    target_power: float = 0.80,
    maximum_n: int = 20_000,
) -> tuple[int, float]:
    """Find the first pair count whose exact McNemar power reaches target."""

    _validate_design(alpha, target_power)
    if maximum_n < 2:
        raise ValueError("maximum_n must be at least 2")
    approximate = approximate_mcnemar_required_n(
        p_control,
        p_candidate,
        correlation,
        alpha=alpha,
        target_power=target_power,
    )
    start = max(2, int(math.floor(approximate)) - 300)
    for n_pairs in range(start, maximum_n + 1):
        power = exact_mcnemar_power(
            n_pairs,
            p_control,
            p_candidate,
            correlation,
            alpha=alpha,
        )
        if power >= target_power:
            return n_pairs, power
    raise RuntimeError(f"target power was not reached by maximum_n={maximum_n}")


def simulate_paired_power(
    n_pairs: int,
    p_control: float,
    p_candidate: float,
    correlation: float,
    *,
    alpha: float = 0.05,
    repetitions: int = 10_000,
    seed: int = 20260810,
) -> MonteCarloEstimate:
    """Estimate exact-McNemar power using multinomial complete trials."""

    if n_pairs <= 0 or repetitions <= 0:
        raise ValueError("n_pairs and repetitions must be positive")
    cells = paired_joint_probabilities(p_control, p_candidate, correlation)
    probabilities = np.array(
        [
            cells["both_safe"],
            cells["candidate_only"],
            cells["control_only"],
            cells["both_collision"],
        ]
    )
    rng = np.random.default_rng(seed)
    counts = rng.multinomial(n_pairs, probabilities, size=repetitions)
    candidate_only = counts[:, 1]
    control_only = counts[:, 2]
    discordant = candidate_only + control_only
    smaller = np.minimum(candidate_only, control_only)
    pvalues = np.ones(repetitions, dtype=float)
    nonzero = discordant > 0
    pvalues[nonzero] = np.minimum(
        1.0,
        2.0 * binom.cdf(smaller[nonzero], discordant[nonzero], 0.5),
    )
    return _mc_estimate(pvalues < alpha)


def design_effect(
    mean_cluster_size: float,
    icc: float,
    cluster_size_cv: float = 0.0,
) -> float:
    """Approximate design effect, allowing unequal cluster sizes."""

    if mean_cluster_size <= 0:
        raise ValueError("mean_cluster_size must be positive")
    _validate_probability(icc, "icc")
    if cluster_size_cv < 0:
        raise ValueError("cluster_size_cv cannot be negative")
    effective_size = (1.0 + cluster_size_cv**2) * mean_cluster_size
    return float(1.0 + (effective_size - 1.0) * icc)


def round_up_to_block(value: float, block_size: int) -> int:
    """Round a positive planning count up to a complete allocation block."""

    if value <= 0 or block_size <= 0:
        raise ValueError("value and block_size must be positive")
    return int(math.ceil(value / block_size) * block_size)


def representative_wilson_interval(
    n: int,
    assumed_rate: float,
    confidence: float = 0.95,
) -> tuple[int, float, float, float]:
    """Wilson interval using the nearest integer to the expected event count."""

    if n <= 0:
        raise ValueError("n must be positive")
    _validate_probability(assumed_rate, "assumed_rate")
    expected_events = int(round(n * assumed_rate))
    low, high = wilson_interval(expected_events, n, confidence)
    return expected_events, low, high, high - low


def normal_risk_difference_half_width(
    n_per_arm: int,
    p_control: float,
    p_candidate: float,
    confidence: float = 0.95,
) -> float:
    """Planning half-width for an independent two-proportion risk difference."""

    if n_per_arm <= 0:
        raise ValueError("n_per_arm must be positive")
    _validate_probability(p_control, "p_control")
    _validate_probability(p_candidate, "p_candidate")
    _validate_probability(confidence, "confidence", strict=True)
    z = norm.ppf(1.0 - (1.0 - confidence) / 2.0)
    variance = (
        p_control * (1.0 - p_control)
        + p_candidate * (1.0 - p_candidate)
    ) / n_per_arm
    return float(z * math.sqrt(variance))


def minimum_n_for_risk_difference_precision(
    half_width: float,
    p_control: float,
    p_candidate: float,
    confidence: float = 0.95,
) -> int:
    """Normal-approximation sample size per arm for a target half-width."""

    if half_width <= 0:
        raise ValueError("half_width must be positive")
    _validate_probability(p_control, "p_control")
    _validate_probability(p_candidate, "p_candidate")
    _validate_probability(confidence, "confidence", strict=True)
    z = norm.ppf(1.0 - (1.0 - confidence) / 2.0)
    variance_sum = (
        p_control * (1.0 - p_control)
        + p_candidate * (1.0 - p_candidate)
    )
    return int(math.ceil((z**2) * variance_sum / (half_width**2)))


def stable_stream_seed(master_seed: int, components: Iterable[object]) -> int:
    """Create a deterministic independent stream seed from labeled components."""

    import hashlib

    payload = "|".join([str(master_seed), *(str(item) for item in components)])
    digest = hashlib.sha256(payload.encode("utf-8")).digest()
    return int.from_bytes(digest[:8], "big", signed=False)


### `src/egms_studies23/__init__.py`

In [ ]:
%%writefile src/egms_studies23/__init__.py
"""EGMS-Drive controlled mechanism validation for Studies 2 and 3."""

__version__ = "1.1.0"


### `src/egms_studies23/common.py`

In [ ]:
%%writefile src/egms_studies23/common.py
from __future__ import annotations

import hashlib
import json
import platform
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy.optimize import minimize_scalar


EPS = 1.0e-12


def stable_seed(*parts: object) -> int:
    payload = "|".join(str(part) for part in parts).encode("utf-8")
    return int.from_bytes(hashlib.sha256(payload).digest()[:8], "little") % (2**32 - 1)


def sha256(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def array_digest(*arrays: np.ndarray) -> str:
    digest = hashlib.sha256()
    for array in arrays:
        contiguous = np.ascontiguousarray(np.asarray(array))
        digest.update(str(contiguous.dtype).encode("ascii"))
        digest.update(str(contiguous.shape).encode("ascii"))
        digest.update(contiguous.tobytes())
    return digest.hexdigest()


def softmax(values: np.ndarray, axis: int = -1) -> np.ndarray:
    x = np.asarray(values, dtype=float)
    shifted = x - np.max(x, axis=axis, keepdims=True)
    exponents = np.exp(shifted)
    return exponents / np.maximum(exponents.sum(axis=axis, keepdims=True), EPS)


def normalize_rows(values: np.ndarray) -> np.ndarray:
    x = np.asarray(values, dtype=float)
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), EPS)


def _probabilities(probabilities: np.ndarray) -> np.ndarray:
    p = np.asarray(probabilities, dtype=float)
    if p.ndim != 2 or p.shape[1] < 2:
        raise ValueError("probabilities must be [n, classes]")
    if not np.all(np.isfinite(p)):
        raise ValueError("probabilities contain non-finite values")
    if np.any(p < -1.0e-10) or np.any(p > 1.0 + 1.0e-10):
        raise ValueError("probability outside [0,1]")
    if not np.allclose(p.sum(axis=1), 1.0, atol=1.0e-8, rtol=0.0):
        raise ValueError("probability rows do not sum to one")
    return np.clip(p, 0.0, 1.0)


def confusion_matrix(y_true: Iterable[int], y_pred: Iterable[int], n_classes: int) -> np.ndarray:
    truth = np.asarray(list(y_true), dtype=int)
    pred = np.asarray(list(y_pred), dtype=int)
    matrix = np.zeros((n_classes, n_classes), dtype=int)
    np.add.at(matrix, (truth, pred), 1)
    return matrix


def macro_f1(y_true: Iterable[int], y_pred: Iterable[int], n_classes: int) -> float:
    matrix = confusion_matrix(y_true, y_pred, n_classes).astype(float)
    scores = []
    for label in range(n_classes):
        tp = matrix[label, label]
        fp = matrix[:, label].sum() - tp
        fn = matrix[label, :].sum() - tp
        denominator = 2 * tp + fp + fn
        scores.append(0.0 if denominator == 0 else 2 * tp / denominator)
    return float(np.mean(scores))


def negative_log_likelihood(y_true: Iterable[int], probabilities: np.ndarray) -> float:
    truth = np.asarray(list(y_true), dtype=int)
    p = _probabilities(probabilities)
    return float(np.mean(-np.log(np.maximum(p[np.arange(len(truth)), truth], EPS))))


def multiclass_brier(y_true: Iterable[int], probabilities: np.ndarray) -> float:
    truth = np.asarray(list(y_true), dtype=int)
    p = _probabilities(probabilities)
    one_hot = np.eye(p.shape[1], dtype=float)[truth]
    return float(np.mean(np.sum((p - one_hot) ** 2, axis=1)))


def calibration_bins(
    y_true: Iterable[int], probabilities: np.ndarray, n_bins: int = 15
) -> pd.DataFrame:
    truth = np.asarray(list(y_true), dtype=int)
    p = _probabilities(probabilities)
    confidence = p.max(axis=1)
    prediction = p.argmax(axis=1)
    correctness = (prediction == truth).astype(float)
    # Equal-mass bins reduce the large finite-sample bias that sparse fixed
    # width bins can introduce. Stable sorting makes tied-confidence handling
    # deterministic. Each observation belongs to exactly one bin.
    ordered = np.argsort(confidence, kind="stable")
    groups = np.array_split(ordered, int(n_bins))
    rows = []
    for bin_index, indices in enumerate(groups):
        active = len(indices) > 0
        rows.append(
            {
                "bin": bin_index + 1,
                "lower": float(confidence[indices].min()) if active else np.nan,
                "upper": float(confidence[indices].max()) if active else np.nan,
                "count": int(len(indices)),
                "mean_confidence": float(confidence[indices].mean()) if active else np.nan,
                "accuracy": float(correctness[indices].mean()) if active else np.nan,
            }
        )
    return pd.DataFrame(rows)


def expected_calibration_error(
    y_true: Iterable[int], probabilities: np.ndarray, n_bins: int = 15
) -> float:
    bins = calibration_bins(y_true, probabilities, n_bins)
    total = bins["count"].sum()
    if total == 0:
        return float("nan")
    gap = (bins["accuracy"] - bins["mean_confidence"]).abs().fillna(0.0)
    return float(np.sum((bins["count"] / total) * gap))


@dataclass
class TemperatureScaler:
    temperature: float = 1.0

    def fit(self, probabilities: np.ndarray, labels: Iterable[int]) -> "TemperatureScaler":
        p = _probabilities(probabilities)
        y = np.asarray(list(labels), dtype=int)
        logits = np.log(np.maximum(p, EPS))

        def objective(log_temperature: float) -> float:
            calibrated = softmax(logits / np.exp(log_temperature), axis=1)
            return negative_log_likelihood(y, calibrated)

        result = minimize_scalar(objective, bounds=(-2.5, 2.5), method="bounded")
        self.temperature = float(np.exp(result.x)) if result.success else 1.0
        return self

    def transform(self, probabilities: np.ndarray) -> np.ndarray:
        p = _probabilities(probabilities)
        return softmax(np.log(np.maximum(p, EPS)) / self.temperature, axis=1)


def seed_summary(values: Iterable[float], confidence: float = 0.95) -> tuple[float, float, float]:
    x = np.asarray(list(values), dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return (float("nan"), float("nan"), float("nan"))
    estimate = float(x.mean())
    if len(x) == 1:
        return (estimate, float("nan"), float("nan"))
    from scipy.stats import t

    half = float(t.ppf((1 + confidence) / 2, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x)))
    return (estimate, estimate - half, estimate + half)


def holm_adjust(p_values: Iterable[float]) -> np.ndarray:
    p = np.asarray(list(p_values), dtype=float)
    order = np.argsort(p)
    adjusted = np.empty_like(p)
    running = 0.0
    count = len(p)
    for rank, index in enumerate(order):
        value = min(1.0, (count - rank) * p[index])
        running = max(running, value)
        adjusted[index] = running
    return adjusted


def software_manifest() -> dict[str, object]:
    return {
        "python": sys.version,
        "platform": platform.platform(),
        "processor": platform.processor(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": scipy.__version__,
        "scikit_learn": sklearn.__version__,
    }


def write_json(path: str | Path, value: object) -> None:
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")


### `src/egms_studies23/plots.py`

In [ ]:
%%writefile src/egms_studies23/plots.py
from __future__ import annotations

import os
from pathlib import Path
import tempfile
from xml.etree import ElementTree
import zlib

_MPL_CACHE = Path(tempfile.gettempdir()) / "egms_studies23_matplotlib"
_MPL_CACHE.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(_MPL_CACHE))

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


COLORS = {
    "Full": "#176B87",
    "Equal weighting": "#64A7A0",
    "No alignment": "#D95F59",
    "No temporal consistency": "#E7A84B",
    "No modality dropout-distillation": "#7B6BA8",
    "Full temporal graph": "#176B87",
    "No graph": "#D95F59",
    "Spatial-only graph": "#E7A84B",
    "No predicted-intent gating": "#7B6BA8",
    "Single mode K=1": "#777777",
}


DISPLAY_LABELS = {
    "Full": "Full",
    "Equal weighting": "Equal",
    "No alignment": "No\nalign.",
    "No temporal consistency": "No\ntemporal",
    "No modality dropout-distillation": "No\ndropout",
    "Full temporal graph": "Full graph",
    "No graph": "No graph",
    "Spatial-only graph": "Spatial",
    "No predicted-intent gating": "No intent gate",
    "Single mode K=1": "K=1",
}


def _style() -> None:
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 9,
            "axes.titlesize": 10,
            "axes.labelsize": 9,
            "legend.fontsize": 8,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "figure.facecolor": "white",
            "axes.facecolor": "white",
        }
    )


def save_figure(fig: plt.Figure, directory: Path, stem: str) -> None:
    directory.mkdir(parents=True, exist_ok=True)
    specifications = (("png", 600), ("pdf", None), ("svg", None))
    for file_format, dpi in specifications:
        destination = directory / f"{stem}.{file_format}"
        temporary = directory / f".{stem}.{file_format}.tmp"
        for attempt in range(3):
            if temporary.exists():
                temporary.unlink()
            kwargs = {
                "format": file_format,
                "bbox_inches": "tight",
                "facecolor": "white",
            }
            if dpi is not None:
                kwargs["dpi"] = dpi
            fig.savefig(temporary, **kwargs)
            if _valid_figure_file(temporary, file_format):
                os.replace(temporary, destination)
                break
            if attempt == 2:
                raise RuntimeError(f"Failed to write a valid {file_format.upper()} figure: {stem}")
    plt.close(fig)


def _valid_figure_file(path: Path, file_format: str) -> bool:
    data = path.read_bytes()
    if file_format == "pdf":
        return data.startswith(b"%PDF-") and b"%%EOF" in data[-1024:]
    if file_format == "svg":
        try:
            ElementTree.parse(path)
            return True
        except ElementTree.ParseError:
            return False
    if not data.startswith(b"\x89PNG\r\n\x1a\n"):
        return False
    position = 8
    while position + 12 <= len(data):
        length = int.from_bytes(data[position : position + 4], "big")
        chunk_type = data[position + 4 : position + 8]
        payload_end = position + 8 + length
        chunk_end = payload_end + 4
        if chunk_end > len(data):
            return False
        expected = int.from_bytes(data[payload_end:chunk_end], "big")
        observed = zlib.crc32(chunk_type + data[position + 8 : payload_end])
        if expected != observed:
            return False
        position = chunk_end
        if chunk_type == b"IEND":
            return position == len(data)
    return False


def _metric_block(table: pd.DataFrame, metric: str, **filters: object) -> pd.DataFrame:
    block = table[table["metric"] == metric]
    for key, value in filters.items():
        block = block[block[key] == value]
    return block.copy()


def _bar_with_ci(ax: plt.Axes, block: pd.DataFrame, category: str, title: str, ylabel: str) -> None:
    labels = block[category].tolist()
    values = block["estimate"].to_numpy(float)
    lower = values - block["ci_low"].to_numpy(float)
    upper = block["ci_high"].to_numpy(float) - values
    positions = np.arange(len(labels))
    ax.bar(
        positions,
        values,
        yerr=np.vstack([lower, upper]),
        capsize=3,
        color=[COLORS.get(label, "#4C78A8") for label in labels],
        edgecolor="white",
        linewidth=0.5,
    )
    display_labels = [DISPLAY_LABELS.get(label, label) for label in labels]
    ax.set_xticks(positions, display_labels, rotation=0, fontsize=7)
    ax.tick_params(axis="x", pad=3)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", color="#D9D9D9", linewidth=0.6, alpha=0.7)


def plot_all(
    s2_results: dict[str, pd.DataFrame],
    s3_results: dict[str, object],
    s2_tables: dict[str, pd.DataFrame],
    s3_tables: dict[str, pd.DataFrame],
    directory: Path,
) -> None:
    _style()
    plot_s2_alignment(s2_tables["table_s2_main"], directory)
    plot_s2_predictive(s2_tables["table_s2_main"], directory)
    plot_s2_missing(s2_results["study2_missing_modality"], directory)
    plot_s2_calibration(s2_results["study2_calibration_bins"], directory)
    plot_s3_metrics(s3_tables["table_s3_main"], directory)
    plot_s3_regimes(s3_tables["table_s3_by_regime"], directory)
    plot_s3_intent_latency(s3_tables, directory)
    plot_s3_confusion(s3_results["study3_predictions"], directory)
    plot_s3_examples(s3_results["study3_examples"], directory)


def plot_s2_alignment(table: pd.DataFrame, directory: Path) -> None:
    clean = table[table["domain"] == "clean_controlled"]
    fig, axes = plt.subplots(1, 3, figsize=(10.8, 4.3))
    _bar_with_ci(axes[0], _metric_block(clean, "SAS (Eq. 40)"), "method", "Standardized alignment", "SAS")
    _bar_with_ci(axes[1], _metric_block(clean, "Bidirectional Recall@1"), "method", "Cross-modal retrieval", "Recall@1")
    _bar_with_ci(axes[2], _metric_block(clean, "Held-out one-step feature prediction error"), "method", "Temporal prediction", "Prediction error")
    fig.suptitle("Study 2 controlled synthetic alignment and temporal diagnostics", fontweight="bold")
    fig.text(
        0.5,
        0.01,
        "Secondary error bars: 95% t CI across training replicates, conditional on fixed scenes; "
        "scene-decomposable primary contrasts use paired crossed bootstrap (SAS uses a paired replicate-only CI).",
        ha="center",
        fontsize=7.5,
    )
    fig.tight_layout(rect=[0, 0.08, 1, 0.92])
    save_figure(fig, directory, "figure_s2_1_alignment_temporal")


def plot_s2_predictive(table: pd.DataFrame, directory: Path) -> None:
    stress = table[table["domain"] == "adverse_controlled_injected"]
    fig, axes = plt.subplots(1, 4, figsize=(13.2, 4.3))
    for ax, metric, title, ylabel in zip(
        axes,
        ["Action macro-F1", "NLL", "Multiclass Brier (0-2)", "ECE (15 equal-mass bins)"],
        ["Action classification", "Proper log score", "Quadratic score", "Calibration gap"],
        ["Macro-F1", "NLL", "Brier", "ECE"],
    ):
        _bar_with_ci(ax, _metric_block(stress, metric), "method", title, ylabel)
    fig.suptitle("Study 2 performance under controlled injected degradation", fontweight="bold")
    fig.text(
        0.5,
        0.01,
        "Macro-F1: higher is better; NLL/Brier/ECE: lower is better. Controlled injected perturbations are not CARLA natural-weather or RADIATE evidence.\n"
        "Secondary error bars: 95% t CI across training replicates, conditional on fixed scenes; the primary contrast uses paired crossed bootstrap.",
        ha="center",
        va="bottom",
        fontsize=7.2,
    )
    fig.tight_layout(rect=[0, 0.11, 1, 0.92])
    save_figure(fig, directory, "figure_s2_2_predictive_stress")


def plot_s2_missing(frame: pd.DataFrame, directory: Path) -> None:
    patterns = [p for p in frame["missing_pattern"].drop_duplicates() if p != "none"]
    methods = frame["method"].drop_duplicates().tolist()
    means = frame.groupby(["method", "missing_pattern"])["macro_f1_absolute_drop"].mean()
    matrix = np.array([[means.loc[(method, pattern)] for pattern in patterns] for method in methods])
    fig, ax = plt.subplots(figsize=(7.8, 4.1))
    limit = max(abs(matrix.min()), abs(matrix.max()), 0.01)
    image = ax.imshow(matrix, cmap="RdBu_r", vmin=-limit, vmax=limit, aspect="auto")
    ax.set_xticks(np.arange(len(patterns)), patterns)
    ax.set_yticks(np.arange(len(methods)), methods)
    for row in range(matrix.shape[0]):
        for col in range(matrix.shape[1]):
            value = 0.0 if abs(matrix[row, col]) < 0.0005 else matrix[row, col]
            ax.text(col, row, f"{value:+.3f}", ha="center", va="center", fontsize=8)
    ax.set_title("Clean - masked macro-F1 difference (controlled synthetic)")
    ax.set_xlabel("Masked modality pattern")
    fig.colorbar(
        image,
        ax=ax,
        label="Clean - masked macro-F1 (positive = drop; negative = apparent gain)",
    )
    fig.tight_layout(rect=[0, 0.02, 1, 1])
    save_figure(fig, directory, "figure_s2_3_missing_modality")


def plot_s2_calibration(frame: pd.DataFrame, directory: Path) -> None:
    summary = frame.groupby("bin", as_index=False).agg(
        mean_confidence=("mean_confidence", "mean"),
        accuracy=("accuracy", "mean"),
        count=("count", "mean"),
    )
    fig, axes = plt.subplots(2, 1, figsize=(5.6, 5.8), gridspec_kw={"height_ratios": [3, 1]}, sharex=True)
    axes[0].plot([0, 1], [0, 1], color="#777777", linestyle="--", linewidth=1, label="Perfect calibration")
    active = summary["count"] > 0
    axes[0].plot(summary.loc[active, "mean_confidence"], summary.loc[active, "accuracy"], marker="o", color=COLORS["Full"], label="Full")
    axes[0].set_ylabel("Observed accuracy")
    axes[0].set_title("Study 2 reliability diagram, clean controlled test")
    axes[0].legend(frameon=False)
    axes[0].grid(color="#D9D9D9", linewidth=0.6)
    axes[1].bar(summary["mean_confidence"].fillna((summary["bin"] - 0.5) / 15), summary["count"], width=0.045, color="#90B6C5")
    axes[1].set_xlabel("Predicted confidence")
    axes[1].set_ylabel("Mean frames per replicate")
    fig.text(
        0.5,
        0.01,
        "15 deterministic equal-mass confidence bins; bin statistics are averaged across 10 training replicates.",
        ha="center",
        fontsize=7.5,
    )
    fig.tight_layout(rect=[0, 0.06, 1, 1])
    save_figure(fig, directory, "figure_s2_4_reliability_diagram")


def plot_s3_metrics(table: pd.DataFrame, directory: Path) -> None:
    k6 = table[(table["K"] == 6) & table["method"].ne("Single mode K=1")]
    fig, axes = plt.subplots(1, 4, figsize=(13.0, 4.2))
    for ax, metric, title, ylabel in zip(
        axes,
        ["minADE_6 (m)", "minFDE_6 (m)", "MR_6 at 2 m", "Brier-minFDE_6"],
        ["ADE of minFDE-selected\ntrajectory", "Best endpoint error", "Endpoint miss rate", "Probability-aware endpoint"],
        ["m", "m", "Rate", "Score"],
    ):
        _bar_with_ci(ax, _metric_block(k6, metric), "method", title, ylabel)
    fig.suptitle("Study 3 K=6 controlled synthetic trajectory metrics", fontweight="bold")
    fig.text(
        0.5,
        0.01,
        "j* = argmin_j FDE_j; the same j* defines minADE_6 and Brier-minFDE_6 = FDE_j* + (1 - p_j*)^2. Lower is better.\n"
        "Secondary error bars: 95% t CI across training replicates, conditional on fixed scenes; primary contrasts use paired crossed bootstrap. "
        "Controlled synthetic only; not an Argoverse 2 benchmark result.",
        ha="center",
        va="bottom",
        fontsize=7.0,
    )
    fig.tight_layout(rect=[0, 0.13, 1, 0.92])
    save_figure(fig, directory, "figure_s3_1_trajectory_metrics")


def plot_s3_regimes(table: pd.DataFrame, directory: Path) -> None:
    block = table[(table["metric"] == "minFDE_K (m)") & table["method"].isin(["Full temporal graph", "No graph", "Spatial-only graph"])]
    regimes = ["independent", "spatial", "history_dependent"]
    methods = ["Full temporal graph", "No graph", "Spatial-only graph"]
    x = np.arange(len(regimes))
    width = 0.24
    fig, ax = plt.subplots(figsize=(7.4, 4.0))
    for index, method in enumerate(methods):
        selected = block[block["method"] == method].set_index("regime").loc[regimes]
        values = selected["estimate"].to_numpy(float)
        yerr = np.vstack([values - selected["ci_low"], selected["ci_high"] - values])
        ax.bar(x + (index - 1) * width, values, width, yerr=yerr, capsize=3, color=COLORS[method], label=method)
    ax.set_xticks(x, ["Independent-agent\nnull", "Spatial-only\ninteraction", "History-dependent\ninteraction"])
    ax.set_ylabel("minFDE₆ (m)")
    ax.set_title("minFDE₆ by prespecified synthetic interaction regime")
    ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.17))
    ax.grid(axis="y", color="#D9D9D9", linewidth=0.6)
    fig.text(
        0.5,
        0.01,
        "Lower is better. Secondary error bars: 95% t CI across training replicates, conditional on fixed scenes; primary contrasts use paired crossed bootstrap.",
        ha="center",
        fontsize=7.5,
    )
    fig.tight_layout(rect=[0, 0.08, 1, 0.92])
    save_figure(fig, directory, "figure_s3_2_mechanism_regimes")


def plot_s3_intent_latency(tables: dict[str, pd.DataFrame], directory: Path) -> None:
    main = _metric_block(tables["table_s3_main"], "Synthetic dominant-maneuver macro-F1")
    latency = _metric_block(tables["table_s3_latency"], "End-to-end batch-1 CPU latency P95 (ms)")
    fig, axes = plt.subplots(1, 2, figsize=(10.0, 4.3))
    _bar_with_ci(axes[0], main, "method", "Synthetic maneuver classification", "Macro-F1")
    _bar_with_ci(axes[1], latency, "method", "End-to-end CPU latency", "P95 ms")
    fig.suptitle("Study 3 classification and runtime diagnostics", fontweight="bold")
    fig.text(
        0.5,
        0.01,
        "Macro-F1 error bars: secondary 95% t CI across training replicates, conditional on fixed scenes; primary contrasts use paired crossed bootstrap.\n"
        "Latency bars: median across replicate-level empirical P95 values with a 95% percentile bootstrap CI (10,000 draws); each P95 uses 120 batch-1 scenes on Linux x86_64, with no dedicated warm-up; diagnostic only, not real-time.\n"
        "Full graph, no intent gate, and K=1 share the intent head, so identical macro-F1 is expected.",
        ha="center",
        va="bottom",
        fontsize=6.8,
    )
    fig.tight_layout(rect=[0, 0.17, 1, 0.92])
    save_figure(fig, directory, "figure_s3_3_intent_latency")


def plot_s3_confusion(frame: pd.DataFrame, directory: Path) -> None:
    block = frame[frame["method"] == "Full temporal graph"]
    matrix = np.zeros((8, 8), dtype=float)
    for truth, pred in zip(block["intent_true"].to_numpy(int), block["intent_pred"].to_numpy(int)):
        matrix[truth, pred] += 1
    matrix /= np.maximum(matrix.sum(axis=1, keepdims=True), 1)
    labels = ["straight", "slow", "stop", "cut-in", "lane change", "turn L", "turn R", "cross"]
    fig, ax = plt.subplots(figsize=(6.4, 5.3))
    image = ax.imshow(matrix, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(np.arange(8), labels, rotation=35, ha="right")
    ax.set_yticks(np.arange(8), labels)
    ax.set_xlabel("Predicted synthetic dominant maneuver")
    ax.set_ylabel("True synthetic dominant maneuver")
    ax.set_title("Full temporal graph: row-normalized confusion matrix")
    for row in range(8):
        for col in range(8):
            ax.text(col, row, f"{matrix[row, col]:.2f}", ha="center", va="center", fontsize=7, color="white" if matrix[row, col] > 0.55 else "black")
    fig.colorbar(image, ax=ax, label="Row proportion")
    fig.text(
        0.5,
        0.01,
        "Row-normalized predictions pooled across 10 training replicates; all 8 synthetic dynamic maneuver classes have test support.",
        ha="center",
        fontsize=7.5,
    )
    fig.tight_layout(rect=[0, 0.06, 1, 1])
    save_figure(fig, directory, "figure_s3_4_intent_confusion")


def plot_s3_examples(payload: dict[str, np.ndarray], directory: Path) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(8.2, 7.8))
    for index, ax in enumerate(axes.flat):
        history = payload["history"][index]
        future = payload["future"][index]
        candidates = payload["candidates"][index]
        probabilities = payload["probabilities"][index]
        ax.plot(history[:, 0], history[:, 1], color="#555555", linewidth=2, label="Observed history")
        for mode in range(candidates.shape[0]):
            alpha = 0.25 + 0.65 * probabilities[mode] / max(probabilities.max(), 1e-12)
            label = "Predicted modes (opacity proportional to probability)" if mode == 0 else "_nolegend_"
            ax.plot(candidates[mode, :, 0], candidates[mode, :, 1], color="#4C9F70", alpha=alpha, linewidth=1.2, label=label)
        ax.plot(future[:, 0], future[:, 1], color="#D95F59", linewidth=2.2, label="Realized future")
        ax.scatter([0], [0], color="#176B87", s=28, zorder=5)
        ax.set_title(str(payload["scene_ids"][index]))
        ax.set_aspect("equal", adjustable="datalim")
        ax.grid(color="#E5E5E5", linewidth=0.5)
        ax.set_xlabel("Longitudinal displacement (m)")
        ax.set_ylabel("Lateral displacement (m)")
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, 0.945))
    fig.suptitle("Full temporal graph K=6 trajectory examples (controlled synthetic)", y=0.99, fontweight="bold")
    fig.text(
        0.5,
        0.01,
        "Fixed test-scene indices [3, 19, 41, 77] from the first prespecified training replicate; examples were not error-selected.",
        ha="center",
        fontsize=7.5,
    )
    fig.tight_layout(rect=[0, 0.06, 1, 0.87])
    save_figure(fig, directory, "figure_s3_5_trajectory_examples")


### `src/egms_studies23/reporting.py`

In [ ]:
%%writefile src/egms_studies23/reporting.py
from __future__ import annotations

import math
from pathlib import Path

import numpy as np
import pandas as pd
from .common import holm_adjust, seed_summary, stable_seed


EVIDENCE_NOTE = (
    "Controlled synthetic mechanism validation only; not CARLA, nuScenes, "
    "RADIATE, Argoverse 2, real-weather, or real-world evidence."
)


def markdown_table(frame: pd.DataFrame) -> str:
    values = frame.copy()
    for column in values.columns:
        values[column] = values[column].map(_format_markdown_value)
    header = "| " + " | ".join(map(str, values.columns)) + " |"
    separator = "| " + " | ".join("---" for _ in values.columns) + " |"
    rows = [
        "| " + " | ".join(str(value).replace("|", "\\|") for value in row) + " |"
        for row in values.itertuples(index=False, name=None)
    ]
    return "\n".join([header, separator, *rows])


def _format_markdown_value(value: object) -> str:
    if value is None or (isinstance(value, float) and not np.isfinite(value)):
        return "N/A"
    if isinstance(value, (float, np.floating)):
        return f"{float(value):.4f}"
    return str(value)


def write_table(frame: pd.DataFrame, output_dir: Path, stem: str) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    frame.to_csv(output_dir / f"{stem}.csv", index=False)
    (output_dir / f"{stem}.md").write_text(markdown_table(frame), encoding="utf-8")
    (output_dir / f"{stem}.tex").write_text(_latex_table(frame), encoding="utf-8")


def _latex_escape(value: object) -> str:
    text = _format_markdown_value(value)
    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    return "".join(replacements.get(character, character) for character in text)


def _latex_table(frame: pd.DataFrame) -> str:
    alignment = "l" * len(frame.columns)
    lines = [f"\\begin{{tabular}}{{{alignment}}}", r"\hline"]
    lines.append(" & ".join(_latex_escape(column) for column in frame.columns) + r" \\")
    lines.append(r"\hline")
    for row in frame.itertuples(index=False, name=None):
        lines.append(" & ".join(_latex_escape(value) for value in row) + r" \\")
    lines.extend([r"\hline", r"\end{tabular}", ""])
    return "\n".join(lines)


def _summarize_long(
    frame: pd.DataFrame,
    group_columns: list[str],
    metrics: list[tuple[str, str, str]],
    confidence: float = 0.95,
) -> pd.DataFrame:
    rows = []
    for keys, group in frame.groupby(group_columns, sort=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        identity = dict(zip(group_columns, keys))
        for column, label, better in metrics:
            estimate, low, high = seed_summary(group[column].to_numpy(float), confidence)
            rows.append(
                {
                    **identity,
                    "metric": label,
                    "better": better,
                    "estimate": estimate,
                    "ci_low": low,
                    "ci_high": high,
                    "training_replicates": group["training_seed"].nunique(),
                    "ci_definition": "95% t CI across independent training-sample replicates; fixed test scenes",
                    "evidence": EVIDENCE_NOTE,
                }
            )
    return pd.DataFrame(rows)


def _summarize_replicate_bootstrap_long(
    frame: pd.DataFrame,
    group_columns: list[str],
    metrics: list[tuple[str, str, str]],
    confidence: float,
    repetitions: int,
) -> pd.DataFrame:
    rows = []
    for keys, group in frame.groupby(group_columns, sort=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        identity = dict(zip(group_columns, keys))
        for column, label, better in metrics:
            values = group[column].to_numpy(float)
            values = values[np.isfinite(values)]
            rng = np.random.default_rng(stable_seed("replicate_bootstrap", *keys, column))
            draws = np.median(
                rng.choice(values, size=(int(repetitions), len(values)), replace=True),
                axis=1,
            )
            alpha = 1.0 - confidence
            low, high = np.quantile(draws, [alpha / 2, 1.0 - alpha / 2])
            rows.append(
                {
                    **identity,
                    "metric": label,
                    "better": better,
                    "estimate": float(np.median(values)),
                    "ci_low": float(low),
                    "ci_high": float(high),
                    "training_replicates": group["training_seed"].nunique(),
                    "ci_definition": f"Median across training-replicate latency summaries with {int(confidence * 100)}% percentile bootstrap CI ({repetitions} draws)",
                    "evidence": EVIDENCE_NOTE,
                }
            )
    return pd.DataFrame(rows)


def study2_tables(results: dict[str, pd.DataFrame], cfg: dict | None = None) -> dict[str, pd.DataFrame]:
    confidence = float(cfg["common"]["confidence_level"]) if cfg else 0.95
    repetitions = int(cfg["common"]["bootstrap_repetitions"]) if cfg else 10000
    seed_metrics = results["study2_seed_metrics"]
    main = _summarize_long(
        seed_metrics,
        ["method", "domain"],
        [
            ("macro_f1", "Action macro-F1", "higher"),
            ("nll", "NLL", "lower"),
            ("brier", "Multiclass Brier (0-2)", "lower"),
            ("ece", "ECE (15 equal-mass bins)", "lower"),
            ("sas", "SAS (Eq. 40)", "higher"),
            ("recall_at_1", "Bidirectional Recall@1", "higher"),
            ("recall_at_5", "Bidirectional Recall@5", "higher"),
            ("recall_at_10", "Bidirectional Recall@10", "higher"),
            ("feature_drift", "Held-out one-step feature prediction error", "lower"),
            ("representation_variance", "Representation variance", "diagnostic"),
        ], confidence,
    )
    missing = _summarize_long(
        results["study2_missing_modality"],
        ["method", "missing_pattern"],
        [
            ("macro_f1", "Action macro-F1", "higher"),
            ("macro_f1_absolute_drop", "Absolute macro-F1 drop", "lower"),
            ("macro_f1_relative_drop", "Relative macro-F1 drop", "lower"),
            ("nll", "NLL", "lower"),
            ("brier", "Multiclass Brier (0-2)", "lower"),
            ("ece", "ECE", "lower"),
        ], confidence,
    )
    latency = _summarize_replicate_bootstrap_long(
        results["study2_latency"],
        ["method"],
        [("p50_ms", "Batch-1 CPU latency P50 (ms)", "lower"), ("p95_ms", "Batch-1 CPU latency P95 (ms)", "lower")], confidence, repetitions,
    )
    contrasts = _study2_contrasts(results, cfg)
    return {
        "table_s2_main": main,
        "table_s2_missing_modality": missing,
        "table_s2_primary_contrasts": contrasts,
        "table_s2_latency": latency,
    }


def _paired_contrast(
    full: pd.Series,
    ablation: pd.Series,
    comparison: str,
    endpoint: str,
    favorable_sign: str,
    confidence: float = 0.95,
) -> dict[str, object]:
    joined = pd.concat([full.rename("full"), ablation.rename("ablation")], axis=1).dropna()
    difference = joined["full"] - joined["ablation"]
    difference_values = difference.to_numpy(float)
    estimate, low, high = seed_summary(difference_values, confidence)
    p_value = _exact_sign_flip_p(difference_values)
    if favorable_sign == "negative":
        favorable = difference < 0
        direction_met = bool(high < 0 and favorable.sum() >= math.ceil(0.8 * len(difference)))
    else:
        favorable = difference > 0
        direction_met = bool(low > 0 and favorable.sum() >= math.ceil(0.8 * len(difference)))
    return {
        "comparison": comparison,
        "primary_endpoint": endpoint,
        "difference_full_minus_ablation": estimate,
        "ci_low": low,
        "ci_high": high,
        "favorable_sign": favorable_sign,
        "favorable_training_replicates": int(favorable.sum()),
        "total_training_replicates": len(difference),
        "paired_exact_sign_flip_p": p_value,
        "ci_method": "95% t CI across paired training replicates; fixed test scenes",
        "direction_ci_consistency_met": direction_met,
        "evidence": EVIDENCE_NOTE,
    }


def _exact_sign_flip_p(difference: np.ndarray) -> float:
    values = np.asarray(difference, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return float("nan")
    combinations = np.arange(2 ** len(values), dtype=np.uint64)[:, None]
    bits = (combinations >> np.arange(len(values), dtype=np.uint64)) & 1
    signs = 1.0 - 2.0 * bits
    distribution = (signs * values[None, :]).mean(axis=1)
    return float(np.mean(np.abs(distribution) >= abs(values.mean()) - 1.0e-15))


def _paired_crossed_contrast(
    frame: pd.DataFrame,
    full_method: str,
    ablation_method: str,
    metric: str,
    comparison: str,
    endpoint: str,
    favorable_sign: str,
    repetitions: int,
    random_seed: int,
    confidence: float,
) -> dict[str, object]:
    grouped = frame.groupby(["method", "training_seed", "scene_id"], as_index=False)[metric].mean()
    full = grouped[grouped["method"] == full_method].set_index(["training_seed", "scene_id"])[metric]
    ablation = grouped[grouped["method"] == ablation_method].set_index(["training_seed", "scene_id"])[metric]
    full = full.sort_index()
    ablation = ablation.sort_index()
    if not full.index.equals(ablation.index):
        raise ValueError(f"Full and ablation replicate-scene indices differ for {comparison}")
    joined = pd.concat([full.rename("full"), ablation.rename("ablation")], axis=1)
    if joined.isna().any().any():
        raise ValueError(f"Missing paired replicate-scene value for {comparison}")
    matrix = (joined["full"] - joined["ablation"]).unstack("scene_id").sort_index()
    if matrix.isna().any().any():
        raise ValueError(f"Incomplete replicate-scene grid for {comparison}")
    values = matrix.to_numpy(float)
    estimate = float(values.mean())
    rng = np.random.default_rng(random_seed)
    bootstrap = np.empty(int(repetitions), dtype=float)
    batch_size = 200
    for start in range(0, int(repetitions), batch_size):
        stop = min(start + batch_size, int(repetitions))
        batch = stop - start
        sampled_replicates = rng.integers(0, values.shape[0], size=(batch, values.shape[0]))
        sampled_scenes = rng.integers(0, values.shape[1], size=(batch, values.shape[1]))
        sampled = values[sampled_replicates[:, :, None], sampled_scenes[:, None, :]]
        bootstrap[start:stop] = sampled.mean(axis=(1, 2))
    alpha = 1.0 - confidence
    low, high = np.quantile(bootstrap, [alpha / 2, 1.0 - alpha / 2])
    replicate_difference = values.mean(axis=1)
    favorable = replicate_difference < 0 if favorable_sign == "negative" else replicate_difference > 0
    if favorable_sign == "negative":
        direction_met = bool(high < 0 and favorable.sum() >= math.ceil(0.8 * len(favorable)))
    else:
        direction_met = bool(low > 0 and favorable.sum() >= math.ceil(0.8 * len(favorable)))
    return {
        "comparison": comparison,
        "primary_endpoint": endpoint,
        "difference_full_minus_ablation": estimate,
        "ci_low": float(low),
        "ci_high": float(high),
        "favorable_sign": favorable_sign,
        "favorable_training_replicates": int(favorable.sum()),
        "total_training_replicates": len(favorable),
        "paired_exact_sign_flip_p": _exact_sign_flip_p(replicate_difference),
        "ci_method": f"{int(confidence * 100)}% paired crossed bootstrap over training replicate and scene ({repetitions} draws)",
        "direction_ci_consistency_met": direction_met,
        "evidence": EVIDENCE_NOTE,
    }


def _finalize_contrasts(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    frame["holm_adjusted_p"] = holm_adjust(frame["paired_exact_sign_flip_p"].to_numpy(float))
    frame["mechanism_support_rule_met"] = (
        frame["direction_ci_consistency_met"] & (frame["holm_adjusted_p"] < 0.05)
    )
    return frame


def _study2_contrasts(results: dict[str, pd.DataFrame], cfg: dict | None) -> pd.DataFrame:
    metrics = results["study2_seed_metrics"]
    scene = results["study2_scene_metrics"]
    missing_scene = results["study2_missing_scene_metrics"]
    repetitions = int(cfg["common"]["bootstrap_repetitions"]) if cfg else 10000
    confidence = float(cfg["common"]["confidence_level"]) if cfg else 0.95
    clean = metrics[metrics["domain"] == "clean_controlled"].set_index(["method", "training_seed"])
    stress = metrics[metrics["domain"] == "adverse_controlled_injected"].set_index(["method", "training_seed"])
    rows = []
    rows.append(
        _paired_crossed_contrast(
            scene[scene["domain"] == "adverse_controlled_injected"],
            "Full",
            "Equal weighting",
            "nll",
            "Full vs Equal weighting",
            "Adverse controlled-injection NLL",
            "negative",
            repetitions,
            stable_seed("study2", "equal_weighting", "bootstrap"),
            confidence,
        )
    )
    rows.append(
        _paired_contrast(
            clean.loc["Full", "sas"],
            clean.loc["No alignment", "sas"],
            "Full vs No alignment",
            "Clean controlled SAS",
            "positive",
            confidence,
        )
    )
    rows.append(
        _paired_crossed_contrast(
            scene[scene["domain"] == "adverse_controlled_injected"],
            "Full",
            "No temporal consistency",
            "feature_drift",
            "Full vs No temporal consistency",
            "Adverse held-out one-step feature prediction error",
            "negative",
            repetitions,
            stable_seed("study2", "temporal", "bootstrap"),
            confidence,
        )
    )
    active_missing = (
        missing_scene[missing_scene["missing_pattern"] != "none"]
        .groupby(["method", "training_seed", "scene_id"], as_index=False)["macro_f1_absolute_drop"]
        .mean()
    )
    rows.append(
        _paired_crossed_contrast(
            active_missing,
            "Full",
            "No modality dropout-distillation",
            "macro_f1_absolute_drop",
            "Full vs No modality dropout-distillation",
            "Mean scene-level single/dual missing-modality macro-F1 drop",
            "negative",
            repetitions,
            stable_seed("study2", "missing", "bootstrap"),
            confidence,
        )
    )
    return _finalize_contrasts(pd.DataFrame(rows))


def study3_tables(results: dict[str, pd.DataFrame], cfg: dict | None = None) -> dict[str, pd.DataFrame]:
    confidence = float(cfg["common"]["confidence_level"]) if cfg else 0.95
    repetitions = int(cfg["common"]["bootstrap_repetitions"]) if cfg else 10000
    seed_metrics = results["study3_seed_metrics"]
    main = _summarize_long(
        seed_metrics,
        ["method", "K"],
        [
            ("intent_macro_f1", "Synthetic dominant-maneuver macro-F1", "higher"),
            ("ADE_1", "ADE_1 (m)", "lower"),
            ("FDE_1", "FDE_1 (m)", "lower"),
            ("MR_1", "MR_1 at 2 m", "lower"),
            ("minADE_6_reportable", "minADE_6 (m)", "lower"),
            ("minFDE_6_reportable", "minFDE_6 (m)", "lower"),
            ("MR_6_reportable", "MR_6 at 2 m", "lower"),
            ("Brier-minFDE_6_reportable", "Brier-minFDE_6", "lower"),
        ], confidence,
    )
    predictions = results["study3_predictions"]
    per_seed_regime = (
        predictions.groupby(["method", "training_seed", "regime"], as_index=False)[
            ["minADE_K", "minFDE_K", "MR_K", "Brier-minFDE_K", "ADE_1", "FDE_1", "MR_1"]
        ]
        .mean()
    )
    regime = _summarize_long(
        per_seed_regime,
        ["method", "regime"],
        [
            ("minADE_K", "minADE_K (m)", "lower"),
            ("minFDE_K", "minFDE_K (m)", "lower"),
            ("MR_K", "MR_K at 2 m", "lower"),
            ("Brier-minFDE_K", "Brier-minFDE_K", "lower"),
        ], confidence,
    )
    latency = _summarize_replicate_bootstrap_long(
        results["study3_latency"],
        ["method"],
        [("p50_ms", "End-to-end batch-1 CPU latency P50 (ms)", "lower"), ("p95_ms", "End-to-end batch-1 CPU latency P95 (ms)", "lower")], confidence, repetitions,
    )
    contrasts, null_checks = _study3_contrasts(results, cfg)
    return {
        "table_s3_main": main,
        "table_s3_by_regime": regime,
        "table_s3_primary_contrasts": contrasts,
        "table_s3_negative_controls": null_checks,
        "table_s3_latency": latency,
    }


def _study3_contrasts(
    results: dict[str, pd.DataFrame], cfg: dict | None
) -> tuple[pd.DataFrame, pd.DataFrame]:
    predictions = results["study3_predictions"]
    repetitions = int(cfg["common"]["bootstrap_repetitions"]) if cfg else 10000
    confidence = float(cfg["common"]["confidence_level"]) if cfg else 0.95
    rows = []
    for ablation in ("No graph", "Spatial-only graph"):
        rows.append(
            _paired_crossed_contrast(
                predictions[predictions["regime"] == "history_dependent"],
                "Full temporal graph",
                ablation,
                "minFDE_K",
                f"Full temporal graph vs {ablation}",
                "History-dependent minFDE_6",
                "negative",
                repetitions,
                stable_seed("study3", ablation, "history", "bootstrap"),
                confidence,
            )
        )
    rows.append(
        _paired_crossed_contrast(
            predictions,
            "Full temporal graph",
            "No predicted-intent gating",
            "Brier-minFDE_K",
            "Full temporal graph vs No predicted-intent gating",
            "Brier-minFDE_6",
            "negative",
            repetitions,
            stable_seed("study3", "intent_gating", "bootstrap"),
            confidence,
        )
    )
    rows.append(
        _paired_crossed_contrast(
            predictions,
            "Full temporal graph",
            "Single mode K=1",
            "MR_1",
            "Full top-1 vs independently trained Single mode K=1",
            "MR_1 at 2 m",
            "negative",
            repetitions,
            stable_seed("study3", "single_mode", "bootstrap"),
            confidence,
        )
    )
    contrasts = _finalize_contrasts(pd.DataFrame(rows))

    null_rows = []
    for regime_name, ablation, margin in (
        ("independent", "No graph", 0.25),
        ("spatial", "Spatial-only graph", 0.25),
    ):
        contrast = _paired_crossed_contrast(
            predictions[predictions["regime"] == regime_name],
            "Full temporal graph",
            ablation,
            "minFDE_K",
            f"{regime_name}: Full vs {ablation}",
            "minFDE_K (m)",
            "negative",
            repetitions,
            stable_seed("study3", regime_name, "null", "bootstrap"),
            confidence,
        )
        null_rows.append(
            {
                "negative_control": f"{regime_name}: Full vs {ablation}",
                "endpoint": "minFDE_K (m)",
                "equivalence_margin": margin,
                "difference_full_minus_ablation": contrast["difference_full_minus_ablation"],
                "ci_low": contrast["ci_low"],
                "ci_high": contrast["ci_high"],
                "descriptive_equivalence_met": bool(
                    contrast["ci_low"] >= -margin and contrast["ci_high"] <= margin
                ),
                "ci_method": contrast["ci_method"],
                "note": "Prespecified descriptive margin; entire crossed-bootstrap CI must lie within it; not TOST.",
            }
        )
    return contrasts, pd.DataFrame(null_rows)


def execution_status_table(cfg: dict) -> pd.DataFrame:
    urls = {
        "CARLA_0.9.16": "https://carla.readthedocs.io/en/0.9.16/ref_sensors/",
        "nuScenes": "https://www.nuscenes.org/",
        "RADIATE": "https://pro.hw.ac.uk/radiate/",
        "Argoverse_2_Motion": "https://argoverse.github.io/user-guide/tasks/motion_forecasting.html",
    }
    rows = [
        {
            "source": source,
            "status": status,
            "raw_files": 0,
            "scenes_or_logs": 0,
            "checkpoints": 0,
            "official_evaluator_outputs": 0,
            "result_claim_allowed": False,
            "official_source": urls[source],
        }
        for source, status in cfg["execution_status"].items()
    ]
    rows.insert(
        0,
        {
            "source": "Controlled_synthetic_generator",
            "status": "EXECUTED",
            "raw_files": "generated from frozen code/seeds",
            "scenes_or_logs": "see split manifests",
            "checkpoints": "lightweight fitted models; hashes in manifest",
            "official_evaluator_outputs": 0,
            "result_claim_allowed": True,
            "official_source": "N/A; controlled data-generating mechanism",
        },
    )
    return pd.DataFrame(rows)


def parameter_table(cfg: dict) -> pd.DataFrame:
    rows: list[dict[str, object]] = []

    def visit(prefix: str, value: object) -> None:
        if isinstance(value, dict):
            for key, item in value.items():
                visit(f"{prefix}.{key}" if prefix else key, item)
        elif isinstance(value, list):
            rows.append({"parameter": prefix, "value": ", ".join(map(str, value))})
        else:
            rows.append({"parameter": prefix, "value": value})

    visit("", cfg)
    return pd.DataFrame(rows)


def manuscript_results(
    s2_tables: dict[str, pd.DataFrame],
    s3_tables: dict[str, pd.DataFrame],
    evidence_label: str,
) -> str:
    s2_contrasts = s2_tables["table_s2_primary_contrasts"].copy()
    s3_contrasts = s3_tables["table_s3_primary_contrasts"].copy()
    return f"""# Controlled synthetic component-validation results

**Evidence label:** `{evidence_label}`

The following text is suitable only for a clearly titled controlled synthetic
mechanism-validation subsection or supplement. It must not be described as a
CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world result.

## Study 2: reliability, alignment, temporal consistency, and missing modalities

Ten independent, equal-sized training-sample replicates were evaluated on
identical frozen synthetic test scenes and corruption realizations. Conditions
within each replicate shared the same training data. Scene-decomposable primary
contrast intervals use a paired crossed bootstrap over training replicate and
test scene; SAS uses a paired replicate-only interval conditional on its fixed
retrieval gallery. The long summary tables additionally show replicate-only
intervals conditional on the fixed test scenes. The mechanism-specific paired
contrasts were:

{markdown_table(s2_contrasts)}

These contrasts should be interpreted separately: SAS diagnoses cross-modal
alignment; held-out one-step feature prediction error diagnoses temporal consistency; adverse NLL
diagnoses probabilistic prediction under controlled injected degradation; and
missing-modality macro-F1 loss diagnoses the combined dropout-distillation
package. They do not establish real-weather robustness.

## Study 3: temporal graph, intent conditioning, and multimodal trajectories

The controlled trajectory task used 5 s observed histories and 6 s futures at
10 Hz. Multi-hypothesis conditions emitted K=6 trajectories; the single-mode
condition emitted K=1 and was not mislabeled with K=6 metrics. Dominant
maneuver labels are synthetic ground truth and are not Argoverse 2 intent
annotations. The prespecified paired contrasts were:

{markdown_table(s3_contrasts)}

The graph comparisons use the history-dependent regime as their primary
mechanism endpoint. The no-gating intervention keeps the same intent head and
24-prototype bank, replacing only the scene-specific predicted-intent posterior
with a training-only fixed prior. Independent-agent and spatial-only regimes
are retained as negative controls. Mixed or unsupported contrasts must remain
reported and must not be removed by post hoc replicate or parameter selection.
"""


### `src/egms_studies23/runner.py`

In [ ]:
%%writefile src/egms_studies23/runner.py
from __future__ import annotations

import argparse
import gc
import json
import shutil
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
import yaml

from .common import sha256, software_manifest, write_json
from .plots import plot_all
from .reporting import (
    execution_status_table,
    manuscript_results,
    parameter_table,
    study2_tables,
    study3_tables,
    write_table,
)
from .study2 import run_study2
from .study3 import run_study3
from .validation import validate_output


def _load_config(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as handle:
        cfg = yaml.safe_load(handle)
    if not isinstance(cfg, dict):
        raise ValueError("Configuration root must be a mapping")
    return cfg


def _validate_config_contract(cfg: dict[str, Any]) -> None:
    expected_s2 = {
        "Equal weighting": "stress_nll",
        "No alignment": "sas",
        "No temporal consistency": "feature_drift",
        "No modality dropout-distillation": "mean_missing_macro_f1_drop",
    }
    expected_s3 = {
        "No graph": "history_dependent_minFDE_K",
        "Spatial-only graph": "history_dependent_minFDE_K",
        "No predicted-intent gating": "Brier-minFDE_K",
        "Single mode K=1": "MR_1",
    }
    if cfg["study2"].get("primary_endpoints") != expected_s2:
        raise ValueError("Study 2 primary endpoint contract differs from the frozen analysis")
    if cfg["study3"].get("primary_endpoints") != expected_s3:
        raise ValueError("Study 3 primary endpoint contract differs from the frozen analysis")
    if int(cfg["study3"]["modes"]) != 6:
        raise ValueError("This implementation requires K=6 for multi-hypothesis conditions")


def _write_frames(frames: dict[str, Any], directory: Path) -> None:
    directory.mkdir(parents=True, exist_ok=True)
    for name, value in frames.items():
        if isinstance(value, pd.DataFrame):
            destination = directory / (
                f"{name}.csv.gz" if name.endswith("predictions") else f"{name}.csv"
            )
            value.to_csv(destination, index=False)


def _write_examples(payload: dict[str, Any], path: Path) -> None:
    import numpy as np

    np.savez_compressed(path, **payload)


def _load_persisted_results(output: Path, prefix: str) -> dict[str, Any]:
    import numpy as np

    results: dict[str, Any] = {}
    for path in sorted((output / "data").glob(f"{prefix}_*.csv*")):
        name = path.name.split(".csv", 1)[0]
        results[name] = pd.read_csv(path)
    if prefix == "study3":
        with np.load(output / "data" / "study3_examples.npz", allow_pickle=True) as payload:
            results["study3_examples"] = {key: payload[key] for key in payload.files}
    return results


def _code_hashes(project_root: Path) -> dict[str, str]:
    candidates = [
        project_root / "run_studies.py",
        project_root / "configs" / "studies23.yaml",
        project_root / "docs" / "STUDIES23_ANALYSIS_PLAN_FROZEN.md",
        project_root / "docs" / "REAL_DATA_EXECUTION_GATE.md",
        project_root / "README.md",
        project_root / "pyproject.toml",
        project_root / "requirements.txt",
        *sorted((project_root / "src" / "egms_studies23").glob("*.py")),
        *sorted((project_root / "tests").glob("*.py")),
    ]
    return {
        str(path.relative_to(project_root)): sha256(path)
        for path in candidates
        if path.is_file()
    }


def _summary_payload(
    cfg: dict[str, Any],
    s2_tables: dict[str, pd.DataFrame],
    s3_tables: dict[str, pd.DataFrame],
) -> dict[str, Any]:
    def records(frame: pd.DataFrame) -> list[dict[str, Any]]:
        return json.loads(frame.to_json(orient="records"))

    s2_main = s2_tables["table_s2_main"]
    s3_main = s3_tables["table_s3_main"]
    s2_full = s2_main[
        (s2_main["method"] == "Full")
        & (s2_main["domain"] == "adverse_controlled_injected")
    ]
    s3_full = s3_main[
        (s3_main["method"] == "Full temporal graph") & (s3_main["K"] == 6)
    ]
    return {
        "evidence_label": cfg["project"]["evidence_label"],
        "interpretation_boundary": (
            "Controlled synthetic mechanism validation only; no CARLA, nuScenes, "
            "RADIATE, or Argoverse 2 files were executed."
        ),
        "formal_training_replicate_seeds": cfg["common"]["training_seeds"],
        "study2_full_adverse_selected_metrics": records(
            s2_full[s2_full["metric"].isin(["Action macro-F1", "NLL", "Multiclass Brier (0-2)", "ECE (15 equal-mass bins)"])]
        ),
        "study2_primary_contrasts": records(s2_tables["table_s2_primary_contrasts"]),
        "study3_full_k6_selected_metrics": records(
            s3_full[s3_full["metric"].isin(["Synthetic dominant-maneuver macro-F1", "minADE_6 (m)", "minFDE_6 (m)", "MR_6 at 2 m", "Brier-minFDE_6"])]
        ),
        "study3_primary_contrasts": records(s3_tables["table_s3_primary_contrasts"]),
        "study3_negative_controls": records(s3_tables["table_s3_negative_controls"]),
    }


def _build_analysis_outputs(
    cfg: dict[str, Any],
    output: Path,
    s2_results: dict[str, Any],
    s3_results: dict[str, Any],
) -> None:
    evidence = cfg["project"]["evidence_label"]
    s2_tables = study2_tables(s2_results, cfg)
    s3_tables = study3_tables(s3_results, cfg)
    tables = {
        "table_execution_status": execution_status_table(cfg),
        "table_parameters": parameter_table(cfg),
        **s2_tables,
        **s3_tables,
    }
    for name, frame in tables.items():
        write_table(frame, output / "tables", name)
    plot_all(s2_results, s3_results, s2_tables, s3_tables, output / "figures")
    manuscript = manuscript_results(s2_tables, s3_tables, evidence)
    (output / "MANUSCRIPT_RESULTS_CONTROLLED_SYNTHETIC.md").write_text(
        manuscript, encoding="utf-8"
    )
    write_json(output / "summary.json", _summary_payload(cfg, s2_tables, s3_tables))


def run(config_path: Path, output: Path, overwrite: bool = False) -> dict[str, Any]:
    config_path = config_path.resolve()
    project_root = Path(__file__).resolve().parents[2]
    cfg = _load_config(config_path)
    _validate_config_contract(cfg)
    initial_code_hashes = _code_hashes(project_root)
    evidence = cfg.get("project", {}).get("evidence_label")
    if evidence != "CONTROLLED_SYNTHETIC_MECHANISM_VALIDATION_NOT_EMPIRICAL_DATASET":
        raise ValueError("The controlled-synthetic evidence label is mandatory")
    if bool(cfg["project"].get("real_datasets_executed")) or bool(cfg["project"].get("carla_executed")):
        raise ValueError("This runner cannot claim CARLA or real-dataset execution")

    allowed_output_root = (project_root / "outputs").resolve()
    resolved_output = output.resolve()
    if resolved_output == allowed_output_root or not resolved_output.is_relative_to(allowed_output_root):
        raise ValueError(f"Output must be a named descendant of {allowed_output_root}")
    if output.is_symlink():
        raise ValueError("Refusing to use a symbolic-link output path")
    if output.exists():
        if not overwrite:
            raise FileExistsError(f"Output already exists: {output}; pass --overwrite to replace it")
        shutil.rmtree(output)
    output.mkdir(parents=True)
    shutil.copy2(config_path, output / "config_resolved.yaml")
    started = datetime.now(timezone.utc)

    print("[1/5] Running Study 2 controlled synthetic ablations", flush=True)
    s2_results = run_study2(cfg)
    print("[2/5] Running Study 3 controlled synthetic ablations", flush=True)
    s3_results = run_study3(cfg)

    data_dir = output / "data"
    _write_frames(s2_results, data_dir)
    _write_frames(s3_results, data_dir)
    _write_examples(s3_results["study3_examples"], data_dir / "study3_examples.npz")

    print("[3/5] Building manuscript tables and figures", flush=True)
    _build_analysis_outputs(cfg, output, s2_results, s3_results)

    finished = datetime.now(timezone.utc)
    final_code_hashes = _code_hashes(project_root)
    if final_code_hashes != initial_code_hashes:
        raise RuntimeError("Source files changed during execution; refusing to write a provenance manifest")
    manifest = {
        "project_id": cfg["project"]["id"],
        "evidence_label": evidence,
        "carla_executed": False,
        "real_datasets_executed": False,
        "named_dataset_result_rows": 0,
        "pilot_training_seeds": cfg["common"]["pilot_training_seeds"],
        "training_seeds": cfg["common"]["training_seeds"],
        "training_replicate_seeds": cfg["common"]["training_seeds"],
        "training_replicate_definition": (
            "Independent equal-sized training samples from the frozen DGM; all methods "
            "within a replicate share the exact training data; validation/test are fixed."
        ),
        "data_seeds": {
            "study2": cfg["study2"]["data_seed"],
            "study3": cfg["study3"]["data_seed"],
        },
        "started_utc": started.isoformat(),
        "finished_utc": finished.isoformat(),
        "elapsed_seconds": (finished - started).total_seconds(),
        "config_path": str(config_path.relative_to(project_root)),
        "config_sha256": sha256(config_path),
        "code_sha256": final_code_hashes,
        "software": software_manifest(),
        "analysis_plan_frozen_before_final_full_run": bool(
            cfg["project"]["analysis_plan_frozen_before_final_full_run"]
        ),
        "preregistered": False,
        "ci_scope": (
            "Scene-decomposable primary contrasts: paired crossed bootstrap over training-sample "
            "replicate and test scene. SAS and long summaries: replicate-only t CI conditional "
            "on the fixed retrieval gallery/test scenes."
        ),
        "selection_rule": "No replicate, method, or parameter was selected based on the final full-run result.",
        "validation_rule_note": "No rule requires the Full condition to win.",
    }
    write_json(output / "run_manifest.json", manifest)

    # Release large raw prediction frames before the validator independently
    # reloads and recomputes metrics from the persisted CSV files.
    del s2_results, s3_results
    gc.collect()
    print("[4/5] Running evidence, integrity, metric, and artifact validation", flush=True)
    report = validate_output(output)
    print("[5/5] PASS: formal controlled-synthetic run validated", flush=True)
    return report


def analyze_existing(output: Path) -> dict[str, Any]:
    output = output.resolve()
    project_root = Path(__file__).resolve().parents[2]
    allowed_output_root = (project_root / "outputs").resolve()
    if output == allowed_output_root or not output.is_relative_to(allowed_output_root):
        raise ValueError(f"Output must be a named descendant of {allowed_output_root}")
    config_path = output / "config_resolved.yaml"
    if not config_path.is_file() or not (output / "run_manifest.json").is_file():
        raise FileNotFoundError("Existing output lacks its resolved config or run manifest")
    cfg = _load_config(config_path)
    _validate_config_contract(cfg)
    initial_code_hashes = _code_hashes(project_root)
    print("[1/3] Loading persisted raw predictions and manifests", flush=True)
    s2_results = _load_persisted_results(output, "study2")
    s3_results = _load_persisted_results(output, "study3")
    print("[2/3] Rebuilding tables, manuscript text, and figures", flush=True)
    _build_analysis_outputs(cfg, output, s2_results, s3_results)
    final_code_hashes = _code_hashes(project_root)
    if final_code_hashes != initial_code_hashes:
        raise RuntimeError("Source files changed during analysis; refusing to update provenance")
    manifest = json.loads((output / "run_manifest.json").read_text(encoding="utf-8"))
    manifest.update(
        {
            "analysis_regenerated_utc": datetime.now(timezone.utc).isoformat(),
            "analysis_revision": (
                "Latency tables use the median across replicate timing summaries with a "
                "percentile-bootstrap interval; "
                "model fits and raw predictions were not rerun or modified."
            ),
            "config_sha256": sha256(config_path),
            "code_sha256": final_code_hashes,
            "software": software_manifest(),
        }
    )
    write_json(output / "run_manifest.json", manifest)
    del s2_results, s3_results
    gc.collect()
    report = validate_output(output)
    print("[3/3] PASS: persisted raw outputs reanalyzed and validated", flush=True)
    return report


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description=__doc__)
    subparsers = parser.add_subparsers(dest="command", required=True)
    run_parser = subparsers.add_parser("run", help="Run both controlled simulations")
    run_parser.add_argument("--config", type=Path, required=True)
    run_parser.add_argument("--output", type=Path, required=True)
    run_parser.add_argument("--overwrite", action="store_true")
    analyze_parser = subparsers.add_parser("analyze", help="Rebuild analysis from persisted raw outputs")
    analyze_parser.add_argument("--output", type=Path, required=True)
    validate_parser = subparsers.add_parser("validate", help="Validate a completed run")
    validate_parser.add_argument("--output", type=Path, required=True)
    return parser


def main() -> None:
    args = build_parser().parse_args()
    if args.command == "run":
        run(args.config, args.output, args.overwrite)
    elif args.command == "analyze":
        analyze_existing(args.output)
    else:
        report = validate_output(args.output)
        print(json.dumps(report, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


### `src/egms_studies23/study2.py`

In [ ]:
%%writefile src/egms_studies23/study2.py
from __future__ import annotations

from dataclasses import asdict, dataclass
import json
from time import perf_counter_ns
from typing import Iterable

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler

from .common import (
    TemperatureScaler,
    array_digest,
    calibration_bins,
    expected_calibration_error,
    macro_f1,
    multiclass_brier,
    negative_log_likelihood,
    normalize_rows,
    softmax,
    stable_seed,
)


ACTION_NAMES = ("KEEP", "SLOW", "YIELD", "STOP")
SCENARIOS = ("cut_in", "platoon_brake", "occluded_crossing", "right_turn", "merge")


def _orthogonal_projection(rng: np.random.Generator, latent_dim: int, raw_dim: int) -> np.ndarray:
    matrix = rng.normal(size=(raw_dim, raw_dim))
    q, _ = np.linalg.qr(matrix)
    return q[:latent_dim, :]


def _weather_for(split: str, index: int) -> str:
    if split == "test":
        return "clear"
    if split == "stress":
        return ("night", "rain", "fog", "snow")[index % 4]
    return ("clear", "night", "rain", "fog")[index % 4]


def _quality_profile(weather: str) -> dict[str, float]:
    return {
        "clear": {"camera": 0.92, "lidar": 0.92, "radar": 0.88, "ego": 0.98},
        "night": {"camera": 0.55, "lidar": 0.90, "radar": 0.87, "ego": 0.98},
        "rain": {"camera": 0.48, "lidar": 0.68, "radar": 0.82, "ego": 0.97},
        "fog": {"camera": 0.34, "lidar": 0.55, "radar": 0.84, "ego": 0.97},
        "snow": {"camera": 0.38, "lidar": 0.48, "radar": 0.73, "ego": 0.95},
    }[weather].copy()


def _latent_sequence(
    data_seed: int,
    split: str,
    scene_index: int,
    length: int,
    latent_dim: int,
    training_replicate: int | None = None,
) -> tuple[np.ndarray, np.ndarray, str, str]:
    # Test and stress use the same latent namespace, producing a paired clean
    # and corrupted counterfactual for every shared scene index.
    namespace = "evaluation" if split in {"test", "stress"} else split
    replicate_namespace = training_replicate if split == "train" else "fixed"
    rng = np.random.default_rng(
        stable_seed(data_seed, "latent", namespace, replicate_namespace, scene_index)
    )
    scenario = SCENARIOS[scene_index % len(SCENARIOS)]
    density = ("low", "medium", "high")[(scene_index // len(SCENARIOS)) % 3]
    class_hint = scene_index % 4
    base_risk = (-1.05, -0.20, 0.48, 1.12)[class_hint]
    state = rng.normal(0.0, 0.35, size=latent_dim)
    state[0] = base_risk + rng.normal(0.0, 0.10)
    state[1] = {
        "cut_in": 0.15,
        "platoon_brake": 0.25,
        "occluded_crossing": 0.72,
        "right_turn": 0.88,
        "merge": 0.50,
    }[scenario] + rng.normal(0.0, 0.08)
    state[4] = {"low": -0.65, "medium": 0.0, "high": 0.65}[density]
    latent = np.zeros((length, latent_dim), dtype=float)
    labels = np.zeros(length, dtype=int)
    event_direction = rng.normal(0.0, 0.08, size=latent_dim)
    event_direction[0] += {"cut_in": 0.10, "platoon_brake": 0.16, "occluded_crossing": 0.08, "right_turn": 0.12, "merge": 0.09}[scenario]
    for t in range(length):
        innovation = rng.normal(0.0, 0.045, size=latent_dim)
        state = 0.94 * state + 0.06 * event_direction * t + innovation
        state[2] = np.sin((t + scene_index % 7) / 5.0) + rng.normal(0, 0.04)
        state[3] = 0.50 - 0.22 * state[0] + rng.normal(0, 0.05)
        latent[t] = state
        score = state[0] + 0.33 * state[1] + 0.10 * state[4] - 0.08 * state[3]
        labels[t] = int(np.digitize(score, [-0.48, 0.20, 0.83]))
    return latent, labels, scenario, density


def generate_study2_split(
    cfg: dict, split: str, training_replicate: int | None = None
) -> pd.DataFrame:
    study = cfg["study2"]
    modalities = tuple(cfg["common"]["modalities"])
    count_key = {
        "train": "train_sequences",
        "validation": "validation_sequences",
        "test": "test_sequences",
        "stress": "stress_sequences",
    }[split]
    n_sequences = int(study[count_key])
    length = int(study["sequence_length"])
    latent_dim = int(study["latent_dim"])
    raw_dim = int(study["raw_dim"])
    data_seed = int(study["data_seed"])
    transform_rng = np.random.default_rng(stable_seed(data_seed, "modality_transforms"))
    transforms = {m: _orthogonal_projection(transform_rng, latent_dim, raw_dim) for m in modalities}
    biases = {m: transform_rng.normal(0.0, 0.08, size=raw_dim) for m in modalities}
    rows: list[dict[str, object]] = []
    for scene_index in range(n_sequences):
        latent, labels, scenario, density = _latent_sequence(
            data_seed, split, scene_index, length, latent_dim, training_replicate
        )
        weather = _weather_for(split, scene_index)
        profile = _quality_profile(weather)
        density_penalty = {"low": 0.0, "medium": 0.035, "high": 0.075}[density]
        latent_pair_id = f"eval_{scene_index:05d}" if split in {"test", "stress"} else f"{split}_{scene_index:05d}"
        corruption_rng = np.random.default_rng(
            stable_seed(
                data_seed,
                "corruption",
                split,
                training_replicate if split == "train" else "fixed",
                scene_index,
                weather,
            )
        )
        modality_lag = {
            m: int(corruption_rng.choice([0, 0, 0, 1])) if m != "ego" else 0
            for m in modalities
        }
        burst_start = int(corruption_rng.integers(2, max(3, length - 2)))
        burst_modality = modalities[int(corruption_rng.integers(0, len(modalities) - 1))]
        previous_observation: dict[str, np.ndarray] = {}
        for t in range(length):
            row: dict[str, object] = {
                "split": split,
                "scene_id": f"{split}_{scene_index:05d}",
                "latent_pair_id": latent_pair_id,
                "frame": t,
                "weather": weather,
                "scenario": scenario,
                "density": density,
                "y_true": int(labels[t]),
            }
            for j in range(latent_dim):
                row[f"latent_{j}"] = float(latent[t, j])
            for modality in modalities:
                q = float(np.clip(profile[modality] - density_penalty + corruption_rng.normal(0, 0.035), 0.08, 0.995))
                source_t = max(0, t - modality_lag[modality])
                semantic = latent[source_t]
                noise_sd = 0.035 + 0.28 * (1.0 - q)
                observed = np.tanh(semantic @ transforms[modality] + biases[modality])
                observed += corruption_rng.normal(0.0, noise_sd, size=raw_dim)
                if split == "stress" and modality != "ego":
                    # Controlled injected sensor-domain perturbation. This is
                    # not claimed to be CARLA's natural weather physics.
                    observed += 0.055 * np.sin(np.arange(raw_dim) + scene_index % 5)
                available = 1
                dropout_probability = 0.01 + 0.10 * (1.0 - q)
                if corruption_rng.random() < dropout_probability:
                    available = 0
                if split == "stress" and modality == burst_modality and burst_start <= t < burst_start + 2:
                    available = 0
                if available == 0:
                    observed[:] = 0.0
                # Health descriptors use sensor observations only. Hidden DGM
                # corruption scale, lag, and q never enter these features.
                signal_energy = float(np.mean(np.abs(observed)))
                observed_spread = float(np.std(observed))
                roughness = float(np.mean(np.abs(np.diff(observed)))) if raw_dim > 1 else 0.0
                if modality in previous_observation:
                    temporal_change = float(
                        np.linalg.norm(observed - previous_observation[modality])
                        / np.sqrt(raw_dim)
                    )
                    temporal_stability = float(np.exp(-temporal_change))
                else:
                    temporal_stability = 1.0
                previous_observation[modality] = observed.copy()
                health = np.array(
                    [
                        signal_energy,
                        float(available),
                        observed_spread + 0.25 * roughness,
                        temporal_stability,
                    ]
                )
                row[f"quality_true_audit_{modality}"] = q
                row[f"available_{modality}"] = available
                for j, value in enumerate(health):
                    row[f"health_{modality}_{j}"] = float(value)
                for j, value in enumerate(observed):
                    row[f"{modality}_{j}"] = float(value)
            rows.append(row)
    frame = pd.DataFrame(rows)
    frame = frame.sort_values(["scene_id", "frame"], kind="stable").reset_index(drop=True)
    if set(frame["y_true"].unique()) != set(range(4)):
        raise RuntimeError(f"Study 2 {split} split lacks one or more action classes")
    return frame


class MultiviewAligner:
    def __init__(self, modalities: tuple[str, ...], raw_dim: int, embedding_dim: int, enabled: bool, seed: int):
        self.modalities = modalities
        self.raw_dim = raw_dim
        self.embedding_dim = embedding_dim
        self.enabled = enabled
        self.seed = seed
        self.scalers: dict[str, StandardScaler] = {}
        self.regressors: dict[str, Ridge] = {}
        self.pca: PCA | None = None

    def columns(self, modality: str) -> list[str]:
        return [f"{modality}_{j}" for j in range(self.raw_dim)]

    def fit(self, frame: pd.DataFrame) -> "MultiviewAligner":
        standardized = []
        for modality in self.modalities:
            scaler = StandardScaler().fit(frame[self.columns(modality)].to_numpy(float))
            self.scalers[modality] = scaler
            standardized.append(scaler.transform(frame[self.columns(modality)].to_numpy(float)))
        if self.enabled:
            concatenated = np.concatenate(standardized, axis=1)
            self.pca = PCA(n_components=self.embedding_dim, whiten=True, random_state=self.seed)
            anchor = self.pca.fit_transform(concatenated)
            for modality, values in zip(self.modalities, standardized):
                self.regressors[modality] = Ridge(alpha=1.0).fit(values, anchor)
        return self

    def transform(self, frame: pd.DataFrame) -> dict[str, np.ndarray]:
        outputs = {}
        for modality in self.modalities:
            values = self.scalers[modality].transform(frame[self.columns(modality)].to_numpy(float))
            if self.enabled:
                embedded = self.regressors[modality].predict(values)
            else:
                embedded = values[:, : self.embedding_dim]
                if embedded.shape[1] < self.embedding_dim:
                    embedded = np.pad(embedded, ((0, 0), (0, self.embedding_dim - embedded.shape[1])))
            outputs[modality] = embedded
        return outputs


class ReliabilityEstimator:
    def __init__(self, modalities: tuple[str, ...], temperature: float):
        self.modalities = modalities
        self.temperature = float(temperature)
        self.models: dict[str, Ridge] = {}

    @staticmethod
    def health_columns(modality: str) -> list[str]:
        return [f"health_{modality}_{j}" for j in range(4)]

    def fit(self, frame: pd.DataFrame, embeddings: dict[str, np.ndarray]) -> "ReliabilityEstimator":
        stack = np.stack([embeddings[m] for m in self.modalities], axis=1)
        anchor = normalize_rows(stack.mean(axis=1))
        for modality in self.modalities:
            target = np.sum(normalize_rows(embeddings[modality]) * anchor, axis=1)
            self.models[modality] = Ridge(alpha=3.0).fit(
                frame[self.health_columns(modality)].to_numpy(float), target
            )
        return self

    def scores(self, frame: pd.DataFrame) -> np.ndarray:
        return np.column_stack(
            [
                self.models[m].predict(frame[self.health_columns(m)].to_numpy(float))
                for m in self.modalities
            ]
        ) * self.temperature


class TemporalCompensator:
    def __init__(self, strength: float, embedding_dim: int):
        self.strength = float(strength)
        self.embedding_dim = embedding_dim
        self.delta_model = Ridge(alpha=2.0)

    def fit(self, frame: pd.DataFrame, fused: np.ndarray, ego_embedding: np.ndarray) -> "TemporalCompensator":
        inputs = []
        targets = []
        for _, indices in frame.groupby("scene_id", sort=False).indices.items():
            idx = np.asarray(indices)
            if len(idx) < 2:
                continue
            inputs.append(ego_embedding[idx[1:]] - ego_embedding[idx[:-1]])
            targets.append(fused[idx[1:]] - fused[idx[:-1]])
        self.delta_model.fit(np.concatenate(inputs), np.concatenate(targets))
        return self

    def apply(self, frame: pd.DataFrame, fused: np.ndarray, ego_embedding: np.ndarray, enabled: bool) -> tuple[np.ndarray, np.ndarray]:
        output = fused.copy()
        prediction_errors = np.full(len(frame), np.nan, dtype=float)
        for _, indices in frame.groupby("scene_id", sort=False).indices.items():
            idx = np.asarray(indices)
            for position in range(1, len(idx)):
                previous = idx[position - 1]
                current = idx[position]
                delta = self.delta_model.predict((ego_embedding[current] - ego_embedding[previous])[None, :])[0]
                warped_previous = output[previous] + delta
                # Held-out one-step prediction error is measured against the
                # current unsmoothed observation. Unlike ||output-warp||, this
                # quantity is not algebraically forced to favor smoothing.
                prediction_errors[current] = float(
                    np.linalg.norm(warped_previous - fused[current])
                )
                if enabled:
                    output[current] = (1.0 - self.strength) * fused[current] + self.strength * warped_previous
        return output, prediction_errors


@dataclass(frozen=True)
class Study2Condition:
    name: str
    alignment: bool = True
    reliability: bool = True
    temporal: bool = True
    dropout_distillation: bool = True


CONDITIONS = {
    "Full": Study2Condition("Full"),
    "Equal weighting": Study2Condition("Equal weighting", reliability=False),
    "No alignment": Study2Condition("No alignment", alignment=False),
    "No temporal consistency": Study2Condition("No temporal consistency", temporal=False),
    "No modality dropout-distillation": Study2Condition(
        "No modality dropout-distillation", dropout_distillation=False
    ),
}


class Study2Model:
    def __init__(self, cfg: dict, condition: Study2Condition, training_seed: int):
        study = cfg["study2"]
        self.cfg = cfg
        self.condition = condition
        self.training_seed = int(training_seed)
        self.modalities = tuple(cfg["common"]["modalities"])
        self.raw_dim = int(study["raw_dim"])
        self.embedding_dim = int(study["embedding_dim"])
        self.aligner = MultiviewAligner(
            self.modalities, self.raw_dim, self.embedding_dim, condition.alignment, training_seed
        )
        self.reliability = ReliabilityEstimator(
            self.modalities, float(study["reliability_temperature"])
        )
        self.temporal = TemporalCompensator(float(study["temporal_strength"]), self.embedding_dim)
        self.distiller: Ridge | None = None
        self.classifier = LogisticRegression(
            C=float(study["classifier_C"]),
            max_iter=int(study["classifier_max_iter"]),
            class_weight="balanced",
            random_state=training_seed,
            solver="lbfgs",
        )
        self.temperature = TemperatureScaler()

    def _available(self, frame: pd.DataFrame, missing_pattern: str = "none") -> np.ndarray:
        available = frame[[f"available_{m}" for m in self.modalities]].to_numpy(float)
        if missing_pattern != "none":
            for modality in missing_pattern.split("+"):
                available[:, self.modalities.index(modality)] = 0.0
        return available

    def _base_fusion(
        self, frame: pd.DataFrame, embeddings: dict[str, np.ndarray], missing_pattern: str
    ) -> tuple[np.ndarray, np.ndarray]:
        stack = np.stack([embeddings[m] for m in self.modalities], axis=1)
        available = self._available(frame, missing_pattern)
        if self.condition.reliability:
            scores = self.reliability.scores(frame)
        else:
            scores = np.zeros_like(available)
        scores = np.where(available > 0.5, scores, -1.0e9)
        all_missing = available.sum(axis=1) == 0
        if all_missing.any():
            scores[all_missing, self.modalities.index("ego")] = 0.0
        weights = softmax(scores, axis=1)
        fused = np.sum(weights[:, :, None] * stack, axis=1)
        return fused, weights

    def _represent(
        self, frame: pd.DataFrame, missing_pattern: str = "none", apply_distiller: bool = True
    ) -> tuple[np.ndarray, dict[str, np.ndarray], np.ndarray, np.ndarray]:
        embeddings = self.aligner.transform(frame)
        base, weights = self._base_fusion(frame, embeddings, missing_pattern)
        fused, drift = self.temporal.apply(
            frame, base, embeddings["ego"], enabled=self.condition.temporal
        )
        if (
            apply_distiller
            and self.condition.dropout_distillation
            and self.distiller is not None
            and missing_pattern != "none"
        ):
            fused = self.distiller.predict(fused)
        return fused, embeddings, weights, drift

    def fit(self, training: pd.DataFrame, validation: pd.DataFrame) -> "Study2Model":
        self.aligner.fit(training)
        train_embeddings = self.aligner.transform(training)
        self.reliability.fit(training, train_embeddings)
        train_base, _ = self._base_fusion(training, train_embeddings, "none")
        self.temporal.fit(training, train_base, train_embeddings["ego"])
        train_full, _, _, _ = self._represent(training, "none", apply_distiller=False)
        x_train = train_full
        y_train = training["y_true"].to_numpy(int)
        if self.condition.dropout_distillation:
            rng = np.random.default_rng(stable_seed(self.training_seed, "modality_dropout"))
            patterns = ("camera", "lidar", "radar", "ego", "camera+lidar")
            masked_blocks = []
            teacher_blocks = []
            label_blocks = []
            scene_ids = training["scene_id"].drop_duplicates().to_numpy()
            selected_scenes = set(
                rng.choice(
                    scene_ids,
                    size=max(1, int(len(scene_ids) * float(self.cfg["study2"]["modality_dropout_probability"]))),
                    replace=False,
                )
            )
            for pattern_index, pattern in enumerate(patterns):
                selected = training["scene_id"].isin(
                    [scene for i, scene in enumerate(sorted(selected_scenes)) if i % len(patterns) == pattern_index]
                )
                if not selected.any():
                    continue
                block = training.loc[selected].copy().reset_index(drop=True)
                masked, _, _, _ = self._represent(block, pattern, apply_distiller=False)
                teacher, _, _, _ = self._represent(block, "none", apply_distiller=False)
                masked_blocks.append(masked)
                teacher_blocks.append(teacher)
                label_blocks.append(block["y_true"].to_numpy(int))
            if masked_blocks:
                masked_all = np.concatenate(masked_blocks)
                teacher_all = np.concatenate(teacher_blocks)
                self.distiller = Ridge(alpha=4.0).fit(masked_all, teacher_all)
                distilled = self.distiller.predict(masked_all)
                x_train = np.concatenate([x_train, distilled])
                y_train = np.concatenate([y_train, np.concatenate(label_blocks)])
        self.classifier.fit(x_train, y_train)
        validation_features, _, _, _ = self._represent(validation, "none")
        validation_raw = self.classifier.predict_proba(validation_features)
        self.temperature.fit(validation_raw, validation["y_true"].to_numpy(int))
        return self

    def predict(
        self, frame: pd.DataFrame, missing_pattern: str = "none"
    ) -> tuple[np.ndarray, dict[str, np.ndarray], np.ndarray, np.ndarray, np.ndarray]:
        features, embeddings, weights, drift = self._represent(frame, missing_pattern)
        probabilities = self.temperature.transform(self.classifier.predict_proba(features))
        return probabilities, embeddings, weights, drift, features


def alignment_metrics(
    frame: pd.DataFrame,
    embeddings: dict[str, np.ndarray],
    max_sequences: int,
) -> dict[str, float]:
    final_rows = frame.groupby("scene_id", sort=False).tail(1).head(int(max_sequences))
    indices = final_rows.index.to_numpy()
    camera = normalize_rows(embeddings["camera"][indices])
    lidar = normalize_rows(embeddings["lidar"][indices])
    similarities = camera @ lidar.T
    positive = np.diag(similarities)
    negative = similarities[~np.eye(len(similarities), dtype=bool)]
    negative_mean = float(negative.mean())
    negative_sd = float(negative.std(ddof=1))
    sas = float(np.mean((positive - negative_mean) / max(negative_sd, 1.0e-8)))
    order = np.argsort(-similarities, axis=1)
    ranks = np.argmax(order == np.arange(len(order))[:, None], axis=1) + 1
    reverse_order = np.argsort(-similarities.T, axis=1)
    reverse_ranks = np.argmax(reverse_order == np.arange(len(order))[:, None], axis=1) + 1
    result = {
        "sas": sas,
        "positive_similarity": float(positive.mean()),
        "negative_similarity": negative_mean,
        "embedding_effective_rank_camera": float(np.exp(-np.sum(_spectrum_entropy(camera) * np.log(np.maximum(_spectrum_entropy(camera), 1e-12))))),
    }
    for k in (1, 5, 10):
        result[f"recall_at_{k}"] = float(0.5 * (np.mean(ranks <= k) + np.mean(reverse_ranks <= k)))
    return result


def _spectrum_entropy(values: np.ndarray) -> np.ndarray:
    singular = np.linalg.svd(values - values.mean(axis=0), compute_uv=False)
    proportions = singular / max(singular.sum(), 1.0e-12)
    return proportions


def classification_metrics(labels: np.ndarray, probabilities: np.ndarray, bins: int) -> dict[str, float]:
    prediction = probabilities.argmax(axis=1)
    return {
        "macro_f1": macro_f1(labels, prediction, probabilities.shape[1]),
        "nll": negative_log_likelihood(labels, probabilities),
        "brier": multiclass_brier(labels, probabilities),
        "ece": expected_calibration_error(labels, probabilities, bins),
    }


def _frame_digest(frame: pd.DataFrame) -> str:
    hashed = pd.util.hash_pandas_object(frame, index=True).to_numpy(dtype=np.uint64)
    columns = np.frombuffer("\x1f".join(frame.columns).encode("utf-8"), dtype=np.uint8)
    return array_digest(hashed, columns)


def _study2_checkpoint_arrays(model: Study2Model) -> list[np.ndarray]:
    arrays: list[np.ndarray] = [
        model.classifier.coef_,
        model.classifier.intercept_,
        model.classifier.classes_,
        np.array([model.temperature.temperature]),
        model.temporal.delta_model.coef_,
        np.atleast_1d(model.temporal.delta_model.intercept_),
        np.frombuffer(json.dumps(asdict(model.condition), sort_keys=True).encode("utf-8"), dtype=np.uint8),
    ]
    if model.aligner.pca is not None:
        arrays.extend(
            [
                model.aligner.pca.components_,
                model.aligner.pca.mean_,
                model.aligner.pca.explained_variance_,
            ]
        )
    for modality in model.modalities:
        scaler = model.aligner.scalers[modality]
        arrays.extend([scaler.mean_, scaler.scale_])
        if modality in model.aligner.regressors:
            regressor = model.aligner.regressors[modality]
            arrays.extend([regressor.coef_, np.atleast_1d(regressor.intercept_)])
        reliability = model.reliability.models[modality]
        arrays.extend([reliability.coef_, np.atleast_1d(reliability.intercept_)])
    if model.distiller is not None:
        arrays.extend([model.distiller.coef_, np.atleast_1d(model.distiller.intercept_)])
    return arrays


def _prediction_frame(
    source: pd.DataFrame,
    probabilities: np.ndarray,
    training_seed: int,
    method: str,
    domain: str,
    missing_pattern: str,
) -> pd.DataFrame:
    output = source[["scene_id", "frame", "y_true"]].copy()
    output.insert(0, "training_seed", int(training_seed))
    output.insert(1, "method", method)
    output.insert(2, "domain", domain)
    output.insert(3, "missing_pattern", missing_pattern)
    output["y_pred"] = probabilities.argmax(axis=1)
    for index, action in enumerate(ACTION_NAMES):
        output[f"p_{action}"] = probabilities[:, index]
    return output


def _scene_metric_rows(
    source: pd.DataFrame,
    probabilities: np.ndarray,
    drift: np.ndarray,
    training_seed: int,
    method: str,
    domain: str,
    bins: int,
) -> list[dict[str, object]]:
    rows: list[dict[str, object]] = []
    for scene_id, indices in source.groupby("scene_id", sort=False).indices.items():
        idx = np.asarray(indices)
        valid = drift[idx][np.isfinite(drift[idx])]
        rows.append(
            {
                "training_seed": training_seed,
                "method": method,
                "domain": domain,
                "scene_id": scene_id,
                **classification_metrics(source.iloc[idx]["y_true"].to_numpy(int), probabilities[idx], bins),
                "feature_drift": float(valid.mean()) if len(valid) else np.nan,
            }
        )
    return rows


def run_study2(cfg: dict) -> dict[str, pd.DataFrame]:
    fixed_splits = {
        split: generate_study2_split(cfg, split)
        for split in ("validation", "test", "stress")
    }
    methods = list(cfg["study2"]["methods"])
    seeds = list(cfg["common"]["training_seeds"])
    bins = int(cfg["common"]["calibration_bins"])
    result_rows = []
    missing_rows = []
    prediction_frames: list[pd.DataFrame] = []
    scene_rows: list[dict[str, object]] = []
    missing_scene_rows: list[dict[str, object]] = []
    calibration_rows = []
    latency_rows = []
    model_rows = []
    split_rows: list[dict[str, object]] = []
    for name, data in fixed_splits.items():
        split_rows.append(
            {
                "split": name,
                "training_seed": np.nan,
                "scenes": data["scene_id"].nunique(),
                "frames": len(data),
                "latent_pair_ids": data["latent_pair_id"].nunique(),
                "weather_levels": ",".join(sorted(data["weather"].unique())),
                "action_classes": ",".join(map(str, sorted(data["y_true"].unique()))),
                "data_sha256": _frame_digest(data),
            }
        )
    for training_seed in seeds:
        training = generate_study2_split(cfg, "train", int(training_seed))
        training_hash = _frame_digest(training)
        split_rows.append(
            {
                "split": "train",
                "training_seed": training_seed,
                "scenes": training["scene_id"].nunique(),
                "frames": len(training),
                "latent_pair_ids": training["latent_pair_id"].nunique(),
                "weather_levels": ",".join(sorted(training["weather"].unique())),
                "action_classes": ",".join(map(str, sorted(training["y_true"].unique()))),
                "data_sha256": training_hash,
            }
        )
        for method in methods:
            model = Study2Model(cfg, CONDITIONS[method], int(training_seed)).fit(
                training, fixed_splits["validation"]
            )
            digest_arrays = _study2_checkpoint_arrays(model)
            model_rows.append(
                {
                    "training_seed": training_seed,
                    "method": method,
                    "model_sha256": array_digest(*digest_arrays),
                    "training_data_sha256": training_hash,
                    "numeric_state_values": int(sum(array.size for array in digest_arrays)),
                    "hash_scope": "complete numeric fitted state + condition",
                    "temperature": float(model.temperature.temperature),
                    "alignment_enabled": model.condition.alignment,
                    "reliability_enabled": model.condition.reliability,
                    "temporal_enabled": model.condition.temporal,
                    "dropout_distillation_enabled": model.condition.dropout_distillation,
                }
            )
            for domain in ("test", "stress"):
                frame = fixed_splits[domain]
                domain_name = "clean_controlled" if domain == "test" else "adverse_controlled_injected"
                start = perf_counter_ns()
                probabilities, embeddings, weights, drift, features = model.predict(frame)
                elapsed_ms = (perf_counter_ns() - start) / 1.0e6
                metrics = classification_metrics(frame["y_true"].to_numpy(int), probabilities, bins)
                align = alignment_metrics(
                    frame.reset_index(drop=True),
                    embeddings,
                    int(cfg["study2"]["retrieval_pool_sequences"]),
                )
                valid_drift = drift[np.isfinite(drift)]
                metrics.update(align)
                metrics["feature_drift"] = float(valid_drift.mean())
                metrics["representation_variance"] = float(np.mean(np.var(features, axis=0)))
                metrics["latency_ms_per_frame"] = float(elapsed_ms / len(frame))
                result_rows.append(
                    {
                        "training_seed": training_seed,
                        "method": method,
                        "domain": domain_name,
                        "frames": len(frame),
                        "scenes": frame["scene_id"].nunique(),
                        **metrics,
                    }
                )
                if method == "Full" and domain == "test":
                    bins_frame = calibration_bins(frame["y_true"], probabilities, bins)
                    bins_frame.insert(0, "training_seed", training_seed)
                    bins_frame.insert(1, "method", method)
                    calibration_rows.append(bins_frame)
                prediction_frames.append(
                    _prediction_frame(
                        frame, probabilities, training_seed, method, domain_name, "none"
                    )
                )
                scene_rows.extend(
                    _scene_metric_rows(
                        frame, probabilities, drift, training_seed, method, domain_name, bins
                    )
                )
            clean_frame = fixed_splits["test"]
            clean_probabilities, *_ = model.predict(clean_frame, "none")
            clean_f1 = classification_metrics(clean_frame["y_true"].to_numpy(int), clean_probabilities, bins)["macro_f1"]
            clean_scene_f1: dict[str, float] = {}
            for scene_id, indices in clean_frame.groupby("scene_id", sort=False).indices.items():
                idx = np.asarray(indices)
                clean_scene_f1[str(scene_id)] = classification_metrics(
                    clean_frame.iloc[idx]["y_true"].to_numpy(int), clean_probabilities[idx], bins
                )["macro_f1"]
            for pattern in cfg["study2"]["missing_patterns"]:
                probabilities, _, _, _, _ = model.predict(clean_frame, pattern)
                metrics = classification_metrics(clean_frame["y_true"].to_numpy(int), probabilities, bins)
                missing_rows.append(
                    {
                        "training_seed": training_seed,
                        "method": method,
                        "missing_pattern": pattern,
                        **metrics,
                        "macro_f1_absolute_drop": float(clean_f1 - metrics["macro_f1"]),
                        "macro_f1_relative_drop": float((clean_f1 - metrics["macro_f1"]) / max(clean_f1, 1e-12)),
                    }
                )
                if pattern != "none":
                    prediction_frames.append(
                        _prediction_frame(
                            clean_frame,
                            probabilities,
                            training_seed,
                            method,
                            "clean_controlled",
                            pattern,
                        )
                    )
                for scene_id, indices in clean_frame.groupby("scene_id", sort=False).indices.items():
                    idx = np.asarray(indices)
                    scene_metrics = classification_metrics(
                        clean_frame.iloc[idx]["y_true"].to_numpy(int), probabilities[idx], bins
                    )
                    base_f1 = clean_scene_f1[str(scene_id)]
                    missing_scene_rows.append(
                        {
                            "training_seed": training_seed,
                            "method": method,
                            "scene_id": scene_id,
                            "missing_pattern": pattern,
                            **scene_metrics,
                            "macro_f1_absolute_drop": base_f1 - scene_metrics["macro_f1"],
                            "macro_f1_relative_drop": (base_f1 - scene_metrics["macro_f1"]) / max(base_f1, 1e-12),
                        }
                    )
            # Batch-1 end-to-end CPU timing includes representation and classifier.
            timing_frame = clean_frame.groupby("scene_id", sort=False).tail(1).head(120).reset_index(drop=True)
            for _ in range(3):
                model.predict(timing_frame.iloc[:8])
            samples = []
            for index in range(len(timing_frame)):
                start = perf_counter_ns()
                model.predict(timing_frame.iloc[index : index + 1])
                samples.append((perf_counter_ns() - start) / 1.0e6)
            latency_rows.append(
                {
                    "training_seed": training_seed,
                    "method": method,
                    "batch_size": 1,
                    "p50_ms": float(np.quantile(samples, 0.50)),
                    "p95_ms": float(np.quantile(samples, 0.95)),
                    "hardware": "CPU; exact platform in run_manifest.json",
                }
            )
    latent_columns = [f"latent_{j}" for j in range(int(cfg["study2"]["latent_dim"]))]
    paired = fixed_splits["test"].merge(
        fixed_splits["stress"], on=["latent_pair_id", "frame"], suffixes=("_clean", "_stress")
    )
    latent_difference = np.max(
        np.abs(
            paired[[f"{column}_clean" for column in latent_columns]].to_numpy(float)
            - paired[[f"{column}_stress" for column in latent_columns]].to_numpy(float)
        )
    )
    return {
        "study2_seed_metrics": pd.DataFrame(result_rows),
        "study2_missing_modality": pd.DataFrame(missing_rows),
        "study2_predictions": pd.concat(prediction_frames, ignore_index=True),
        "study2_scene_metrics": pd.DataFrame(scene_rows),
        "study2_missing_scene_metrics": pd.DataFrame(missing_scene_rows),
        "study2_calibration_bins": pd.concat(calibration_rows, ignore_index=True),
        "study2_latency": pd.DataFrame(latency_rows),
        "study2_split_manifest": pd.DataFrame(split_rows),
        "study2_pair_integrity": pd.DataFrame(
            [
                {
                    "paired_clean_stress_scenes": fixed_splits["stress"]["latent_pair_id"].nunique(),
                    "paired_frames": len(paired),
                    "max_absolute_latent_difference": float(latent_difference),
                    "pass_exact_latent_pairing": bool(latent_difference == 0.0),
                }
            ]
        ),
        "study2_model_manifest": pd.DataFrame(model_rows),
    }


### `src/egms_studies23/study3.py`

In [ ]:
%%writefile src/egms_studies23/study3.py
from __future__ import annotations

from dataclasses import asdict, dataclass
import json
from time import perf_counter_ns

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from .common import TemperatureScaler, array_digest, macro_f1, softmax, stable_seed


@dataclass
class TrajectoryDataset:
    split: str
    scene_ids: np.ndarray
    regime: np.ndarray
    intent: np.ndarray
    history: np.ndarray
    neighbors: np.ndarray
    neighbor_mask: np.ndarray
    map_features: np.ndarray
    future: np.ndarray
    branch_mode: np.ndarray

    def __len__(self) -> int:
        return len(self.scene_ids)


def _integrate_velocity(velocity: np.ndarray, dt: float) -> np.ndarray:
    return np.cumsum(velocity * dt, axis=0)


def _future_template(
    intent: int,
    mode: int,
    speed: float,
    future_steps: int,
    dt: float,
    interaction_strength: float,
    rng: np.random.Generator,
) -> np.ndarray:
    t = np.arange(1, future_steps + 1, dtype=float) * dt
    mode_scale = (0.82, 1.0, 1.18)[mode]
    effective_speed = speed * mode_scale
    x = effective_speed * t
    y = np.zeros_like(t)
    if intent == 0:  # straight
        x += 0.10 * np.sin(0.6 * t)
    elif intent == 1:  # slowing
        deceleration = 0.55 + 0.55 * mode
        x = effective_speed * t - 0.5 * deceleration * t**2
        x = np.maximum.accumulate(np.maximum(x, 0.0))
    elif intent == 2:  # stopping
        stop_time = np.clip(2.2 + 0.55 * mode - 0.45 * interaction_strength, 1.4, 4.2)
        deceleration = effective_speed / stop_time
        x = effective_speed * np.minimum(t, stop_time) - 0.5 * deceleration * np.minimum(t, stop_time) ** 2
        x = np.maximum.accumulate(np.maximum(x, 0.0))
    elif intent == 3:  # cut in
        lane = (2.8, 3.5, 4.1)[mode]
        y = lane / (1.0 + np.exp(-(t - (2.6 - 0.35 * interaction_strength)) * 2.0))
        y -= y[0]
        x *= 0.94
    elif intent == 4:  # lane change
        direction = -1.0 if mode == 0 else 1.0
        lane = (3.2, 3.5, 3.8)[mode]
        y = direction * lane / (1.0 + np.exp(-(t - 3.0) * 1.55))
        y -= y[0]
    elif intent == 5:  # turn left
        radius = (13.0, 16.0, 19.0)[mode]
        theta = np.minimum(effective_speed * t / radius, np.pi / 2)
        x = radius * np.sin(theta)
        y = radius * (1.0 - np.cos(theta))
    elif intent == 6:  # turn right
        radius = (11.0, 14.0, 17.0)[mode]
        theta = np.minimum(effective_speed * t / radius, np.pi / 2)
        x = radius * np.sin(theta)
        y = -radius * (1.0 - np.cos(theta))
    else:  # crossing
        crossing_speed = np.clip(2.2 + 0.55 * mode, 1.5, 4.0)
        x = 0.30 * effective_speed * t
        y = crossing_speed * t * (1.0 + 0.08 * interaction_strength)
    # Smooth correlated model discrepancy prevents a tiny set of prototypes
    # from trivially reproducing every synthetic future.
    innovations = rng.normal(0.0, 0.045 + 0.012 * mode, size=(future_steps, 2))
    discrepancy = np.cumsum(innovations, axis=0)
    return np.column_stack([x, y]) + discrepancy


def generate_study3_split(
    cfg: dict, split: str, training_replicate: int | None = None
) -> TrajectoryDataset:
    study = cfg["study3"]
    n = int(study[{"train": "train_scenes", "validation": "validation_scenes", "test": "test_scenes"}[split]])
    history_steps = int(study["history_steps"])
    future_steps = int(study["future_steps"])
    dt = float(study["sample_period_s"])
    max_neighbors = int(study["max_neighbors"])
    data_seed = int(study["data_seed"])
    history = np.zeros((n, history_steps, 4), dtype=float)
    neighbors = np.zeros((n, history_steps, max_neighbors, 4), dtype=float)
    neighbor_mask = np.zeros((n, max_neighbors), dtype=bool)
    map_features = np.zeros((n, 8), dtype=float)
    future = np.zeros((n, future_steps, 2), dtype=float)
    intent = np.zeros(n, dtype=int)
    branch_mode = np.zeros(n, dtype=int)
    regime = np.empty(n, dtype=object)
    scene_ids = np.array([f"{split}_{index:05d}" for index in range(n)], dtype=object)
    for index in range(n):
        rng = np.random.default_rng(
            stable_seed(
                data_seed,
                "study3",
                split,
                training_replicate if split == "train" else "fixed",
                index,
            )
        )
        intent_index = index % 8
        intent[index] = intent_index
        mode = int(rng.integers(0, 3))
        branch_mode[index] = mode
        regime_name = ("independent", "spatial", "history_dependent")[index % 3]
        regime[index] = regime_name
        speed = float(rng.uniform(6.0, 14.0))
        history_t = (np.arange(history_steps) - history_steps + 1) * dt
        acceleration = {
            0: 0.0,
            1: -0.08,
            2: -0.12,
            3: 0.05,
            4: 0.0,
            5: -0.08,
            6: -0.08,
            7: 0.0,
        }[intent_index]
        vx = np.clip(speed + acceleration * history_t + rng.normal(0, 0.06, history_steps), 0.3, None)
        vy = rng.normal(0, 0.025, history_steps)
        # Target-only cues are intentionally incomplete for interaction-driven
        # maneuvers; this lets graph ablations test their designated mechanism.
        if intent_index in {3, 4}:
            vy += np.linspace(0.0, 0.12 if intent_index == 3 else 0.08 * (-1 if mode == 0 else 1), history_steps)
        if intent_index in {5, 6}:
            vy += np.linspace(0.0, 0.16 * (1 if intent_index == 5 else -1), history_steps)
        target_velocity = np.column_stack([vx, vy])
        target_position = _integrate_velocity(target_velocity, dt)
        target_position -= target_position[-1]
        history[index, :, :2] = target_position
        history[index, :, 2:] = target_velocity

        neighbor_count = int(rng.integers(2, max_neighbors + 1))
        neighbor_mask[index, :neighbor_count] = True
        strongest_interaction = 0.0
        for neighbor in range(neighbor_count):
            rel_x0 = float(rng.uniform(8.0, 45.0) * (-1 if rng.random() < 0.18 else 1))
            rel_y0 = float(rng.choice([-7.0, -3.5, 0.0, 3.5, 7.0]) + rng.normal(0, 0.35))
            rel_vx = float(rng.normal(-0.5, 1.7))
            rel_vy = float(rng.normal(0.0, 0.25))
            if regime_name == "independent":
                rel_x0 = float(rng.uniform(38.0, 60.0))
                rel_y0 = float(rng.choice([-10.5, -7.0, 7.0, 10.5]) + rng.normal(0, 0.3))
                rel_vx = float(rng.normal(0.0, 0.35))
                rel_vy = float(rng.normal(0.0, 0.12))
            if regime_name == "history_dependent" and neighbor > 0:
                rel_x0 = float(rng.uniform(30.0, 48.0))
                rel_y0 = float(rng.choice([-7.0, 7.0]) + rng.normal(0, 0.35))
            history_curve_x = np.zeros(history_steps)
            history_curve_y = np.zeros(history_steps)
            history_curve_vx = np.zeros(history_steps)
            history_curve_vy = np.zeros(history_steps)
            if regime_name == "history_dependent" and neighbor == 0:
                # All ambiguous maneuvers converge to similar current spatial
                # relations. Only the observed relation history identifies the
                # approach pattern; no future value is used.
                rel_x0 = 13.5 + rng.normal(0, 0.25)
                rel_y0 = 3.5 + rng.normal(0, 0.18)
                rel_vx = -0.45 + rng.normal(0, 0.08)
                rel_vy = rng.normal(0, 0.04)
                amplitude_x = {0: 0.0, 1: 4.0, 2: 7.0, 3: 2.0, 4: -2.0, 5: 1.0, 6: -1.0, 7: 5.0}[intent_index]
                amplitude_y = {0: 0.0, 1: 0.5, 2: 0.8, 3: 5.0, 4: -5.0, 5: 2.5, 6: -2.5, 7: 7.0}[intent_index]
                horizon = abs(history_t[0]) + dt
                phase = history_t / horizon
                history_curve_x = amplitude_x * phase**2
                history_curve_y = amplitude_y * phase**2
                history_curve_vx = 2.0 * amplitude_x * history_t / (horizon**2)
                history_curve_vy = 2.0 * amplitude_y * history_t / (horizon**2)
            elif regime_name == "spatial" and neighbor == 0:
                rel_x0 = 10.0 + 5.0 * (intent_index not in {1, 2, 3, 7}) + rng.normal(0, 1.1)
                rel_y0 = {3: 3.5, 4: -3.5 if mode == 0 else 3.5, 7: -8.0}.get(intent_index, rel_y0)
            rel_position = np.column_stack(
                [
                    rel_x0 + rel_vx * history_t + history_curve_x,
                    rel_y0 + rel_vy * history_t + history_curve_y,
                ]
            )
            absolute_position = target_position + rel_position
            absolute_velocity = target_velocity + np.column_stack(
                [rel_vx + history_curve_vx, rel_vy + history_curve_vy]
            )
            neighbors[index, :, neighbor, :2] = absolute_position
            neighbors[index, :, neighbor, 2:] = absolute_velocity
            closing = max(-rel_vx, 0.0)
            ttc = rel_x0 / closing if rel_x0 > 0 and closing > 0.1 else 20.0
            interaction = float(np.exp(-abs(rel_y0) / 4.0) * np.exp(-min(ttc, 20.0) / 7.0))
            strongest_interaction = max(strongest_interaction, interaction)

        intersection_probability = 0.72 if intent_index in {5, 6, 7} else 0.32
        is_intersection = float(rng.random() < intersection_probability)
        map_features[index] = np.array(
            [
                is_intersection,
                float(rng.random() < (0.68 if intent_index == 7 else 0.28)),
                float(rng.random() < (0.58 if intent_index in {3, 4} else 0.30)),
                float(rng.random() < (0.72 if intent_index == 5 else 0.34)),
                float(rng.random() < (0.72 if intent_index == 6 else 0.34)),
                float(rng.integers(1, 4)) / 3.0,
                float(rng.uniform(-0.45, 0.45)),
                float(rng.uniform(0.0, 1.0)),
            ]
        )
        interaction_strength = strongest_interaction if regime_name != "independent" else 0.0
        future[index] = _future_template(
            intent_index,
            mode,
            speed,
            future_steps,
            dt,
            interaction_strength,
            rng,
        )
    return TrajectoryDataset(
        split=split,
        scene_ids=scene_ids,
        regime=regime,
        intent=intent,
        history=history,
        neighbors=neighbors,
        neighbor_mask=neighbor_mask,
        map_features=map_features,
        future=future,
        branch_mode=branch_mode,
    )


def _linear_slope(values: np.ndarray) -> np.ndarray:
    time = np.linspace(-1.0, 0.0, values.shape[1])
    centered = time - time.mean()
    denominator = np.sum(centered**2)
    return np.sum((values - values.mean(axis=1, keepdims=True)) * centered[None, :, None], axis=1) / denominator


def graph_features(dataset: TrajectoryDataset, graph_level: str) -> np.ndarray:
    history = dataset.history
    velocity = history[:, :, 2:]
    acceleration = np.diff(velocity, axis=1) / 0.1
    speed = np.linalg.norm(velocity, axis=2)
    displacement = history[:, -1, :2] - history[:, 0, :2]
    base = np.column_stack(
        [
            velocity[:, -1, :],
            acceleration[:, -1, :],
            speed[:, -1],
            speed.mean(axis=1),
            speed.std(axis=1),
            displacement,
            _linear_slope(velocity),
            dataset.map_features,
        ]
    )
    # Current relation-aware permutation-invariant graph aggregation.
    current_target = history[:, -1, :]
    current_neighbors = dataset.neighbors[:, -1]
    rel_position = current_neighbors[:, :, :2] - current_target[:, None, :2]
    rel_velocity = current_neighbors[:, :, 2:] - current_target[:, None, 2:]
    distance = np.linalg.norm(rel_position, axis=2)
    closing = -np.sum(rel_position * rel_velocity, axis=2) / np.maximum(distance, 1.0e-6)
    ttc = np.full_like(distance, 20.0)
    np.divide(distance, closing, out=ttc, where=closing > 0.1)
    mask = dataset.neighbor_mask
    edge_mask = mask & ((distance < 25.0) | (ttc < 5.0))
    attention_logit = -distance / 18.0 - np.minimum(ttc, 20.0) / 9.0
    unnormalized = np.where(edge_mask, np.exp(attention_logit), 0.0)
    attention = unnormalized / np.maximum(unnormalized.sum(axis=1, keepdims=True), 1.0e-12)
    current_messages = np.concatenate(
        [
            rel_position,
            rel_velocity,
            distance[:, :, None],
            np.minimum(ttc, 20.0)[:, :, None],
        ],
        axis=2,
    )
    weighted = np.sum(attention[:, :, None] * current_messages, axis=1)
    min_distance = np.min(np.where(edge_mask, distance, np.inf), axis=1)
    min_ttc = np.min(np.where(edge_mask, ttc, np.inf), axis=1)
    min_distance = np.where(np.isfinite(min_distance), min_distance, 0.0)
    min_ttc = np.where(np.isfinite(min_ttc), np.minimum(min_ttc, 20.0), 0.0)
    spatial = np.column_stack(
        [weighted, min_distance, min_ttc, edge_mask.sum(axis=1)]
    )

    # Temporal graph summaries use only the observed history.
    target_position = history[:, :, None, :2]
    target_velocity = history[:, :, None, 2:]
    rel_pos_hist = dataset.neighbors[:, :, :, :2] - target_position
    rel_vel_hist = dataset.neighbors[:, :, :, 2:] - target_velocity
    distance_hist = np.linalg.norm(rel_pos_hist, axis=3)
    closing_hist = -np.sum(rel_pos_hist * rel_vel_hist, axis=3) / np.maximum(distance_hist, 1.0e-6)
    ttc_hist = np.full_like(distance_hist, 20.0)
    np.divide(distance_hist, closing_hist, out=ttc_hist, where=closing_hist > 0.1)
    nearest_distance = np.min(
        np.where(edge_mask[:, None, :], distance_hist, np.inf), axis=2
    )
    nearest_ttc = np.min(
        np.where(edge_mask[:, None, :], ttc_hist, np.inf), axis=2
    )
    nearest_distance = np.where(np.isfinite(nearest_distance), nearest_distance, 0.0)
    nearest_ttc = np.where(np.isfinite(nearest_ttc), nearest_ttc, 0.0)
    edge_count = np.maximum(edge_mask.sum(axis=1), 1)[:, None]
    mean_rel_x = np.sum(
        np.where(edge_mask[:, None, :], rel_pos_hist[:, :, :, 0], 0.0), axis=2
    ) / edge_count
    mean_rel_y = np.sum(
        np.where(edge_mask[:, None, :], rel_pos_hist[:, :, :, 1], 0.0), axis=2
    ) / edge_count
    attended_rel_x = np.sum(attention[:, None, :] * rel_pos_hist[:, :, :, 0], axis=2)
    attended_rel_y = np.sum(attention[:, None, :] * rel_pos_hist[:, :, :, 1], axis=2)
    temporal = np.column_stack(
        [
            nearest_distance[:, -1] - nearest_distance[:, 0],
            nearest_ttc[:, -1] - nearest_ttc[:, 0],
            np.min(nearest_distance, axis=1),
            np.min(nearest_ttc, axis=1),
            np.std(nearest_distance, axis=1),
            np.std(nearest_ttc, axis=1),
            _linear_slope(nearest_distance[:, :, None])[:, 0],
            _linear_slope(nearest_ttc[:, :, None])[:, 0],
            mean_rel_x[:, -1] - mean_rel_x[:, 0],
            mean_rel_y[:, -1] - mean_rel_y[:, 0],
            _linear_slope(mean_rel_x[:, :, None])[:, 0],
            _linear_slope(mean_rel_y[:, :, None])[:, 0],
            attended_rel_x[:, -1] - attended_rel_x[:, 0],
            attended_rel_y[:, -1] - attended_rel_y[:, 0],
            _linear_slope(attended_rel_x[:, :, None])[:, 0],
            _linear_slope(attended_rel_y[:, :, None])[:, 0],
        ]
    )
    full = np.concatenate([base, spatial, temporal], axis=1)
    if graph_level == "full":
        return full
    if graph_level == "spatial":
        return np.concatenate([base, spatial, np.zeros_like(temporal)], axis=1)
    if graph_level == "none":
        return np.concatenate([base, np.zeros_like(spatial), np.zeros_like(temporal)], axis=1)
    raise ValueError(f"Unknown graph level: {graph_level}")


@dataclass(frozen=True)
class Study3Condition:
    name: str
    graph_level: str = "full"
    predicted_intent_gating: bool = True
    modes: int = 6


CONDITIONS = {
    "Full temporal graph": Study3Condition("Full temporal graph"),
    "No graph": Study3Condition("No graph", graph_level="none"),
    "Spatial-only graph": Study3Condition("Spatial-only graph", graph_level="spatial"),
    "No predicted-intent gating": Study3Condition(
        "No predicted-intent gating", graph_level="full", predicted_intent_gating=False
    ),
    "Single mode K=1": Study3Condition("Single mode K=1", graph_level="full", modes=1),
}


class Study3Model:
    def __init__(self, cfg: dict, condition: Study3Condition, training_seed: int):
        self.cfg = cfg
        self.condition = condition
        self.training_seed = int(training_seed)
        study = cfg["study3"]
        self.intent_scaler = StandardScaler()
        self.intent_classifier = LogisticRegression(
            C=float(study["classifier_C"]),
            max_iter=int(study["classifier_max_iter"]),
            class_weight="balanced",
            random_state=training_seed,
            solver="lbfgs",
        )
        self.intent_temperature = TemperatureScaler()
        self.intent_prototypes: dict[int, np.ndarray] = {}
        self.intent_priors: dict[int, np.ndarray] = {}
        self.training_intent_prior = np.full(
            len(self.cfg["study3"]["intent_classes"]),
            1.0 / len(self.cfg["study3"]["intent_classes"]),
        )

    @staticmethod
    def _speed_scale(dataset: TrajectoryDataset) -> np.ndarray:
        return np.maximum(np.linalg.norm(dataset.history[:, -1, 2:], axis=1), 2.0)

    def fit(self, training: TrajectoryDataset, validation: TrajectoryDataset) -> "Study3Model":
        x_train = graph_features(training, self.condition.graph_level)
        x_validation = graph_features(validation, self.condition.graph_level)
        self.intent_scaler.fit(x_train)
        self.intent_classifier.fit(self.intent_scaler.transform(x_train), training.intent)
        validation_raw = self.intent_classifier.predict_proba(self.intent_scaler.transform(x_validation))
        self.intent_temperature.fit(validation_raw, validation.intent)
        normalized_future = training.future / self._speed_scale(training)[:, None, None]
        downsample = normalized_future[:, 4::5, :].reshape(len(training), -1)
        prototypes_per_intent = int(self.cfg["study3"]["prototypes_per_intent"])
        counts = np.bincount(
            training.intent,
            minlength=len(self.cfg["study3"]["intent_classes"]),
        ).astype(float)
        self.training_intent_prior = counts / counts.sum()
        cluster_count = 1 if self.condition.modes == 1 else prototypes_per_intent
        for intent_index in range(len(self.cfg["study3"]["intent_classes"])):
            mask = training.intent == intent_index
            kmeans = KMeans(
                n_clusters=cluster_count,
                random_state=stable_seed(self.training_seed, "intent_cluster", intent_index),
                n_init=10,
            ).fit(downsample[mask])
            labels = kmeans.labels_
            prototypes = []
            priors = []
            for cluster in range(cluster_count):
                members = normalized_future[mask][labels == cluster]
                prototypes.append(members.mean(axis=0))
                priors.append(len(members))
            self.intent_prototypes[intent_index] = np.stack(prototypes)
            self.intent_priors[intent_index] = np.asarray(priors, dtype=float) / np.sum(priors)
        return self

    def intent_probabilities(self, dataset: TrajectoryDataset) -> np.ndarray:
        features = graph_features(dataset, self.condition.graph_level)
        raw = self.intent_classifier.predict_proba(self.intent_scaler.transform(features))
        return self.intent_temperature.transform(raw)

    def predict(self, dataset: TrajectoryDataset) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        intent_probabilities = self.intent_probabilities(dataset)
        n = len(dataset)
        output_k = int(self.condition.modes)
        future_steps = int(self.cfg["study3"]["future_steps"])
        candidates = np.zeros((n, output_k, future_steps, 2), dtype=float)
        probabilities = np.zeros((n, output_k), dtype=float)
        scales = self._speed_scale(dataset)
        gating = (
            intent_probabilities
            if self.condition.predicted_intent_gating
            else np.broadcast_to(self.training_intent_prior, intent_probabilities.shape)
        )
        for row in range(n):
            options = []
            for intent_index, prototypes in self.intent_prototypes.items():
                for mode_index, prototype in enumerate(prototypes):
                    probability = gating[row, intent_index] * self.intent_priors[intent_index][mode_index]
                    options.append((float(probability), intent_index, mode_index, prototype))
            options.sort(key=lambda item: (-item[0], item[1], item[2]))
            selected = options[:output_k]
            probabilities[row] = np.array([item[0] for item in selected])
            probabilities[row] /= max(probabilities[row].sum(), 1.0e-12)
            candidates[row] = np.stack([item[3] * scales[row] for item in selected])
        return candidates, probabilities, intent_probabilities


def trajectory_scene_metrics(
    ground_truth: np.ndarray,
    candidates: np.ndarray,
    probabilities: np.ndarray,
    miss_threshold: float,
) -> dict[str, np.ndarray]:
    errors = np.linalg.norm(candidates - ground_truth[:, None, :, :], axis=3)
    ade_by_mode = errors.mean(axis=2)
    fde_by_mode = errors[:, :, -1]
    best_fde_mode = fde_by_mode.argmin(axis=1)
    row = np.arange(len(ground_truth))
    # Official AV2 aggregation selects k* by minimum endpoint error and then
    # uses that same mode for minADE and Brier-minFDE.
    min_ade = ade_by_mode[row, best_fde_mode]
    min_fde = fde_by_mode[row, best_fde_mode]
    miss = (min_fde > miss_threshold).astype(float)
    brier_min_fde = min_fde + (1.0 - probabilities[row, best_fde_mode]) ** 2
    top = probabilities.argmax(axis=1)
    top1_ade = ade_by_mode[np.arange(len(ground_truth)), top]
    top1_fde = fde_by_mode[np.arange(len(ground_truth)), top]
    top1_miss = (top1_fde > miss_threshold).astype(float)
    return {
        "minADE_K": min_ade,
        "minFDE_K": min_fde,
        "MR_K": miss,
        "Brier-minFDE_K": brier_min_fde,
        "ADE_1": top1_ade,
        "FDE_1": top1_fde,
        "MR_1": top1_miss,
    }


def trajectory_mode_errors(
    ground_truth: np.ndarray, candidates: np.ndarray
) -> tuple[np.ndarray, np.ndarray]:
    errors = np.linalg.norm(candidates - ground_truth[:, None, :, :], axis=3)
    return errors.mean(axis=2), errors[:, :, -1]


def _trajectory_dataset_digest(dataset: TrajectoryDataset) -> str:
    text = "\x1f".join(
        [
            dataset.split,
            *map(str, dataset.scene_ids),
            *map(str, dataset.regime),
        ]
    )
    return array_digest(
        np.frombuffer(text.encode("utf-8"), dtype=np.uint8),
        dataset.intent,
        dataset.history,
        dataset.neighbors,
        dataset.neighbor_mask.astype(np.uint8),
        dataset.map_features,
        dataset.future,
        dataset.branch_mode,
    )


def _study3_parameter_arrays(model: Study3Model) -> list[np.ndarray]:
    arrays: list[np.ndarray] = [
        model.intent_scaler.mean_,
        model.intent_scaler.scale_,
        model.intent_classifier.coef_,
        model.intent_classifier.intercept_,
        model.intent_classifier.classes_,
        np.array([model.intent_temperature.temperature]),
        model.training_intent_prior,
    ]
    for intent_index in sorted(model.intent_prototypes):
        arrays.extend(
            [model.intent_prototypes[intent_index], model.intent_priors[intent_index]]
        )
    return arrays


def run_study3(cfg: dict) -> dict[str, pd.DataFrame | np.ndarray]:
    fixed_splits = {
        split: generate_study3_split(cfg, split)
        for split in ("validation", "test")
    }
    methods = list(cfg["study3"]["methods"])
    seeds = list(cfg["common"]["training_seeds"])
    prediction_rows = []
    seed_rows = []
    latency_rows = []
    model_rows = []
    split_rows = []
    example_payload = None
    for name, data in fixed_splits.items():
        split_rows.append(
            {
                "split": name,
                "training_seed": np.nan,
                "scenes": len(data),
                "intent_classes": len(np.unique(data.intent)),
                "regimes": ",".join(sorted(np.unique(data.regime))),
                "history_steps": data.history.shape[1],
                "future_steps": data.future.shape[1],
                "agent_history_future_overlap": 0,
                "data_sha256": _trajectory_dataset_digest(data),
            }
        )
    for training_seed in seeds:
        training = generate_study3_split(cfg, "train", int(training_seed))
        training_hash = _trajectory_dataset_digest(training)
        split_rows.append(
            {
                "split": "train",
                "training_seed": training_seed,
                "scenes": len(training),
                "intent_classes": len(np.unique(training.intent)),
                "regimes": ",".join(sorted(np.unique(training.regime))),
                "history_steps": training.history.shape[1],
                "future_steps": training.future.shape[1],
                "agent_history_future_overlap": 0,
                "data_sha256": training_hash,
            }
        )
        for method in methods:
            model = Study3Model(cfg, CONDITIONS[method], int(training_seed)).fit(
                training, fixed_splits["validation"]
            )
            parameter_arrays = _study3_parameter_arrays(model)
            inference_array = np.frombuffer(
                json.dumps(asdict(model.condition), sort_keys=True).encode("utf-8"),
                dtype=np.uint8,
            )
            model_rows.append(
                {
                    "training_seed": training_seed,
                    "method": method,
                    "model_sha256": array_digest(*parameter_arrays, inference_array),
                    "parameter_sha256": array_digest(*parameter_arrays),
                    "inference_rule_sha256": array_digest(inference_array),
                    "training_data_sha256": training_hash,
                    "numeric_state_values": int(sum(array.size for array in parameter_arrays)),
                    "hash_scope": "complete numeric fitted state + inference condition",
                    "temperature": float(model.intent_temperature.temperature),
                    "graph_level": model.condition.graph_level,
                    "predicted_intent_gating": model.condition.predicted_intent_gating,
                    "K": model.condition.modes,
                }
            )
            candidates, probabilities, intent_probabilities = model.predict(fixed_splits["test"])
            metrics = trajectory_scene_metrics(
                fixed_splits["test"].future,
                candidates,
                probabilities,
                float(cfg["study3"]["miss_threshold_m"]),
            )
            ade_by_mode, fde_by_mode = trajectory_mode_errors(
                fixed_splits["test"].future, candidates
            )
            intent_prediction = intent_probabilities.argmax(axis=1)
            seed_row = {
                "training_seed": training_seed,
                "method": method,
                "K": int(CONDITIONS[method].modes),
                "intent_macro_f1": macro_f1(fixed_splits["test"].intent, intent_prediction, 8),
                **{name: float(values.mean()) for name, values in metrics.items()},
            }
            if int(CONDITIONS[method].modes) == 1:
                # Never mislabel K=1 as K=6 in manuscript tables.
                seed_row["minADE_6_reportable"] = np.nan
                seed_row["minFDE_6_reportable"] = np.nan
                seed_row["MR_6_reportable"] = np.nan
                seed_row["Brier-minFDE_6_reportable"] = np.nan
            else:
                seed_row["minADE_6_reportable"] = seed_row["minADE_K"]
                seed_row["minFDE_6_reportable"] = seed_row["minFDE_K"]
                seed_row["MR_6_reportable"] = seed_row["MR_K"]
                seed_row["Brier-minFDE_6_reportable"] = seed_row["Brier-minFDE_K"]
            seed_rows.append(seed_row)
            max_modes = int(cfg["study3"]["modes"])
            for row in range(len(fixed_splits["test"])):
                record = {
                        "training_seed": training_seed,
                        "method": method,
                        "scene_id": fixed_splits["test"].scene_ids[row],
                        "regime": fixed_splits["test"].regime[row],
                        "K": int(CONDITIONS[method].modes),
                        "intent_true": int(fixed_splits["test"].intent[row]),
                        "intent_pred": int(intent_prediction[row]),
                        **{name: float(values[row]) for name, values in metrics.items()},
                    }
                for intent_index in range(intent_probabilities.shape[1]):
                    record[f"intent_p_{intent_index}"] = float(intent_probabilities[row, intent_index])
                for mode_index in range(max_modes):
                    active = mode_index < probabilities.shape[1]
                    record[f"p_mode_{mode_index + 1}"] = float(probabilities[row, mode_index]) if active else np.nan
                    record[f"ade_mode_{mode_index + 1}"] = float(ade_by_mode[row, mode_index]) if active else np.nan
                    record[f"fde_mode_{mode_index + 1}"] = float(fde_by_mode[row, mode_index]) if active else np.nan
                record["best_fde_mode"] = int(np.argmin(fde_by_mode[row]) + 1)
                prediction_rows.append(record)
            timing_indices = np.arange(min(120, len(fixed_splits["test"])))
            samples = []
            for index in timing_indices:
                single = subset_dataset(fixed_splits["test"], np.array([index]))
                start = perf_counter_ns()
                model.predict(single)
                samples.append((perf_counter_ns() - start) / 1.0e6)
            latency_rows.append(
                {
                    "training_seed": training_seed,
                    "method": method,
                    "batch_size": 1,
                    "p50_ms": float(np.quantile(samples, 0.50)),
                    "p95_ms": float(np.quantile(samples, 0.95)),
                    "includes": "feature extraction + graph aggregation + intent + trajectory decoding",
                }
            )
            if training_seed == seeds[0] and method == "Full temporal graph":
                example_indices = np.array([3, 19, 41, 77])
                example_payload = {
                    "scene_ids": fixed_splits["test"].scene_ids[example_indices],
                    "history": fixed_splits["test"].history[example_indices, :, :2],
                    "future": fixed_splits["test"].future[example_indices],
                    "candidates": candidates[example_indices],
                    "probabilities": probabilities[example_indices],
                    "intent": fixed_splits["test"].intent[example_indices],
                }
    assert example_payload is not None
    return {
        "study3_seed_metrics": pd.DataFrame(seed_rows),
        "study3_predictions": pd.DataFrame(prediction_rows),
        "study3_latency": pd.DataFrame(latency_rows),
        "study3_split_manifest": pd.DataFrame(split_rows),
        "study3_model_manifest": pd.DataFrame(model_rows),
        "study3_examples": example_payload,
    }


def subset_dataset(dataset: TrajectoryDataset, indices: np.ndarray) -> TrajectoryDataset:
    return TrajectoryDataset(
        split=dataset.split,
        scene_ids=dataset.scene_ids[indices],
        regime=dataset.regime[indices],
        intent=dataset.intent[indices],
        history=dataset.history[indices],
        neighbors=dataset.neighbors[indices],
        neighbor_mask=dataset.neighbor_mask[indices],
        map_features=dataset.map_features[indices],
        future=dataset.future[indices],
        branch_mode=dataset.branch_mode[indices],
    )


### `src/egms_studies23/validation.py`

In [ ]:
%%writefile src/egms_studies23/validation.py
from __future__ import annotations

import json
import zlib
from pathlib import Path
from typing import Any
from xml.etree import ElementTree

import numpy as np
import pandas as pd
import yaml

from .common import (
    expected_calibration_error,
    macro_f1,
    multiclass_brier,
    negative_log_likelihood,
    sha256,
    write_json,
)


EXPECTED_EVIDENCE = "CONTROLLED_SYNTHETIC_MECHANISM_VALIDATION_NOT_EMPIRICAL_DATASET"


def _valid_png(path: Path) -> bool:
    data = path.read_bytes()
    if not data.startswith(b"\x89PNG\r\n\x1a\n"):
        return False
    position = 8
    while position + 12 <= len(data):
        length = int.from_bytes(data[position : position + 4], "big")
        chunk_type = data[position + 4 : position + 8]
        payload_end = position + 8 + length
        chunk_end = payload_end + 4
        if chunk_end > len(data):
            return False
        expected = int.from_bytes(data[payload_end:chunk_end], "big")
        observed = zlib.crc32(chunk_type + data[position + 8 : payload_end])
        if expected != observed:
            return False
        position = chunk_end
        if chunk_type == b"IEND":
            return position == len(data)
    return False


def _append_if(errors: list[str], condition: bool, message: str) -> None:
    if condition:
        errors.append(message)


def _allclose(frame: pd.DataFrame, columns: list[str], expected: np.ndarray, atol: float = 1e-9) -> bool:
    return np.allclose(frame[columns].to_numpy(float), expected, atol=atol, rtol=0.0, equal_nan=True)


def _validate_study2(output: Path, cfg: dict, errors: list[str]) -> dict[str, int]:
    seeds = list(cfg["common"]["training_seeds"])
    methods = list(cfg["study2"]["methods"])
    patterns = list(cfg["study2"]["missing_patterns"])
    sequence_length = int(cfg["study2"]["sequence_length"])
    test_scenes = int(cfg["study2"]["test_sequences"])
    stress_scenes = int(cfg["study2"]["stress_sequences"])
    bins = int(cfg["common"]["calibration_bins"])

    seed_metrics = pd.read_csv(output / "data" / "study2_seed_metrics.csv")
    _append_if(errors, len(seed_metrics) != len(seeds) * len(methods) * 2, "Study 2 seed-metric grid is incomplete")
    _append_if(errors, set(seed_metrics["training_seed"]) != set(seeds), "Study 2 training seeds differ from config")
    _append_if(errors, set(seed_metrics["method"]) != set(methods), "Study 2 methods differ from config")
    for column in ("macro_f1", "ece", "recall_at_1", "recall_at_5", "recall_at_10"):
        _append_if(errors, not seed_metrics[column].between(0, 1).all(), f"Study 2 {column} outside [0,1]")
    _append_if(
        errors,
        not np.isfinite(seed_metrics[["nll", "brier", "feature_drift"]].to_numpy(float)).all()
        or (seed_metrics[["nll", "brier", "feature_drift"]].to_numpy(float) < 0).any(),
        "Study 2 loss/prediction error is non-finite or negative",
    )

    predictions = pd.read_csv(output / "data" / "study2_predictions.csv.gz")
    key = ["training_seed", "method", "domain", "missing_pattern", "scene_id", "frame"]
    _append_if(errors, predictions.duplicated(key).any(), "Study 2 prediction keys are duplicated")
    expected_rows = len(seeds) * len(methods) * sequence_length * (
        test_scenes * len(patterns) + stress_scenes
    )
    _append_if(errors, len(predictions) != expected_rows, f"Study 2 predictions have {len(predictions)} rows, expected {expected_rows}")
    p_cols = ["p_KEEP", "p_SLOW", "p_YIELD", "p_STOP"]
    probabilities = predictions[p_cols].to_numpy(float)
    _append_if(errors, not np.isfinite(probabilities).all(), "Study 2 probabilities contain non-finite values")
    _append_if(errors, (probabilities < 0).any() or (probabilities > 1).any(), "Study 2 probability outside [0,1]")
    _append_if(errors, not np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-8), "Study 2 probability rows do not sum to one")
    _append_if(errors, not np.array_equal(predictions["y_pred"].to_numpy(int), probabilities.argmax(axis=1)), "Study 2 y_pred differs from argmax")
    _append_if(errors, set(predictions["y_true"].unique()) != set(range(4)), "Study 2 predictions lack an action class")

    group_counts = predictions.groupby(["training_seed", "method", "domain", "missing_pattern"]).agg(
        rows=("scene_id", "size"), scenes=("scene_id", "nunique"), frames=("frame", "nunique")
    )
    for seed in seeds:
        for method in methods:
            for pattern in patterns:
                expected = (test_scenes * sequence_length, test_scenes, sequence_length)
                observed = tuple(group_counts.loc[(seed, method, "clean_controlled", pattern)])
                _append_if(errors, observed != expected, f"Study 2 clean grid mismatch: {seed}/{method}/{pattern}")
            expected = (stress_scenes * sequence_length, stress_scenes, sequence_length)
            observed = tuple(group_counts.loc[(seed, method, "adverse_controlled_injected", "none")])
            _append_if(errors, observed != expected, f"Study 2 stress grid mismatch: {seed}/{method}")

    # Independently recompute predictive seed metrics from raw frame probabilities.
    metric_index = seed_metrics.set_index(["training_seed", "method", "domain"])
    for (seed, method, domain), group in predictions[predictions["missing_pattern"] == "none"].groupby(
        ["training_seed", "method", "domain"], sort=False
    ):
        truth = group["y_true"].to_numpy(int)
        p = group[p_cols].to_numpy(float)
        expected = np.array(
            [
                macro_f1(truth, p.argmax(axis=1), 4),
                negative_log_likelihood(truth, p),
                multiclass_brier(truth, p),
                expected_calibration_error(truth, p, bins),
            ]
        )
        observed = metric_index.loc[(seed, method, domain), ["macro_f1", "nll", "brier", "ece"]].to_numpy(float)
        _append_if(errors, not np.allclose(observed, expected, atol=1e-10), f"Study 2 seed metrics fail raw recomputation: {seed}/{method}/{domain}")

    missing = pd.read_csv(output / "data" / "study2_missing_modality.csv")
    _append_if(errors, len(missing) != len(seeds) * len(methods) * len(patterns), "Study 2 missing-modality grid incomplete")
    missing_index = missing.set_index(["training_seed", "method", "missing_pattern"])
    for (seed, method, pattern), group in predictions[predictions["domain"] == "clean_controlled"].groupby(
        ["training_seed", "method", "missing_pattern"], sort=False
    ):
        truth = group["y_true"].to_numpy(int)
        p = group[p_cols].to_numpy(float)
        expected = np.array(
            [
                macro_f1(truth, p.argmax(axis=1), 4),
                negative_log_likelihood(truth, p),
                multiclass_brier(truth, p),
                expected_calibration_error(truth, p, bins),
            ]
        )
        observed = missing_index.loc[(seed, method, pattern), ["macro_f1", "nll", "brier", "ece"]].to_numpy(float)
        _append_if(errors, not np.allclose(observed, expected, atol=1e-10), f"Study 2 missing metrics fail raw recomputation: {seed}/{method}/{pattern}")

    scene_metrics = pd.read_csv(output / "data" / "study2_scene_metrics.csv")
    missing_scene = pd.read_csv(output / "data" / "study2_missing_scene_metrics.csv")
    _append_if(errors, len(scene_metrics) != len(seeds) * len(methods) * (test_scenes + stress_scenes), "Study 2 scene-metric grid incomplete")
    _append_if(errors, len(missing_scene) != len(seeds) * len(methods) * len(patterns) * test_scenes, "Study 2 missing scene-metric grid incomplete")
    _append_if(errors, scene_metrics.duplicated(["training_seed", "method", "domain", "scene_id"]).any(), "Study 2 scene metrics contain duplicate keys")
    _append_if(errors, missing_scene.duplicated(["training_seed", "method", "missing_pattern", "scene_id"]).any(), "Study 2 missing scene metrics contain duplicate keys")
    scene_counts = scene_metrics.groupby(["training_seed", "method", "domain"])["scene_id"].agg(["size", "nunique"])
    for seed in seeds:
        for method in methods:
            for domain, expected_scenes in (("clean_controlled", test_scenes), ("adverse_controlled_injected", stress_scenes)):
                observed = tuple(scene_counts.loc[(seed, method, domain)])
                _append_if(errors, observed != (expected_scenes, expected_scenes), f"Study 2 scene grid mismatch: {seed}/{method}/{domain}")
    missing_counts = missing_scene.groupby(["training_seed", "method", "missing_pattern"])["scene_id"].agg(["size", "nunique"])
    _append_if(errors, not ((missing_counts["size"] == test_scenes) & (missing_counts["nunique"] == test_scenes)).all(), "Study 2 missing-scene Cartesian grid incomplete")
    drift_from_scene = scene_metrics.groupby(["training_seed", "method", "domain"])["feature_drift"].mean().sort_index()
    drift_reported = seed_metrics.set_index(["training_seed", "method", "domain"])["feature_drift"].sort_index()
    _append_if(errors, not np.allclose(drift_from_scene, drift_reported, atol=1e-10), "Study 2 feature prediction error fails scene aggregation")

    pair = pd.read_csv(output / "data" / "study2_pair_integrity.csv")
    passed = str(pair.loc[0, "pass_exact_latent_pairing"]).lower() == "true"
    _append_if(errors, not passed or float(pair.loc[0, "max_absolute_latent_difference"]) != 0.0, "Study 2 exact latent counterfactual pairing failed")

    split = pd.read_csv(output / "data" / "study2_split_manifest.csv")
    train = split[split["split"] == "train"]
    _append_if(errors, len(train) != len(seeds) or train["data_sha256"].nunique() != len(seeds), "Study 2 training replicates are not independently hashed")
    models = pd.read_csv(output / "data" / "study2_model_manifest.csv")
    _append_if(errors, len(models) != len(seeds) * len(methods), "Study 2 model manifest incomplete")
    _append_if(errors, models["model_sha256"].str.fullmatch(r"[0-9a-f]{64}").eq(False).any(), "Study 2 checkpoint hash malformed")
    _append_if(errors, models.groupby("training_seed")["training_data_sha256"].nunique().max() != 1, "Study 2 methods within replicate used different training data")
    _append_if(errors, models.drop_duplicates("training_seed")["training_data_sha256"].nunique() != len(seeds), "Study 2 training replicate hashes are not unique")
    return {"study2_prediction_rows": len(predictions), "study2_seed_metric_rows": len(seed_metrics), "study2_model_records": len(models)}


def _validate_study3(output: Path, cfg: dict, errors: list[str]) -> dict[str, int]:
    seeds = list(cfg["common"]["training_seeds"])
    methods = list(cfg["study3"]["methods"])
    scenes = int(cfg["study3"]["test_scenes"])
    max_modes = int(cfg["study3"]["modes"])
    miss_threshold = float(cfg["study3"]["miss_threshold_m"])

    seed_metrics = pd.read_csv(output / "data" / "study3_seed_metrics.csv")
    _append_if(errors, len(seed_metrics) != len(seeds) * len(methods), "Study 3 seed-metric grid incomplete")
    _append_if(errors, set(seed_metrics["training_seed"]) != set(seeds), "Study 3 training seeds differ from config")
    _append_if(errors, set(seed_metrics["method"]) != set(methods), "Study 3 methods differ from config")
    predictions = pd.read_csv(output / "data" / "study3_predictions.csv.gz")
    key = ["training_seed", "method", "scene_id"]
    _append_if(errors, predictions.duplicated(key).any(), "Study 3 prediction keys are duplicated")
    expected_rows = len(seeds) * len(methods) * scenes
    _append_if(errors, len(predictions) != expected_rows, f"Study 3 predictions have {len(predictions)} rows, expected {expected_rows}")
    counts = predictions.groupby(["training_seed", "method"])["scene_id"].agg(["size", "nunique"])
    _append_if(errors, not ((counts["size"] == scenes) & (counts["nunique"] == scenes)).all(), "Study 3 per-condition scene grid incomplete")

    intent_cols = [f"intent_p_{index}" for index in range(8)]
    intent_p = predictions[intent_cols].to_numpy(float)
    _append_if(errors, not np.isfinite(intent_p).all() or (intent_p < 0).any() or (intent_p > 1).any(), "Study 3 intent probabilities invalid")
    _append_if(errors, not np.allclose(intent_p.sum(axis=1), 1.0, atol=1e-8), "Study 3 intent probabilities do not sum to one")
    _append_if(errors, not np.array_equal(predictions["intent_pred"].to_numpy(int), intent_p.argmax(axis=1)), "Study 3 intent_pred differs from argmax")

    p_cols = [f"p_mode_{index}" for index in range(1, max_modes + 1)]
    ade_cols = [f"ade_mode_{index}" for index in range(1, max_modes + 1)]
    fde_cols = [f"fde_mode_{index}" for index in range(1, max_modes + 1)]
    p_all = predictions[p_cols].to_numpy(float)
    ade_all = predictions[ade_cols].to_numpy(float)
    fde_all = predictions[fde_cols].to_numpy(float)
    recomputed = {name: np.empty(len(predictions), dtype=float) for name in ["minADE_K", "minFDE_K", "MR_K", "Brier-minFDE_K", "ADE_1", "FDE_1", "MR_1"]}
    for k in (1, max_modes):
        mask = predictions["K"].to_numpy(int) == k
        p = p_all[mask, :k]
        ade = ade_all[mask, :k]
        fde = fde_all[mask, :k]
        _append_if(errors, not np.isfinite(p).all() or not np.isfinite(ade).all() or not np.isfinite(fde).all(), f"Study 3 K={k} mode outputs non-finite")
        _append_if(errors, (p < 0).any() or (p > 1).any() or not np.allclose(p.sum(axis=1), 1.0, atol=1e-8), f"Study 3 K={k} mode probabilities invalid")
        if k == 1:
            _append_if(errors, not np.isnan(p_all[mask, 1:]).all() or not np.isnan(ade_all[mask, 1:]).all() or not np.isnan(fde_all[mask, 1:]).all(), "Study 3 K=1 trailing mode fields must be N/A")
        best = fde.argmin(axis=1)
        row = np.arange(len(best))
        top = p.argmax(axis=1)
        recomputed["minADE_K"][mask] = ade[row, best]
        recomputed["minFDE_K"][mask] = fde[row, best]
        recomputed["MR_K"][mask] = (fde[row, best] > miss_threshold).astype(float)
        recomputed["Brier-minFDE_K"][mask] = fde[row, best] + (1.0 - p[row, best]) ** 2
        recomputed["ADE_1"][mask] = ade[row, top]
        recomputed["FDE_1"][mask] = fde[row, top]
        recomputed["MR_1"][mask] = (fde[row, top] > miss_threshold).astype(float)
        _append_if(errors, not np.array_equal(predictions.loc[mask, "best_fde_mode"].to_numpy(int), best + 1), f"Study 3 K={k} best-FDE mode index incorrect")
    for name, expected in recomputed.items():
        _append_if(errors, not np.allclose(predictions[name].to_numpy(float), expected, atol=1e-10), f"Study 3 {name} fails raw mode-level recomputation")

    # Recompute all seed-level endpoints from per-scene raw predictions.
    grouped = predictions.groupby(["training_seed", "method"], as_index=False)
    expected_rows_list = []
    for (seed, method), group in grouped:
        row = {"training_seed": seed, "method": method}
        row["intent_macro_f1"] = macro_f1(group["intent_true"], group["intent_pred"], 8)
        for name in recomputed:
            row[name] = float(group[name].mean())
        expected_rows_list.append(row)
    expected_frame = pd.DataFrame(expected_rows_list).set_index(["training_seed", "method"]).sort_index()
    observed_frame = seed_metrics.set_index(["training_seed", "method"]).sort_index()
    compare = ["intent_macro_f1", *recomputed.keys()]
    _append_if(errors, not np.allclose(observed_frame[compare], expected_frame[compare], atol=1e-10), "Study 3 seed metrics fail raw recomputation")

    single = seed_metrics[seed_metrics["method"] == "Single mode K=1"]
    reportable = ["minADE_6_reportable", "minFDE_6_reportable", "MR_6_reportable", "Brier-minFDE_6_reportable"]
    _append_if(errors, not single[reportable].isna().all().all(), "K=1 condition is mislabeled with K=6 metrics")
    multi = seed_metrics[seed_metrics["method"] != "Single mode K=1"]
    _append_if(errors, multi[reportable].isna().any().any(), "A K=6 condition lacks reportable K=6 metrics")

    split = pd.read_csv(output / "data" / "study3_split_manifest.csv")
    train = split[split["split"] == "train"]
    _append_if(errors, len(train) != len(seeds) or train["data_sha256"].nunique() != len(seeds), "Study 3 training replicates are not independently hashed")
    models = pd.read_csv(output / "data" / "study3_model_manifest.csv")
    _append_if(errors, len(models) != len(seeds) * len(methods), "Study 3 model manifest incomplete")
    _append_if(errors, models["model_sha256"].str.fullmatch(r"[0-9a-f]{64}").eq(False).any(), "Study 3 checkpoint hash malformed")
    _append_if(errors, models.groupby("training_seed")["training_data_sha256"].nunique().max() != 1, "Study 3 methods within replicate used different training data")
    _append_if(errors, models.drop_duplicates("training_seed")["training_data_sha256"].nunique() != len(seeds), "Study 3 training replicate hashes are not unique")
    parameter_pivot = models[models["method"].isin(["Full temporal graph", "No predicted-intent gating"])].pivot(index="training_seed", columns="method", values="parameter_sha256")
    _append_if(errors, not (parameter_pivot["Full temporal graph"] == parameter_pivot["No predicted-intent gating"]).all(), "No-gating intervention does not share Full fitted parameters")
    inference_pivot = models[models["method"].isin(["Full temporal graph", "No predicted-intent gating"])].pivot(index="training_seed", columns="method", values="inference_rule_sha256")
    _append_if(errors, (inference_pivot["Full temporal graph"] == inference_pivot["No predicted-intent gating"]).any(), "No-gating inference rule hash equals Full")
    return {"study3_prediction_rows": len(predictions), "study3_seed_metric_rows": len(seed_metrics), "study3_model_records": len(models)}


def validate_output(output_dir: str | Path) -> dict[str, Any]:
    output = Path(output_dir)
    required = [
        output / "config_resolved.yaml",
        output / "run_manifest.json",
        output / "summary.json",
        output / "MANUSCRIPT_RESULTS_CONTROLLED_SYNTHETIC.md",
        output / "data" / "study2_seed_metrics.csv",
        output / "data" / "study2_predictions.csv.gz",
        output / "data" / "study2_scene_metrics.csv",
        output / "data" / "study2_missing_modality.csv",
        output / "data" / "study2_missing_scene_metrics.csv",
        output / "data" / "study2_model_manifest.csv",
        output / "data" / "study3_seed_metrics.csv",
        output / "data" / "study3_predictions.csv.gz",
        output / "data" / "study3_model_manifest.csv",
        output / "tables" / "table_execution_status.csv",
        output / "tables" / "table_s2_primary_contrasts.csv",
        output / "tables" / "table_s3_primary_contrasts.csv",
    ]
    errors = [f"Missing required file: {path}" for path in required if not path.is_file()]
    if errors:
        raise RuntimeError("; ".join(errors))
    with (output / "config_resolved.yaml").open("r", encoding="utf-8") as handle:
        cfg = yaml.safe_load(handle)
    manifest = json.loads((output / "run_manifest.json").read_text(encoding="utf-8"))
    _append_if(errors, manifest.get("evidence_label") != EXPECTED_EVIDENCE, "Incorrect controlled-synthetic evidence label")
    _append_if(errors, manifest.get("carla_executed") is not False, "CARLA executed flag must be false")
    _append_if(errors, manifest.get("real_datasets_executed") is not False, "Real-dataset executed flag must be false")
    formal_seeds = manifest.get("training_replicate_seeds", [])
    _append_if(errors, len(formal_seeds) != 10 or len(set(formal_seeds)) != 10, "Exactly 10 unique formal training replicate seeds required")
    _append_if(errors, bool(set(formal_seeds) & set(manifest.get("pilot_training_seeds", []))), "Pilot and final full-run seeds overlap")
    _append_if(errors, manifest.get("preregistered") is not False, "This run must not claim preregistration")

    status = pd.read_csv(output / "tables" / "table_execution_status.csv")
    named = status[status["source"].isin(["CARLA_0.9.16", "nuScenes", "RADIATE", "Argoverse_2_Motion"])]
    _append_if(errors, not named["status"].str.startswith("N/A").all(), "One or more named sources are not N/A")
    _append_if(errors, (named["scenes_or_logs"].astype(str) != "0").any(), "Named sources must have zero executed scenes/logs")

    counts = {}
    counts.update(_validate_study2(output, cfg, errors))
    counts.update(_validate_study3(output, cfg, errors))

    for table_name in ("table_s2_primary_contrasts.csv", "table_s3_primary_contrasts.csv"):
        contrast = pd.read_csv(output / "tables" / table_name)
        expected_support = contrast["direction_ci_consistency_met"].astype(bool) & (contrast["holm_adjusted_p"] < 0.05)
        _append_if(errors, not np.array_equal(contrast["mechanism_support_rule_met"].astype(bool), expected_support), f"{table_name} support flag ignores Holm correction")

    figure_files = sorted((output / "figures").glob("figure_*.*"))
    _append_if(errors, len(figure_files) != 27, f"Expected 27 figure files, found {len(figure_files)}")
    _append_if(errors, any(path.stat().st_size == 0 for path in figure_files), "At least one figure is zero bytes")
    for path in figure_files:
        if path.suffix == ".png" and not _valid_png(path):
            errors.append(f"Invalid PNG: {path.name}")
        elif path.suffix == ".pdf":
            data = path.read_bytes()
            if not (data.startswith(b"%PDF-") and b"%%EOF" in data[-1024:]):
                errors.append(f"Invalid PDF: {path.name}")
        elif path.suffix == ".svg":
            try:
                ElementTree.parse(path)
            except ElementTree.ParseError:
                errors.append(f"Invalid SVG: {path.name}")

    file_hashes = {
        str(path.relative_to(output)): sha256(path)
        for path in [*required, *figure_files]
        if path.is_file()
    }
    report = {
        "passed": not errors,
        "errors": errors,
        "evidence_label": manifest.get("evidence_label"),
        "formal_training_replicate_seeds": formal_seeds,
        **counts,
        "fitted_model_records": counts["study2_model_records"] + counts["study3_model_records"],
        "figure_files": len(figure_files),
        "file_sha256": file_hashes,
        "validation_rule_note": "No rule requires the Full condition to win.",
        "raw_recomputation": "Study 2 predictive metrics and all Study 3 trajectory metrics recomputed from raw saved probabilities/errors.",
    }
    write_json(output / "validation_report.json", report)
    if errors:
        raise RuntimeError("Validation failed: " + "; ".join(errors))
    return report


### `configs/power_protocol.yaml`

In [ ]:
%%writefile configs/power_protocol.yaml
project:
  title: "EGMS-Drive: A Prospective Evaluation Protocol and Simulation-Based Power Analysis"
  version: "1.0.0"
  status: prospective_design_only
  empirical_data_present: false
  empirical_performance_claims: false
  protocol_frozen: true
  frozen_date: "2026-08-10"

primary_design:
  endpoint: episode_level_collision
  endpoint_definition: "At least one physical collision during one prespecified evaluation episode."
  estimand: candidate_minus_control_risk_difference
  primary_comparison: "future EGMS-Drive implementation versus aligned/no-graph comparator"
  control_rate_assumption: 0.06
  candidate_rate_assumption: 0.03
  assumption_source: reviewer_requested_planning_scenario_not_observed_data
  alpha: 0.05
  sidedness: two-sided
  target_power: 0.80
  allocation_ratio: 1.0
  independent_test: pooled_null_unpooled_alternative_score_approximation
  paired_test: exact_mcnemar

simulation:
  master_seed: 20260810
  canonical_repetitions: 50000
  exploratory_repetitions: 10000
  monte_carlo_confidence: 0.95
  sample_sizes: [150, 300, 450, 600, 749, 765, 900, 1000, 1200, 1500]
  power_curve_start: 50
  power_curve_stop: 2000
  power_curve_step: 10

paired_design:
  correlations: [-0.04, 0.0, 0.1, 0.2, 0.4, 0.6]
  planning_correlation: -0.04
  report_joint_probabilities: true

clustered_design:
  mean_cluster_size: 9
  cluster_size_cv_values: [0.0, 0.25, 0.50]
  icc_values: [0.0, 0.01, 0.05, 0.10, 0.20]
  planning_icc: 0.05
  invalid_episode_fraction: 0.05
  note: "ICC is distinct from cross-method paired correlation. Design-effect results are planning approximations."

scenario_matrix:
  scenario_families: 5
  environments: 3
  traffic_densities: 3
  allocation_block: 45
  cluster_interpretation: "One scenario-seed cluster contains nine environment-density episodes."

multiplicity:
  primary_comparisons: 1
  secondary_adjustment: holm
  ablation_comparisons: 4
  conservative_ablation_alpha: 0.0125

sensitivity:
  control_rates: [0.03, 0.06, 0.10, 0.15]
  absolute_reductions: [0.01, 0.02, 0.03, 0.04]
  target_powers: [0.80, 0.90]
  precision_half_widths: [0.02, 0.015, 0.01]
  representative_ci_sample_sizes: [150, 450, 749, 765, 900, 1200, 1500]

ablation_planning:
  collision_scenarios:
    - label: headline_6_to_3_percent
      control_rate: 0.06
      candidate_rate: 0.03
    - label: two_events_in_150_gap
      control_rate: 0.06
      candidate_rate: 0.04666666666666667
  module_specific_endpoints:
    no_cross_modal_alignment: [macro_f1, ece, nll]
    no_temporal_consistency: [temporal_flip_rate, minade, minfde, miss_rate]
    no_scene_graph: [relation_f1, decision_error]
    no_explanation_verifier: [entailment_accuracy, false_accept_rate, fallback_rate]
  rule: "Every learned ablation must be independently retrained across multiple training seeds."


### `configs/studies23.yaml`

In [ ]:
%%writefile configs/studies23.yaml
project:
  id: EGMS_DRIVE_STUDIES_2_3
  title: Controlled mechanism validation for EGMS-Drive Eqs. (4)-(19)
  evidence_label: CONTROLLED_SYNTHETIC_MECHANISM_VALIDATION_NOT_EMPIRICAL_DATASET
  real_datasets_executed: false
  carla_executed: false
  preregistered: false
  analysis_plan_frozen_before_final_full_run: true

common:
  modalities: [camera, lidar, radar, ego]
  pilot_training_seeds: [2211, 2521, 2551, 2591, 2621, 2659, 2687, 2711, 2741, 2777, 2801]
  training_seeds: [3109, 3163, 3203, 3251, 3299, 3343, 3389, 3449, 3491, 3533]
  bootstrap_repetitions: 10000
  confidence_level: 0.95
  calibration_bins: 15
  calibration_binning: equal_mass

study2:
  data_seed: 22001
  sequence_length: 10
  train_sequences: 800
  validation_sequences: 200
  test_sequences: 400
  stress_sequences: 300
  latent_dim: 6
  raw_dim: 10
  embedding_dim: 8
  reliability_temperature: 4.0
  temporal_strength: 0.40
  modality_dropout_probability: 0.35
  classifier_C: 1.0
  classifier_max_iter: 800
  retrieval_pool_sequences: 300
  methods:
    - Full
    - Equal weighting
    - No alignment
    - No temporal consistency
    - No modality dropout-distillation
  missing_patterns:
    - none
    - camera
    - lidar
    - radar
    - ego
    - camera+lidar
  primary_endpoints:
    Equal weighting: stress_nll
    No alignment: sas
    No temporal consistency: feature_drift
    No modality dropout-distillation: mean_missing_macro_f1_drop

study3:
  data_seed: 23001
  train_scenes: 2500
  validation_scenes: 500
  test_scenes: 800
  history_steps: 50
  future_steps: 60
  sample_period_s: 0.1
  modes: 6
  miss_threshold_m: 2.0
  max_neighbors: 6
  intent_classes: [straight, slowing, stopping, cut_in, lane_change, turn_left, turn_right, crossing]
  classifier_C: 0.3
  classifier_max_iter: 1000
  prototypes_per_intent: 3
  methods:
    - Full temporal graph
    - No graph
    - Spatial-only graph
    - No predicted-intent gating
    - Single mode K=1
  primary_endpoints:
    No graph: history_dependent_minFDE_K
    Spatial-only graph: history_dependent_minFDE_K
    No predicted-intent gating: Brier-minFDE_K
    Single mode K=1: MR_1

execution_status:
  CARLA_0.9.16: N/A_NOT_EXECUTED
  nuScenes: N/A_NOT_DOWNLOADED
  RADIATE: N/A_NOT_DOWNLOADED
  Argoverse_2_Motion: N/A_NOT_DOWNLOADED


### `data/study1/study1_figure_inputs.csv`

In [ ]:
%%writefile data/study1/study1_figure_inputs.csv
panel,metric,condition,estimate,ci_low,ci_high,unit,orientation,provenance
action,Macro-F1,Structured minus Baseline B,0.0129,0.0081,0.0175,raw difference,higher_is_better,exported workbook summary; original episode-balanced interval
action,NLL,Structured minus Baseline B,-0.0228,-0.0310,-0.0142,raw difference,lower_is_better,exported workbook summary; original episode-balanced interval
action,Brier,Structured minus Baseline B,-0.0195,-0.0252,-0.0137,raw difference,lower_is_better,exported workbook summary; original episode-balanced interval
action,ECE,Structured minus Baseline B,0.0007,-0.0051,0.0053,raw difference,lower_is_better,exported workbook summary; original episode-balanced interval
event,Collision,Structured minus Baseline B,1.11,-3.89,6.39,percentage points,lower_is_better,45-cell by 8-seed crossed-bootstrap summary
event,Near miss,Structured minus Baseline B,-9.44,-19.17,0.00,percentage points,lower_is_better,45-cell by 8-seed crossed-bootstrap summary
event,Critical event,Structured minus Baseline B,-8.33,-17.47,0.28,percentage points,lower_is_better,reported point estimate; interval retained from audited publication graphic
event,Route completion,Structured minus Baseline B,3.28,-1.16,7.87,percentage points,higher_is_better,45-cell by 8-seed crossed-bootstrap summary
conditional_ttc,TTC-P5,Baseline B,0.665,0.605,0.754,seconds,higher_is_better,marginal descriptive interval retained from audited publication graphic
conditional_ttc,TTC-P5,Structured fusion,1.042,0.863,1.354,seconds,higher_is_better,marginal descriptive interval retained from audited publication graphic
conditional_ttc,TTC-P5 difference,Structured minus Baseline B,0.377,,,seconds,higher_is_better,conditional descriptive difference; no inferential interval
jerk,Jerk-P95,Baseline B,32.88,30.82,34.94,m/s^3,lower_is_better,marginal interval retained from audited publication graphic
jerk,Jerk-P95,Structured fusion,23.12,21.08,25.29,m/s^3,lower_is_better,marginal interval retained from audited publication graphic
jerk,Jerk-P95 difference,Structured minus Baseline B,-9.7624,-11.7983,-7.8408,m/s^3,lower_is_better,exported paired dry-run interval


### `data/study1/study1_table2_inputs.csv`

In [ ]:
%%writefile data/study1/study1_table2_inputs.csv
study_module,comparison_endpoint,effect,ci_low,ci_high,effect_display,ci_display,evidence_status
S1 action,Structured vs Baseline B; macro-F1 up,0.0129,0.0081,0.0175,+0.0129,+0.0081 to +0.0175,Supports synthetic held-out classification improvement
S1 safety proxy,Structured vs Baseline B; collision rate down,1.11,-3.89,6.39,+1.11 pp,-3.89 to +6.39 pp,Safety superiority not established
S1 proxy,Near-miss rate down,-9.44,-19.17,0.00,-9.44 pp,-19.17 to 0.00 pp,Boundary-level proxy evidence
S1 completion,Mean route completion up,3.28,-1.16,7.87,+3.28 pp,-1.16 to +7.87 pp,Improvement not clearly established
S1 conditional TTC,Collision-free TTC-P5 up,0.377,,, +0.377 s,---,Conditional descriptive estimate
S1 jerk proxy,Mean episode jerk-P95 down,-9.7624,-11.7983,-7.8408,-9.7624,-11.7983 to -7.8408,Supports synthetic kinematic smoothness proxy


### `data/studies23_frozen/tables/table_s2_primary_contrasts.csv`

In [ ]:
%%writefile data/studies23_frozen/tables/table_s2_primary_contrasts.csv
comparison,primary_endpoint,difference_full_minus_ablation,ci_low,ci_high,favorable_sign,favorable_training_replicates,total_training_replicates,paired_exact_sign_flip_p,ci_method,direction_ci_consistency_met,evidence,holm_adjusted_p,mechanism_support_rule_met
Full vs Equal weighting,Adverse controlled-injection NLL,-0.008043036991918065,-0.010715029062158767,-0.005579844448871242,negative,10,10,0.001953125,95% paired crossed bootstrap over training replicate and scene (10000 draws),True,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence.",0.0078125,True
Full vs No alignment,Clean controlled SAS,2.6020567699263126,2.5998905091692497,2.6042230306833756,positive,10,10,0.001953125,95% t CI across paired training replicates; fixed test scenes,True,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence.",0.0078125,True
Full vs No temporal consistency,Adverse held-out one-step feature prediction error,-0.05484648121694357,-0.05932775815555361,-0.050484757557415806,negative,10,10,0.001953125,95% paired crossed bootstrap over training replicate and scene (10000 draws),True,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence.",0.0078125,True
Full vs No modality dropout-distillation,Mean scene-level single/dual missing-modality macro-F1 drop,-0.0012720004597655521,-0.002742996479561567,0.00018251496290570875,negative,8,10,0.017578125,95% paired crossed bootstrap over training replicate and scene (10000 draws),False,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence.",0.017578125,False


### `data/studies23_frozen/tables/table_s3_by_regime.csv`

In [ ]:
%%writefile data/studies23_frozen/tables/table_s3_by_regime.csv
method,regime,metric,better,estimate,ci_low,ci_high,training_replicates,ci_definition,evidence
Full temporal graph,history_dependent,minADE_K (m),lower,1.1414401056268553,1.114444538376673,1.1684356728770375,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Full temporal graph,history_dependent,minFDE_K (m),lower,1.6232532719445434,1.5976431933294946,1.6488633505595922,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Full temporal graph,history_dependent,MR_K at 2 m,lower,0.31315789473684214,0.30160463069348514,0.3247111587801991,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Full temporal graph,history_dependent,Brier-minFDE_K,lower,2.0613626795554265,2.0354424395307236,2.0872829195801295,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Full temporal graph,independent,minADE_K (m),lower,1.5730813831459969,1.51869992459967,1.6274628416923238,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Full temporal graph,independent,minFDE_K (m),lower,2.524027719318003,2.425264660349801,2.6227907782862054,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Full temporal graph,independent,MR_K at 2 m,lower,0.31535580524344564,0.3065328278414009,0.3241787826454904,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Full temporal graph,independent,Brier-minFDE_K,lower,3.02844833931604,2.9289057452622447,3.1279909333698357,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Full temporal graph,spatial,minADE_K (m),lower,1.2542073705748191,1.226714969784825,1.2816997713648133,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Full temporal graph,spatial,minFDE_K (m),lower,1.8261736075053734,1.7721742012710415,1.8801730137397052,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Full temporal graph,spatial,MR_K at 2 m,lower,0.2820224719101124,0.2742316623871616,0.28981328143306323,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Full temporal graph,spatial,Brier-minFDE_K,lower,2.328686405582139,2.2731433690076543,2.3842294421566232,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No graph,history_dependent,minADE_K (m),lower,1.3398001517743303,1.2651451825876354,1.4144551209610252,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No graph,history_dependent,minFDE_K (m),lower,2.006925459230217,1.8646113522597116,2.149239566200722,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No graph,history_dependent,MR_K at 2 m,lower,0.35563909774436087,0.33770139110153585,0.3735768043871859,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No graph,history_dependent,Brier-minFDE_K,lower,2.5225467185191093,2.378673605301132,2.6664198317370866,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No graph,independent,minADE_K (m),lower,1.539040821784393,1.5024432576502522,1.5756383859185337,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No graph,independent,minFDE_K (m),lower,2.456540188572881,2.384032986918227,2.5290473902275346,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No graph,independent,MR_K at 2 m,lower,0.31385767790262176,0.3046802238861948,0.3230351319190487,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No graph,independent,Brier-minFDE_K,lower,2.9641983070538283,2.8909947873709525,3.037401826736704,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No graph,spatial,minADE_K (m),lower,1.1628058882410761,1.1260415043234917,1.1995702721586605,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No graph,spatial,minFDE_K (m),lower,1.6566830623521518,1.5906930007173292,1.7226731239869744,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No graph,spatial,MR_K at 2 m,lower,0.2891385767790262,0.2803155993769815,0.29796155418107095,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No graph,spatial,Brier-minFDE_K,lower,2.17101706127786,2.1052943280584206,2.2367397944972995,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No predicted-intent gating,history_dependent,minADE_K (m),lower,4.845231589923481,4.375570292818231,5.314892887028731,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No predicted-intent gating,history_dependent,minFDE_K (m),lower,7.841266343938921,6.53025372126738,9.152278966610462,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No predicted-intent gating,history_dependent,MR_K at 2 m,lower,0.8281954887218046,0.8093468434902419,0.8470441339533673,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No predicted-intent gating,history_dependent,Brier-minFDE_K,lower,8.537983020718771,7.229665321766007,9.846300719671536,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No predicted-intent gating,independent,minADE_K (m),lower,5.225371558033535,4.672657962793728,5.7780851532733415,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No predicted-intent gating,independent,minFDE_K (m),lower,8.477012536532433,6.989042332955288,9.964982740109578,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No predicted-intent gating,independent,MR_K at 2 m,lower,0.8243445692883894,0.8114365537940889,0.8372525847826899,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No predicted-intent gating,independent,Brier-minFDE_K,lower,9.17410917433285,7.688632703196805,10.659585645468894,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No predicted-intent gating,spatial,minADE_K (m),lower,5.151812349637571,4.687767216537581,5.615857482737561,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No predicted-intent gating,spatial,minFDE_K (m),lower,8.178384316443232,6.818333392853795,9.538435240032669,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No predicted-intent gating,spatial,MR_K at 2 m,lower,0.8314606741573035,0.8163573245777456,0.8465640237368615,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No predicted-intent gating,spatial,Brier-minFDE_K,lower,8.875095146510466,7.517785880065412,10.23240441295552,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Single mode K=1,history_dependent,minADE_K (m),lower,3.4874613082525245,3.4772480670074692,3.49767454949758,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Single mode K=1,history_dependent,minFDE_K (m),lower,5.93534087239634,5.916270110313933,5.954411634478747,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Single mode K=1,history_dependent,MR_K at 2 m,lower,0.7849624060150375,0.7787776387852062,0.7911471732448688,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Single mode K=1,history_dependent,Brier-minFDE_K,lower,5.93534087239634,5.916270110313933,5.954411634478747,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Single mode K=1,independent,minADE_K (m),lower,5.391298216863168,5.237868836140167,5.54472759758617,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Single mode K=1,independent,minFDE_K (m),lower,9.612658362451278,9.3168597435155,9.908456981387054,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Single mode K=1,independent,MR_K at 2 m,lower,0.8131086142322097,0.8042270716193062,0.8219901568451132,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Single mode K=1,independent,Brier-minFDE_K,lower,9.612658362451278,9.3168597435155,9.908456981387054,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Single mode K=1,spatial,minADE_K (m),lower,4.173281959390634,4.111538482759677,4.235025436021591,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Single mode K=1,spatial,minFDE_K (m),lower,7.361339014088079,7.2409163410004025,7.481761687175756,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Single mode K=1,spatial,MR_K at 2 m,lower,0.7715355805243446,0.7656115594299769,0.7774596016187122,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Single mode K=1,spatial,Brier-minFDE_K,lower,7.361339014088079,7.2409163410004025,7.481761687175756,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Spatial-only graph,history_dependent,minADE_K (m),lower,1.5158330228345718,1.4438268569905184,1.5878391886786252,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Spatial-only graph,history_dependent,minFDE_K (m),lower,2.351795509541895,2.21041528762118,2.49317573146261,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Spatial-only graph,history_dependent,MR_K at 2 m,lower,0.35037593984962406,0.33366273382922096,0.36708914587002717,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Spatial-only graph,history_dependent,Brier-minFDE_K,lower,2.869155217829726,2.728176863363423,3.010133572296029,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Spatial-only graph,independent,minADE_K (m),lower,1.5488460586417667,1.5073287520236274,1.590363365259906,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Spatial-only graph,independent,minFDE_K (m),lower,2.475965113559677,2.4037246089204425,2.548205618198912,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Spatial-only graph,independent,MR_K at 2 m,lower,0.31498127340823967,0.30601037701036055,0.3239521698061188,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Spatial-only graph,independent,Brier-minFDE_K,lower,2.984373931768881,2.9130990599471653,3.0556488035905964,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Spatial-only graph,spatial,minADE_K (m),lower,1.1481935042324516,1.124263795709267,1.1721232127556362,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Spatial-only graph,spatial,minFDE_K (m),lower,1.6276803292978665,1.5680937926349012,1.6872668659608319,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Spatial-only graph,spatial,MR_K at 2 m,lower,0.28202247191011237,0.27130179399734355,0.2927431498228812,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Spatial-only graph,spatial,Brier-minFDE_K,lower,2.1319055869535104,2.0728258184831443,2.1909853554238765,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."


### `data/studies23_frozen/tables/table_s3_latency.csv`

In [ ]:
%%writefile data/studies23_frozen/tables/table_s3_latency.csv
method,metric,better,estimate,ci_low,ci_high,training_replicates,ci_definition,evidence
Full temporal graph,End-to-end batch-1 CPU latency P50 (ms),lower,0.55731875,0.5486885,0.5669405000000001,10,Median across training-replicate latency summaries with 95% percentile bootstrap CI (10000 draws),"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Full temporal graph,End-to-end batch-1 CPU latency P95 (ms),lower,0.7721313,0.6140213,0.7898394499999999,10,Median across training-replicate latency summaries with 95% percentile bootstrap CI (10000 draws),"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No graph,End-to-end batch-1 CPU latency P50 (ms),lower,0.5701827500000001,0.5618855,0.5809960000000001,10,Median across training-replicate latency summaries with 95% percentile bootstrap CI (10000 draws),"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No graph,End-to-end batch-1 CPU latency P95 (ms),lower,0.802110425,0.7167140999999999,0.9007828,10,Median across training-replicate latency summaries with 95% percentile bootstrap CI (10000 draws),"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Spatial-only graph,End-to-end batch-1 CPU latency P50 (ms),lower,0.5591087499999999,0.5559745,0.5629195,10,Median across training-replicate latency summaries with 95% percentile bootstrap CI (10000 draws),"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Spatial-only graph,End-to-end batch-1 CPU latency P95 (ms),lower,0.7244422,0.6868170999999998,0.751104525,10,Median across training-replicate latency summaries with 95% percentile bootstrap CI (10000 draws),"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No predicted-intent gating,End-to-end batch-1 CPU latency P50 (ms),lower,0.5846845,0.5698749999999999,0.5984825,10,Median across training-replicate latency summaries with 95% percentile bootstrap CI (10000 draws),"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No predicted-intent gating,End-to-end batch-1 CPU latency P95 (ms),lower,0.796205775,0.6969499499999999,1.0303905999999998,10,Median across training-replicate latency summaries with 95% percentile bootstrap CI (10000 draws),"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Single mode K=1,End-to-end batch-1 CPU latency P50 (ms),lower,0.5385932499999999,0.532239,0.54777225,10,Median across training-replicate latency summaries with 95% percentile bootstrap CI (10000 draws),"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Single mode K=1,End-to-end batch-1 CPU latency P95 (ms),lower,0.78024845,0.686703,0.9204925249999998,10,Median across training-replicate latency summaries with 95% percentile bootstrap CI (10000 draws),"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."


### `data/studies23_frozen/tables/table_s3_main.csv`

In [ ]:
%%writefile data/studies23_frozen/tables/table_s3_main.csv
method,K,metric,better,estimate,ci_low,ci_high,training_replicates,ci_definition,evidence
Full temporal graph,6,Synthetic dominant-maneuver macro-F1,higher,0.9170132543403067,0.9141245997461886,0.9199019089344247,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Full temporal graph,6,ADE_1 (m),lower,5.017491804514813,4.812299705769688,5.222683903259937,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Full temporal graph,6,FDE_1 (m),lower,8.8172402695738,8.45382835507643,9.18065218407117,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Full temporal graph,6,MR_1 at 2 m,lower,0.76925,0.7597473691036248,0.7787526308963751,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Full temporal graph,6,minADE_6 (m),lower,1.3231364566752517,1.3059997514571653,1.3402731618933381,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Full temporal graph,6,minFDE_6 (m),lower,1.9916114057488628,1.960116528376709,2.0231062831210163,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Full temporal graph,6,MR_6 at 2 m,lower,0.3035,0.2999033993377344,0.3070966006622656,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Full temporal graph,6,Brier-minFDE_6,lower,2.4733468120619464,2.441126730217616,2.505566893906277,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No graph,6,Synthetic dominant-maneuver macro-F1,higher,0.8548591818601782,0.8519903044276921,0.8577280592926643,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No graph,6,ADE_1 (m),lower,5.7297548153201,5.5072897131774825,5.952219917462718,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No graph,6,FDE_1 (m),lower,10.2173387673669,9.810813170346279,10.623864364387522,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No graph,6,MR_1 at 2 m,lower,0.7875,0.775510507756468,0.7994894922435319,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No graph,6,minADE_6 (m),lower,1.3472248899359651,1.3138784531859766,1.3805713266859536,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No graph,6,minFDE_6 (m),lower,2.0400909751902767,1.9686549561676092,2.111526994212944,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No graph,6,MR_6 at 2 m,lower,0.3195,0.3139205218498834,0.3250794781501166,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No graph,6,Brier-minFDE_6,lower,2.5526249130883043,2.480670885721251,2.6245789404553577,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Spatial-only graph,6,Synthetic dominant-maneuver macro-F1,higher,0.8591837045628598,0.8564106818966284,0.8619567272290911,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Spatial-only graph,6,ADE_1 (m),lower,5.577014386593577,5.34487288229753,5.809155890889624,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Spatial-only graph,6,FDE_1 (m),lower,9.915614400958543,9.493734584023844,10.33749421789324,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Spatial-only graph,6,MR_1 at 2 m,lower,0.7859999999999999,0.7753211032714892,0.7966788967285107,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Spatial-only graph,6,minADE_6 (m),lower,1.4041514342017656,1.368952208842474,1.4393506595610572,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Spatial-only graph,6,minFDE_6 (m),lower,2.1515636734763857,2.081819697783888,2.2213076491688835,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Spatial-only graph,6,MR_6 at 2 m,lower,0.31575000000000003,0.31095865120702004,0.32054134879298,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Spatial-only graph,6,Brier-minFDE_6,lower,2.6615523993019816,2.5926929610533636,2.7304118375505997,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No predicted-intent gating,6,Synthetic dominant-maneuver macro-F1,higher,0.9170132543403067,0.9141245997461886,0.9199019089344247,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No predicted-intent gating,6,ADE_1 (m),lower,13.927053171901871,11.769413496962207,16.084692846841534,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No predicted-intent gating,6,FDE_1 (m),lower,26.259264850102788,23.267290587195145,29.25123911301043,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No predicted-intent gating,6,MR_1 at 2 m,lower,0.9705,0.9662386031731232,0.9747613968268769,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No predicted-intent gating,6,minADE_6 (m),lower,5.0744246328347895,4.580394295325154,5.568454970344425,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No predicted-intent gating,6,minFDE_6 (m),lower,8.16595975904032,6.781404864169719,9.55051465391092,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No predicted-intent gating,6,MR_6 at 2 m,lower,0.828,0.8149925963139103,0.8410074036860896,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
No predicted-intent gating,6,Brier-minFDE_6,lower,8.86280129647045,7.480880144480457,10.244722448460442,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Single mode K=1,1,Synthetic dominant-maneuver macro-F1,higher,0.9170132543403067,0.9141245997461886,0.9199019089344247,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Single mode K=1,1,ADE_1 (m),lower,4.351759518818669,4.3053019179661645,4.398217119671174,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Single mode K=1,1,FDE_1 (m),lower,7.6385724644917925,7.551273578424232,7.725871350559353,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Single mode K=1,1,MR_1 at 2 m,lower,0.789875,0.7843074770342868,0.7954425229657132,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Single mode K=1,1,minADE_6 (m),lower,,,,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Single mode K=1,1,minFDE_6 (m),lower,,,,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Single mode K=1,1,MR_6 at 2 m,lower,,,,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."
Single mode K=1,1,Brier-minFDE_6,lower,,,,10,95% t CI across independent training-sample replicates; fixed test scenes,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence."


### `data/studies23_frozen/tables/table_s3_negative_controls.csv`

In [ ]:
%%writefile data/studies23_frozen/tables/table_s3_negative_controls.csv
negative_control,endpoint,equivalence_margin,difference_full_minus_ablation,ci_low,ci_high,descriptive_equivalence_met,ci_method,note
independent: Full vs No graph,minFDE_K (m),0.25,0.06748753074512252,-0.062396826832390284,0.21396219946224113,True,95% paired crossed bootstrap over training replicate and scene (10000 draws),Prespecified descriptive margin; entire crossed-bootstrap CI must lie within it; not TOST.
spatial: Full vs Spatial-only graph,minFDE_K (m),0.25,0.19849327820750656,-0.017819392009632687,0.5431535434812688,False,95% paired crossed bootstrap over training replicate and scene (10000 draws),Prespecified descriptive margin; entire crossed-bootstrap CI must lie within it; not TOST.


### `data/studies23_frozen/tables/table_s3_primary_contrasts.csv`

In [ ]:
%%writefile data/studies23_frozen/tables/table_s3_primary_contrasts.csv
comparison,primary_endpoint,difference_full_minus_ablation,ci_low,ci_high,favorable_sign,favorable_training_replicates,total_training_replicates,paired_exact_sign_flip_p,ci_method,direction_ci_consistency_met,evidence,holm_adjusted_p,mechanism_support_rule_met
Full temporal graph vs No graph,History-dependent minFDE_6,-0.3836721872856735,-0.7087138440557708,-0.15635558967286728,negative,10,10,0.001953125,95% paired crossed bootstrap over training replicate and scene (10000 draws),True,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence.",0.0078125,True
Full temporal graph vs Spatial-only graph,History-dependent minFDE_6,-0.7285422375973519,-1.253633697701952,-0.3138584440136221,negative,10,10,0.001953125,95% paired crossed bootstrap over training replicate and scene (10000 draws),True,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence.",0.0078125,True
Full temporal graph vs No predicted-intent gating,Brier-minFDE_6,-6.389454484408502,-7.6836999179955825,-5.203129529174992,negative,10,10,0.001953125,95% paired crossed bootstrap over training replicate and scene (10000 draws),True,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence.",0.0078125,True
Full top-1 vs independently trained Single mode K=1,MR_1 at 2 m,-0.020625,-0.04650312499999997,0.004128124999999955,negative,9,10,0.00390625,95% paired crossed bootstrap over training replicate and scene (10000 draws),False,"Controlled synthetic mechanism validation only; not CARLA, nuScenes, RADIATE, Argoverse 2, real-weather, or real-world evidence.",0.0078125,False


### Verify every written canonical input against its release SHA-256

In [ ]:
import hashlib
import os
import sys
from pathlib import Path

EXPECTED_HASHES = {
  "configs/power_protocol.yaml": "153d25f3faf70de159a0f0f96e72c654cad62c3808ac498133a1975ef9993e6f",
  "configs/studies23.yaml": "dbf401c42ebd076a05a344c419a898d562f51ab7fea497cdb3bd979c5899264e",
  "data/studies23_frozen/tables/table_s2_primary_contrasts.csv": "2005eae78e1ebca5a783c959cbc91d34366424368916d1688d3f2be120636c55",
  "data/studies23_frozen/tables/table_s3_by_regime.csv": "f3ab6c5483acecf1b491f519ce79074c42fded0faeaecec58ee619d89cfe1dd8",
  "data/studies23_frozen/tables/table_s3_latency.csv": "75479c25a27098d0f66cc30624f031d24dd6504b2d0671456a14e4aef3d5166b",
  "data/studies23_frozen/tables/table_s3_main.csv": "c07335c3f274c00804b74f0fbfa29f6cef0c03ab55a7b0cd61b25e4801f40816",
  "data/studies23_frozen/tables/table_s3_negative_controls.csv": "fd728aa94f58a44e5839d0af472f5eb6cc58f8789402dfade421db110d3638fa",
  "data/studies23_frozen/tables/table_s3_primary_contrasts.csv": "4db7dfdcaac4ec1f65a45d3e930630d1498aa04124fc25046318366a1be96a4b",
  "data/study1/study1_figure_inputs.csv": "69151b0e5ebb7cd8c3b2e488a64855e817035393f5c34ecdac4153808d528de9",
  "data/study1/study1_table2_inputs.csv": "fb89c41a62c26f959b16cde741a0cb8fa28fc86e7757a5b6856fa44c85832972",
  "run_publication.py": "808842fa11f8552fb1667d9318842845100df586ea40ec1fc140fbe7bff38059",
  "run_studies.py": "85b411e70ed5251cd182f6bccadcb61bdb4907b1d088ff0a792d9abcec5aa948",
  "src/egms_power/__init__.py": "644856e8526a4fdd5e2db3cac02c7a74e8d5cd010cbb4eed8593ecb53027c72d",
  "src/egms_power/cli.py": "ca1e11acfbaf8595a85bd15df1904bfd5f444977f03badcccd985ede8beae4f6",
  "src/egms_power/config.py": "55bdca4987e53737c3cc16dc4cd56a7b712c7bc334a6190879fcaf077cba13c5",
  "src/egms_power/pipeline.py": "e12b45dd4856b6af741e9b7126d85d46be1f99ed2a998929e13ce7ac8d991a95",
  "src/egms_power/statistics.py": "4e89b5e94979244ced4238ee13fe151cd2c2ab2adddc70c5b7588024830b08bc",
  "src/egms_publication/__init__.py": "fa0a0705c8a61bdb72145aec3b50a3255f412372f89ec99e7c43f7729b75d4c7",
  "src/egms_publication/figures.py": "ea8830203d838ae0503bf7945d507b9faf1b888da48686fbac053cf77e8bafca",
  "src/egms_publication/runner.py": "3ab9303e1502db7f2b0301e7a64f32f76e3302d0ae2d3877964f693d5296de91",
  "src/egms_publication/style.py": "b83f463863e12c4441ee08d0108ec9fd6c2c6aae3740050e8a88cd8e0fc10969",
  "src/egms_publication/tables.py": "d7ef477d356c64150752dae8845bbfbf4d4dcb884d3f5581effdab8c20412876",
  "src/egms_publication/utils.py": "12fe7ca58cd64c235f24b78cc11feac2d805e420091a30c9d7cf0300a8f795c4",
  "src/egms_publication/validation.py": "76f9a1f41e34a9d16f1221d418087e20664863e098f5630004ac9ab5e0f2024b",
  "src/egms_studies23/__init__.py": "712de5162d097cc5acf72b98c091d79b029c6d4c45878af7d8a0d894256abdb8",
  "src/egms_studies23/common.py": "80405ee26c91bd85b99eebfc087f716d2a3b1355522bcdb254c06d33358c171a",
  "src/egms_studies23/plots.py": "245ae8ce8f1a55a17e964b6e7f0a520cf69d9b7dbf2a0d40890aca9c7f3e1ec6",
  "src/egms_studies23/reporting.py": "9926cc96bfa781fcd9509f059440095f27d51fb878277b82c3b39af4389d9ec1",
  "src/egms_studies23/runner.py": "daf4ab5f5899db6e94e4fb5d9df4d4247402a6e654f32a0ea85dff7a2b1d112a",
  "src/egms_studies23/study2.py": "4d0a42e5e67a912021483bc871a6fa687a912f6d4973f13abfb82a57c4c1a1a4",
  "src/egms_studies23/study3.py": "df3592b5ebed376481044fff53f2d320063970db99a3f67329e27b7e9c69a3a9",
  "src/egms_studies23/validation.py": "252b65b19e0d709742ca5c9a2f948da548af69549eb28e91fbb931f76d84fc22"
}
FORBIDDEN_INPUT_SUFFIXES = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".pdf", ".svg", ".docx", ".gz", ".zip"}

assert set(CANONICAL_PATHS) == set(EXPECTED_HASHES)
assert not any(Path(relative).suffix.lower() in FORBIDDEN_INPUT_SUFFIXES for relative in CANONICAL_PATHS)
for relative in CANONICAL_PATHS:
    source_path = WORK_ROOT / relative
    if not source_path.is_file():
        raise FileNotFoundError(source_path)
    observed = hashlib.sha256(source_path.read_bytes()).hexdigest()
    if observed != EXPECTED_HASHES[relative]:
        raise RuntimeError(f"Canonical source hash mismatch: {relative}")

os.environ["MPLBACKEND"] = "Agg"
os.environ["MPLCONFIGDIR"] = str(WORK_ROOT / ".mplconfig")
sys.path.insert(0, str(WORK_ROOT / "src"))
print(f"SHA-256 verified {len(CANONICAL_PATHS)} readable source/config/CSV files")
print("No image, PDF, Word, archive, or compressed data file was embedded as an input")


## 3. Execute the complete publication-reproduction pipeline

In [ ]:
from pathlib import Path
from egms_publication.runner import run_publication

result = run_publication(Path("outputs/colab_publication_reproduction"))
OUTPUT_DIR = Path(result["output_directory"])
if not OUTPUT_DIR.is_absolute():
    OUTPUT_DIR = WORK_ROOT / OUTPUT_DIR
assert result["validation"]["passed"] is True
print("Publication reproduction: PASS")
print("Output directory:", OUTPUT_DIR)
print("Table rows:", result["table_rows"])


## 4. Display Table 2 and Supplementary Tables S2–S4

In [ ]:
import html
import importlib.util
import pandas as pd

table_paths = [
    ("Table 2 — principal effect estimates", OUTPUT_DIR / "manuscript/tables/Table_2_main_effects.csv"),
    ("Table S2 — planning summary", OUTPUT_DIR / "supplement/compact/Tables/Table_S2_planning_summary.csv"),
    ("Table S3 — condensed planning sensitivity", OUTPUT_DIR / "supplement/compact/Tables/Table_S3_condensed_planning.csv"),
    ("Table S4 — 33 validation checks", OUTPUT_DIR / "supplement/compact/Tables/Table_S4_validation_checks.csv"),
]

HAS_IPYTHON = importlib.util.find_spec("IPython") is not None
if HAS_IPYTHON:
    from IPython.display import HTML, display

TABLE_CSS = """
<style>
.egms-table-card {font-family: Arial, sans-serif; margin: 1.25rem 0 2rem;}
.egms-table-card h3 {color: #17365d; margin: 0 0 .35rem; font-size: 1.15rem;}
.egms-table-card .egms-meta {color: #5f6b7a; margin: 0 0 .6rem; font-size: .9rem;}
.egms-table-wrap {overflow: auto; max-height: 640px; border: 1px solid #cbd5e1; border-radius: 6px;}
.egms-table {border-collapse: collapse; width: 100%; font-size: .88rem; line-height: 1.35;}
.egms-table th {background: #17365d; color: white; padding: .55rem; text-align: left; white-space: nowrap; position: sticky; top: 0; z-index: 1;}
.egms-table td {border-top: 1px solid #dbe3ec; padding: .5rem .55rem; vertical-align: top;}
.egms-table tbody tr:nth-child(even) {background: #f5f8fb;}
.egms-table tbody tr:hover {background: #eaf2fb;}
</style>
"""

def render_table_html(title, path, frame):
    table = frame.to_html(index=False, border=0, classes="egms-table", na_rep="—", escape=True)
    return (
        TABLE_CSS
        + '<section class="egms-table-card">'
        + f"<h3>{html.escape(title)}</h3>"
        + f'<p class="egms-meta">{len(frame)} rows · Source: {html.escape(str(path.relative_to(OUTPUT_DIR)))}</p>'
        + f'<div class="egms-table-wrap">{table}</div></section>'
    )

for title, path in table_paths:
    frame = pd.read_csv(path)
    if HAS_IPYTHON:
        display(HTML(render_table_html(title, path, frame)))
    else:
        print("\n", title)
        print(frame.to_string(index=False))


## 5. Display generated Figures 2–4 and Figure S1

In [ ]:
from PIL import Image as PILImage

figure_paths = [
    ("Figure 2 — Study 1 exported-summary diagnostics", OUTPUT_DIR / "manuscript/figures/Figure_2_Study1.png"),
    ("Figure 3 — Study 2 mechanism contrasts", OUTPUT_DIR / "manuscript/figures/Figure_3_Study2.png"),
    ("Figure 4 — Study 3 graph, intent, and trajectory diagnostics", OUTPUT_DIR / "manuscript/figures/Figure_4_Study3.png"),
    ("Figure S1 — prospective independent-design power", OUTPUT_DIR / "power_full/figures/figure1_unpaired_power_curve.png"),
]

if HAS_IPYTHON:
    from IPython.display import HTML, Image as IPythonImage, display

for title, path in figure_paths:
    assert path.is_file() and path.stat().st_size > 0
    with PILImage.open(path) as generated_figure:
        width_px, height_px = generated_figure.size
    if HAS_IPYTHON:
        relative_path = path.relative_to(OUTPUT_DIR)
        display(HTML(
            '<section style="font-family:Arial,sans-serif;margin:1.25rem 0 .5rem">'
            f'<h3 style="color:#17365d;margin:0 0 .3rem">{title}</h3>'
            f'<p style="color:#5f6b7a;margin:0">{width_px}×{height_px} px · Generated from code · {relative_path}</p>'
            '</section>'
        ))
        display(IPythonImage(filename=str(path), width=1050))
    else:
        print(f"{title} [{width_px}×{height_px} px] -> {path}")


## 6. Verify and download the generated result ZIP

In [ ]:
import hashlib
import json

validation_path = OUTPUT_DIR / "validation_report.json"
validation = json.loads(validation_path.read_text(encoding="utf-8"))
assert validation["passed"] is True, validation["failures"]

ZIP_PATH = Path(result["output_zip"])
if not ZIP_PATH.is_absolute():
    ZIP_PATH = WORK_ROOT / ZIP_PATH
assert ZIP_PATH.is_file() and ZIP_PATH.stat().st_size > 0
zip_sha256 = hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()
print("Validation: PASS")
print("Generated ZIP:", ZIP_PATH)
print("ZIP bytes:", ZIP_PATH.stat().st_size)
print("ZIP SHA-256:", zip_sha256)

if IS_COLAB:
    from google.colab import files
    files.download(str(ZIP_PATH))


## 7. Optional full controlled Studies 2–3 refit

The readable notebook includes `run_studies.py` and every module under `src/egms_studies23/`, so the complete controlled synthetic refit is inspectable. It is intentionally disabled by default because it is much slower and produces large intermediate prediction files. Set the flag in the next cell to `True` only when that full refit is required.

In [ ]:
RUN_FULL_CONTROLLED_REFIT = False

if RUN_FULL_CONTROLLED_REFIT:
    import subprocess
    import sys

    subprocess.run(
        [
            sys.executable,
            "run_studies.py",
            "run",
            "--config",
            "configs/studies23.yaml",
            "--output",
            "outputs/full_controlled_refit",
        ],
        check=True,
    )
    print("Full controlled Studies 2–3 refit: complete")
else:
    print("Full controlled Studies 2–3 refit skipped (default).")


## Interpretation boundary

The notebook regenerates the publication graphics and tables from numeric inputs and executable statistical code; it never copies an attached manuscript image. Study 1 remains limited by unavailable raw training/evaluation provenance. Studies 2–3 remain controlled synthetic mechanism validation, and all power values remain prospective planning quantities.